# INUTS Urdu TTS — corrected training and inference notebook

> **Important:** This notebook continues the existing checkpoint-based experiment. It preserves the current model architecture and checkpoint format, adds supervised duration-predictor training, and changes inference to use predicted durations instead of evenly dividing a hard-coded number of frames.
>
> **Kaggle setup:** Attach the Mozilla Common Voice Urdu dataset and the existing checkpoint/notebook output used by the restore cell. Select a GPU accelerator. Run the notebook in order. The training target is set to global step 1500, so a checkpoint at step 1000 will receive 500 additional steps. If your available checkpoint is only step 500, the notebook will continue training to step 1500.
>
> **Quality note:** This is a corrected experimental model, not a guarantee of production-quality speech. Listen to the generated WAV and evaluate pronunciation, intelligibility, duration, and speaker consistency before making further architecture changes.



# INUTS Urdu TTS — Kaggle preparation
This notebook prepares a **small Urdu pilot** for the original from-scratch INUTS acoustic model. It does not download the 5.79 GB archive automatically and does not use pretrained TTS weights.

## Safe workflow
1. Attach the Common Voice Urdu archive or extracted files as a Kaggle input (or upload a small pilot subset).
2. Run the checks below to locate `validated.tsv` and `clips/`.
3. Generate a manifest for the validated clips only.
4. Alignment and feature extraction are separate steps; do not start model training until alignments are verified.

The source is primarily an ASR corpus. Check the dataset's current license and terms for your intended use before processing. Do not redistribute the recordings.


In [ ]:
from pathlib import Path
import os, csv, json, hashlib

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/inuts_urdu_tts')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Kaggle inputs:', INPUT_ROOT.exists())
if INPUT_ROOT.exists():
    for p in list(INPUT_ROOT.iterdir())[:30]:
        print(p)


## 1. Find the Common Voice files
Expected layout includes `validated.tsv` and a `clips/` directory. This cell only searches Kaggle's attached inputs; it does not download anything.


In [ ]:
validated_files = list(INPUT_ROOT.rglob('validated.tsv')) if INPUT_ROOT.exists() else []
clip_dirs = [p for p in INPUT_ROOT.rglob('clips') if p.is_dir()] if INPUT_ROOT.exists() else []
print('validated.tsv:', *validated_files, sep='\\n  ')
print('clips dirs:', *clip_dirs, sep='\\n  ')
if not validated_files or not clip_dirs:
    print('No dataset attached yet. Add the Urdu Common Voice files as a Kaggle input, then rerun this cell.')


## 2. Build a conservative pilot manifest
Only rows marked validated are read. The manifest stores relative audio paths and a one-way speaker key; it does not copy audio files. This is a preparation check, not phone alignment.


In [ ]:
if validated_files and clip_dirs:
    tsv = validated_files[0]
    clips = next((d for d in clip_dirs if tsv.parent in d.parents or d.parent == tsv.parent), clip_dirs[0])
    rows = []
    with tsv.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\\t')
        for row in reader:
            rel = row.get('path', '').strip()
            text = ' '.join((row.get('sentence') or '').split())
            if not rel or not text:
                continue
            audio = clips / rel
            if not audio.is_file():
                continue
            speaker = (row.get('client_id') or 'unknown').strip()
            speaker_key = hashlib.sha256(speaker.encode('utf-8')).hexdigest()[:16]
            rows.append({'audio': str(audio), 'text': text, 'speaker_key': speaker_key, 'language': 'ur'})
    out = WORK_ROOT / 'urdu_pilot_manifest.jsonl'
    with out.open('w', encoding='utf-8') as f:
        for item in rows:
            f.write(json.dumps(item, ensure_ascii=False) + '\\n')
    print('Manifest:', out, '| matched clips:', len(rows))
else:
    print('Attach the dataset first; no manifest generated.')


## 3. Pilot quality gate
Before alignment, inspect a sample manually. Do not assume that every validated clip has perfect transcript/audio agreement. Phone-level alignments require an Urdu-capable aligner and Urdu pronunciation resources; English alignment models are not suitable substitutes.


In [ ]:
if 'rows' in globals() and rows:
    print('Pilot manifest rows:', len(rows))
    for item in rows[:10]:
        print(item['text'][:180], ' | ', Path(item['audio']).name)
else:
    print('No rows to inspect yet.')


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("\nFolder:", root)
    print("Subfolders:", dirs[:20])
    print("Files:", files[:20])

In [ ]:
import pandas as pd
import os

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"

df = pd.read_csv(os.path.join(base, "validated.tsv"), sep="\t")

print("Total validated records:", len(df))
print("\nColumns:", df.columns.tolist())
print("\nSample records:")
display(df.head(5))

clips = os.path.join(base, "clips")
print("\nAudio files available:", len(os.listdir(clips)))

In [ ]:
import pandas as pd
import os

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
clips = os.path.join(base, "clips")

df = pd.read_csv(os.path.join(base, "validated.tsv"), sep="\t")

print("Validated records:", len(df))
print("Available audio files:", len(os.listdir(clips)))
print("\nColumns:", df.columns.tolist())

matched = df["path"].apply(
    lambda x: os.path.isfile(os.path.join(clips, x))
).sum()

print("\nMatched audio files:", matched)
print("Missing audio files:", len(df) - matched)

display(df[["path", "sentence"]].head(5))

In [ ]:
print("INUTS Kaggle is working!")

In [ ]:
import os
import pandas as pd

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
clips = os.path.join(base, "clips")

df = pd.read_csv(os.path.join(base, "validated.tsv"), sep="\t")

audio_names = set(os.listdir(clips))
matched = df["path"].isin(audio_names).sum()

print("Validated records:", len(df))
print("Matched audio:", matched)
print("Missing audio:", len(df) - matched)
print("Unique speakers:", df["client_id"].nunique())

display(df[["sentence", "path"]].head(5))

In [ ]:
print("Validated records:", len(df))
print("Unique speakers:", df["client_id"].nunique())
print("Matched audio:", df["path"].isin(audio_names).sum())

In [ ]:
import soundfile as sf
import os
import numpy as np

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"

files = df["path"].head(10).tolist()

for f in files:
    audio, sr = sf.read(os.path.join(base, f))
    print(f, "| Sample rate:", sr, "| Duration:", round(len(audio)/sr, 2), "seconds")

In [ ]:
import pandas as pd
import os

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"

duration_file = os.path.join(base, "clip_durations.tsv")

durations = pd.read_csv(duration_file, sep="\t")

print("Total duration records:", len(durations))
print(durations.head())
print("\nTotal hours:", round(durations.iloc[:, -1].sum() / 3600, 2))

In [ ]:
duration_map = durations.set_index("clip")["duration[ms]"]

validated_ms = df["path"].map(duration_map).sum()

print("Validated recordings:", len(df))
print("Validated audio hours:", round(validated_ms / 3600000, 2))
print("Missing duration records:", df["path"].map(duration_map).isna().sum())

In [ ]:
print("Empty transcripts:", df["sentence"].isna().sum())

print("Duplicate audio paths:", df["path"].duplicated().sum())

print("Duplicate sentences:", df["sentence"].duplicated().sum())

print("\nText length statistics:")
print(df["sentence"].astype(str).str.len().describe())

print("\nSample Urdu sentences:")
for sentence in df["sentence"].head(10):
    print(sentence)

In [ ]:
df["text_length"] = df["sentence"].astype(str).str.len()

print("Above 200 characters:", (df["text_length"] > 200).sum())
print("Above 500 characters:", (df["text_length"] > 500).sum())
print("Above 1000 characters:", (df["text_length"] > 1000).sum())

print("\nLongest 10 transcripts:")

longest = df.nlargest(10, "text_length")

for _, row in longest.iterrows():
    print("\nLength:", row["text_length"])
    print("Audio:", row["path"])
    print("Text:", repr(row["sentence"][:300]))

In [ ]:
clean_df = df[df["sentence"].astype(str).str.len() <= 200].copy()

print("Original records:", len(df))
print("Clean records:", len(clean_df))
print("Removed records:", len(df) - len(clean_df))
print("Remaining long transcripts:",
      (clean_df["sentence"].astype(str).str.len() > 200).sum())

clean_df.to_csv(
    "/kaggle/working/inuts_urdu_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nClean dataset saved successfully!")

In [ ]:
print("Unique speakers:", clean_df["client_id"].nunique())

speaker_counts = clean_df.groupby("client_id").size()

print("\nSpeakers with most recordings:")
print(speaker_counts.sort_values(ascending=False).head(10))

print("\nSpeaker distribution:")
print(speaker_counts.describe())

print("\nGender distribution:")
print(clean_df["gender"].value_counts(dropna=False))

print("\nAge distribution:")
print(clean_df["age"].value_counts(dropna=False))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 10% test speakers
split1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)

train_val_idx, test_idx = next(
    split1.split(clean_df, groups=clean_df["client_id"])
)

train_val = clean_df.iloc[train_val_idx].copy()
test_df = clean_df.iloc[test_idx].copy()

# 10% of remaining speakers for validation
split2 = GroupShuffleSplit(n_splits=1, test_size=0.1111, random_state=42)

train_idx, val_idx = next(
    split2.split(train_val, groups=train_val["client_id"])
)

train_df = train_val.iloc[train_idx].copy()
val_df = train_val.iloc[val_idx].copy()

print("Train:", len(train_df), "records |", train_df["client_id"].nunique(), "speakers")
print("Validation:", len(val_df), "records |", val_df["client_id"].nunique(), "speakers")
print("Test:", len(test_df), "records |", test_df["client_id"].nunique(), "speakers")

print("\nSpeaker overlap:")
print("Train-Test:", len(set(train_df.client_id) & set(test_df.client_id)))
print("Train-Val:", len(set(train_df.client_id) & set(val_df.client_id)))
print("Val-Test:", len(set(val_df.client_id) & set(test_df.client_id)))

In [ ]:
output = "/kaggle/working"

train_df.to_csv(f"{output}/inuts_train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(f"{output}/inuts_validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(f"{output}/inuts_test.csv", index=False, encoding="utf-8-sig")

print("All three datasets saved!")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
import unicodedata

def normalize_urdu(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("ـ", "")
    text = " ".join(text.split())
    return text.strip()

train_df["normalized_text"] = train_df["sentence"].apply(normalize_urdu)

print("Original:")
print(train_df["sentence"].iloc[0])

print("\nNormalized:")
print(train_df["normalized_text"].iloc[0])

print("\nEmpty after normalization:",
      (train_df["normalized_text"].str.len() == 0).sum())

print("\nUnique characters:",
      len(set("".join(train_df["normalized_text"]))))

In [ ]:
import json

all_text = "".join(train_df["normalized_text"].astype(str))

vocab = sorted(set(all_text))

# Padding aur unknown character tokens
symbols = ["<PAD>", "<UNK>"] + vocab

char_to_id = {
    char: idx for idx, char in enumerate(symbols)
}

id_to_char = {
    idx: char for char, idx in char_to_id.items()
}

with open("/kaggle/working/inuts_urdu_vocab.json", "w", encoding="utf-8") as f:
    json.dump(symbols, f, ensure_ascii=False, indent=2)

print("Vocabulary size:", len(symbols))
print("First 50 tokens:", symbols[:50])
print("Vocabulary saved successfully!")

In [ ]:
import soundfile as sf
import os
import numpy as np

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"

sample = train_df.sample(20, random_state=42)

for _, row in sample.iterrows():
    audio, sr = sf.read(os.path.join(base, row["path"]))
    duration = len(audio) / sr

    print(
        "Duration:", round(duration, 2),
        "| Text chars:", len(row["normalized_text"]),
        "| Sample rate:", sr
    )

In [ ]:
import soundfile as sf
import os
import numpy as np

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"
sample = train_df.sample(20, random_state=42)

for _, row in sample.iterrows():
    audio, sr = sf.read(os.path.join(base, row["path"]))

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    peak = np.max(np.abs(audio))
    rms = np.sqrt(np.mean(audio ** 2))
    silence = np.mean(np.abs(audio) < 0.005) * 100
    clipping = np.mean(np.abs(audio) >= 0.99) * 100

    print(
        "RMS:", round(rms, 4),
        "| Peak:", round(peak, 4),
        "| Silence:", round(silence, 1), "%",
        "| Clipping:", round(clipping, 2), "%"
    )

In [ ]:
import matplotlib.pyplot as plt
import soundfile as sf
import os
from IPython.display import Audio, display

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"

row = train_df.sample(1, random_state=10).iloc[0]

audio, sr = sf.read(os.path.join(base, row["path"]))

print("Urdu Text:")
print(row["normalized_text"])

print("\nDuration:", round(len(audio)/sr, 2), "seconds")
print("Sample Rate:", sr)

display(Audio(audio, rate=sr))

plt.figure(figsize=(12, 3))
plt.plot(audio)
plt.title("INUTS Urdu Audio Waveform")
plt.show()

In [ ]:
import librosa
import numpy as np
import soundfile as sf
import os

TARGET_SR = 24000
N_MELS = 80

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"

# Ek sample audio load karo
row = train_df.sample(1, random_state=42).iloc[0]

audio, sr = sf.read(os.path.join(base, row["path"]))

# Stereo ho to mono mein convert
if audio.ndim == 2:
    audio = audio.mean(axis=1)

# 24 kHz mein resample
audio = librosa.resample(
    audio.astype(np.float32),
    orig_sr=sr,
    target_sr=TARGET_SR
)

# Mel spectrogram
mel = librosa.feature.melspectrogram(
    y=audio,
    sr=TARGET_SR,
    n_fft=1024,
    hop_length=256,
    n_mels=N_MELS
)

mel_db = librosa.power_to_db(mel, ref=np.max)

print("Original sample rate:", sr)
print("New sample rate:", TARGET_SR)
print("Audio shape:", audio.shape)
print("Mel shape:", mel_db.shape)
print("Transcript:", row["normalized_text"])

In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf
from tqdm.auto import tqdm

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"
output = "/kaggle/working/inuts_mels"

os.makedirs(output, exist_ok=True)

TARGET_SR = 24000
success = 0
failed = 0

for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):

    try:
        path = os.path.join(base, row["path"])
        audio, sr = sf.read(path)

        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        audio = librosa.resample(
            audio.astype(np.float32),
            orig_sr=sr,
            target_sr=TARGET_SR
        )

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=TARGET_SR,
            n_fft=1024,
            hop_length=256,
            n_mels=80
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        np.save(
            os.path.join(output, f"{idx}.npy"),
            mel_db.astype(np.float32)
        )

        success += 1

    except Exception:
        failed += 1

print("Successful:", success)
print("Failed:", failed)
print("Saved folder:", output)

In [ ]:
import shutil
shutil.make_archive(
    "/kaggle/working/inuts_mels_backup",
    "zip",
    "/kaggle/working/inuts_mels"
)

print("Backup ready!")

In [ ]:
import os

print("Working folders:")
print(os.listdir("/kaggle/working"))

print("\nSearching for inuts_mels:")
for root, dirs, files in os.walk("/kaggle/working"):
    if "inuts_mels" in dirs:
        print("FOUND:", os.path.join(root, "inuts_mels"))

In [ ]:
import os

print("Working files:", os.listdir("/kaggle/working"))
print("Dataset folders:", os.listdir("/kaggle/input"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    for f in files:
        if f in ["validated.tsv", "train.tsv", "test.tsv", "dev.tsv"]:
            print(os.path.join(root, f))

In [ ]:
import os
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
CLIPS = os.path.join(BASE, "clips")

# Validated dataset load
df = pd.read_csv(
    os.path.join(BASE, "validated.tsv"),
    sep="\t",
    on_bad_lines="skip"
)

# Required columns check
df = df.dropna(subset=["client_id", "path", "sentence"])

# Text cleanup
df["sentence"] = (
    df["sentence"].astype(str)
    .str.normalize("NFC")
    .str.replace("ـ", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["sentence"] != ""].reset_index(drop=True)

# Audio files verification
df["audio_exists"] = df["path"].apply(
    lambda x: os.path.isfile(os.path.join(CLIPS, x))
)
df = df[df["audio_exists"]].copy()
df = df.drop(columns=["audio_exists"]).reset_index(drop=True)

# Speaker-wise split
speakers = df["client_id"].unique().copy()
rng = np.random.default_rng(42)
rng.shuffle(speakers)

train_speakers = set(speakers[:330])
val_speakers = set(speakers[330:372])
test_speakers = set(speakers[372:])

train_df = df[df["client_id"].isin(train_speakers)].copy()
validation_df = df[df["client_id"].isin(val_speakers)].copy()
test_df = df[df["client_id"].isin(test_speakers)].copy()

# Save CSV files
train_df.to_csv("/kaggle/working/inuts_train.csv", index=False)
validation_df.to_csv("/kaggle/working/inuts_validation.csv", index=False)
test_df.to_csv("/kaggle/working/inuts_test.csv", index=False)

print("Training:", len(train_df), "Speakers:", len(train_speakers))
print("Validation:", len(validation_df), "Speakers:", len(val_speakers))
print("Testing:", len(test_df), "Speakers:", len(test_speakers))
print("Total:", len(df))

In [ ]:
print("Train speakers:", train_df["client_id"].nunique())
print("Validation speakers:", validation_df["client_id"].nunique())
print("Test speakers:", test_df["client_id"].nunique())

print("Train/Validation overlap:",
      len(set(train_df.client_id) & set(validation_df.client_id)))

print("Train/Test overlap:",
      len(set(train_df.client_id) & set(test_df.client_id)))

print("Validation/Test overlap:",
      len(set(validation_df.client_id) & set(test_df.client_id)))

In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf
from tqdm.auto import tqdm

base = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu/clips"
output = "/kaggle/working/inuts_mels"

os.makedirs(output, exist_ok=True)

success = 0
failed = 0
skipped = 0
errors = []

for idx, row in tqdm(
    train_df.iterrows(),
    total=len(train_df),
    desc="INUTS Mel Extraction",
    unit="file"
):
    save_path = os.path.join(output, f"{idx}.npy")

    if os.path.exists(save_path):
        skipped += 1
        continue

    try:
        audio, sr = sf.read(os.path.join(base, row["path"]))

        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        audio = librosa.resample(
            audio.astype(np.float32),
            orig_sr=sr,
            target_sr=24000
        )

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=24000,
            n_fft=1024,
            hop_length=256,
            n_mels=80
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        np.save(save_path, mel_db.astype(np.float32))
        success += 1

    except Exception as e:
        failed += 1
        errors.append((idx, str(e)))

    if (success + failed) % 500 == 0:
        print(
            f"\nCheckpoint: Success={success}, "
            f"Failed={failed}, Skipped={skipped}"
        )

print("\nEXTRACTION COMPLETE")
print("Successful:", success)
print("Failed:", failed)
print("Skipped:", skipped)
print("Total:", len(train_df))
print("Output:", output)

In [ ]:
import os

print("Working:", os.listdir("/kaggle/working"))
print("Dataset:", os.listdir("/kaggle/input/datasets"))

In [ ]:
import os
import zipfile
from tqdm.auto import tqdm

folder = "/kaggle/working/inuts_mels"
zip_path = "/kaggle/working/inuts_mels_backup.zip"

files = [
    f for f in os.listdir(folder)
    if f.endswith(".npy")
]

print("Mel files found:", len(files))

assert len(files) == 55117, "File count mismatch!"

with zipfile.ZipFile(
    zip_path, "w",
    compression=zipfile.ZIP_STORED
) as zf:

    for f in tqdm(files, desc="Creating backup", unit="file"):
        zf.write(
            os.path.join(folder, f),
            arcname=f
        )

print("\nBackup created!")
print("ZIP size:", round(os.path.getsize(zip_path) / (1024**3), 2), "GB")
print("Location:", zip_path)

In [ ]:
import zipfile

zip_path = "/kaggle/working/inuts_mels_backup.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:", len(z.namelist()))
    print("Integrity check:", z.testzip())

print("Kaggle runtime is working!")

In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

# Paths
BASE = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
CLIPS = os.path.join(BASE, "clips")
OUTPUT = "/kaggle/working/inuts_mels"

os.makedirs(OUTPUT, exist_ok=True)

# Check existing datasets
assert "validation_df" in globals(), "validation_df missing! Run split cell first."
assert "test_df" in globals(), "test_df missing! Run split cell first."

# Confirm no index overlap with training
train_indices = set(train_df.index)
val_indices = set(validation_df.index)
test_indices = set(test_df.index)

assert not (train_indices & val_indices), "Training/Validation overlap!"
assert not (train_indices & test_indices), "Training/Test overlap!"
assert not (val_indices & test_indices), "Validation/Test overlap!"

TARGET_SR = 24000
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 80

def extract_split(dataframe, split_name):
    success = 0
    skipped = 0
    failed = []

    progress = tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"{split_name} Mel Extraction",
        unit="audio"
    )

    for idx, row in progress:
        output_file = os.path.join(OUTPUT, f"{idx}.npy")

        # Resume safely if a feature already exists
        if os.path.isfile(output_file):
            skipped += 1
            continue

        try:
            audio_path = os.path.join(CLIPS, row["path"])

            audio, sr = sf.read(audio_path)

            # Stereo to mono
            if audio.ndim > 1:
                audio = audio.mean(axis=1)

            audio = audio.astype(np.float32)

            # Resample to 24 kHz
            if sr != TARGET_SR:
                audio = librosa.resample(
                    audio,
                    orig_sr=sr,
                    target_sr=TARGET_SR
                )

            # 80-band Mel spectrogram
            mel = librosa.feature.melspectrogram(
                y=audio,
                sr=TARGET_SR,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                n_mels=N_MELS
            )

            mel_db = librosa.power_to_db(mel, ref=np.max)

            np.save(output_file, mel_db.astype(np.float32))
            success += 1

        except Exception as e:
            failed.append({
                "index": idx,
                "path": row["path"],
                "error": str(e)
            })

        progress.set_postfix(
            success=success,
            skipped=skipped,
            failed=len(failed)
        )

    # Save failure report
    if failed:
        pd.DataFrame(failed).to_csv(
            f"/kaggle/working/{split_name.lower()}_failures.csv",
            index=False
        )

    print(f"\n{split_name} COMPLETE")
    print("Successful:", success)
    print("Already present:", skipped)
    print("Failed:", len(failed))
    print("Total:", len(dataframe))

    return success, skipped, failed


# Save dataset manifests
train_df.to_csv("/kaggle/working/inuts_train.csv", index=False)
validation_df.to_csv("/kaggle/working/inuts_validation.csv", index=False)
test_df.to_csv("/kaggle/working/inuts_test.csv", index=False)

# Extract validation and test
val_result = extract_split(validation_df, "Validation")
test_result = extract_split(test_df, "Test")

print("\nALL EXTRACTION TASKS FINISHED!")
print("Mel folder:", OUTPUT)


In [ ]:
import os
import numpy as np

folder = "/kaggle/working/inuts_mels"

files = [f for f in os.listdir(folder) if f.endswith(".npy")]

print("Total Mel files:", len(files))

assert len(files) == 64720, "Expected 64,720 files!"

# Check a few feature files
for f in files[:5]:
    data = np.load(os.path.join(folder, f))
    print(f, "Shape:", data.shape, "Finite:", np.isfinite(data).all())

print("\nALL CHECKS PASSED!")


In [ ]:
import os
import json
import zipfile
import pandas as pd
from tqdm.auto import tqdm

OUT = "/kaggle/working"
MEL_DIR = os.path.join(OUT, "inuts_mels")

# Confirm datasets exist
assert len(train_df) == 55117
assert len(validation_df) == 5495
assert len(test_df) == 4108

# 1. Create character vocabulary from training text
PAD = "<PAD>"
UNK = "<UNK>"

characters = sorted(set("".join(train_df["sentence"].astype(str))))

vocab = {
    PAD: 0,
    UNK: 1
}

for char in characters:
    if char not in vocab:
        vocab[char] = len(vocab)

with open(os.path.join(OUT, "inuts_urdu_vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

# 2. Create speaker mapping using training speakers
speaker_names = sorted(train_df["client_id"].astype(str).unique())

speaker_map = {
    name: idx for idx, name in enumerate(speaker_names)
}

with open(os.path.join(OUT, "inuts_speakers.json"), "w", encoding="utf-8") as f:
    json.dump(speaker_map, f, ensure_ascii=False, indent=2)

# 3. Save manifests with mel_id
for name, frame in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    manifest = frame.copy()
    manifest.insert(0, "mel_id", manifest.index.astype(str) + ".npy")
    manifest.to_csv(
        os.path.join(OUT, f"inuts_{name}_manifest.csv"),
        index=False
    )

# 4. Create separate validation + test backup
backup_path = os.path.join(OUT, "inuts_validation_test_backup.zip")

rows = pd.concat([validation_df, test_df])

with zipfile.ZipFile(
    backup_path, "w", compression=zipfile.ZIP_STORED
) as z:
    for idx in tqdm(rows.index, desc="Backing up validation/test"):
        filename = f"{idx}.npy"
        source = os.path.join(MEL_DIR, filename)
        assert os.path.isfile(source), f"Missing: {filename}"
        z.write(source, arcname=filename)

print("\nPREPARATION COMPLETE")
print("Vocabulary size:", len(vocab))
print("Training speakers:", len(speaker_map))
print("Validation + Test files:", len(rows))
print("Backup size (GB):", round(os.path.getsize(backup_path) / 1024**3, 2))
print("Backup:", backup_path)


In [ ]:
import os
import zipfile

OUT = "/kaggle/working"

# Verify validation/test backup
backup = os.path.join(
    OUT, "inuts_validation_test_backup.zip"
)

with zipfile.ZipFile(backup, "r") as z:
    print("Validation/Test ZIP files:", len(z.namelist()))
    print("Integrity:", z.testzip())

# Verify metadata files
metadata_files = [
    "inuts_urdu_vocab.json",
    "inuts_speakers.json",
    "inuts_train_manifest.csv",
    "inuts_validation_manifest.csv",
    "inuts_test_manifest.csv"
]

for filename in metadata_files:
    path = os.path.join(OUT, filename)
    assert os.path.isfile(path), f"Missing: {filename}"
    print(filename, round(os.path.getsize(path) / 1024, 2), "KB")

print("\nMETADATA VERIFICATION PASSED!")


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torchaudio
import librosa
import numpy as np

print("GPU:", torch.cuda.get_device_name(0))
print("GPU Memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

print("Torch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Librosa:", librosa.__version__)

print("CUDA test:", torch.randn(1000, 1000, device="cuda").mean().item())

print("\nLIBRARIES CHECK COMPLETE")


In [ ]:
import os
import json
import numpy as np
import librosa
from tqdm.auto import tqdm

BASE = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
CLIPS = os.path.join(BASE, "clips")

# Load vocabulary
with open("/kaggle/working/inuts_urdu_vocab.json", encoding="utf-8") as f:
    vocab = json.load(f)

print("Vocabulary:", len(vocab))
print("Training speakers:", train_df["client_id"].nunique())

# Check unseen characters in validation and test
def find_unseen_chars(frame):
    chars = set("".join(frame["sentence"].astype(str)))
    return sorted(chars - set(vocab.keys()))

print("\nValidation unseen characters:", find_unseen_chars(validation_df))
print("Test unseen characters:", find_unseen_chars(test_df))

# Check unseen speakers
train_speakers = set(train_df["client_id"].astype(str))
val_speakers = set(validation_df["client_id"].astype(str))
test_speakers = set(test_df["client_id"].astype(str))

print("\nValidation unseen speakers:", len(val_speakers - train_speakers))
print("Test unseen speakers:", len(test_speakers - train_speakers))

# Check actual original audio from all three splits
for split_name, frame in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    row = frame.iloc[0]
    path = os.path.join(CLIPS, row["path"])

    audio, sr = librosa.load(path, sr=24000, mono=True)

    print(f"\n{split_name} sample:")
    print("Text:", row["sentence"])
    print("Audio rate:", 24000)
    print("Duration:", round(len(audio) / 24000, 2), "seconds")
    print("Samples:", len(audio))
    print("RMS:", round(float(np.sqrt(np.mean(audio**2))), 5))
    print("Peak:", round(float(np.max(np.abs(audio))), 5))
    print("Finite:", bool(np.isfinite(audio).all()))

print("\nCOMPATIBILITY CHECK COMPLETE")


> **Legacy cell skipped:** Legacy duplicate data-loader setup depended on vocabulary files before the notebook generated them. The canonical data pipeline is restored in cells 125–128.


In [ ]:
import os

print("WORKING DIRECTORY:")
print(os.listdir("/kaggle/working"))

print("\nINPUT DIRECTORIES:")
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = dirs[:]
    matching = [
        f for f in files
        if f.endswith((".json", ".csv", ".zip"))
        and ("inuts" in f.lower())
    ]
    if matching:
        print(root)
        for f in matching:
            print("  ", f)


In [ ]:
import os
import pandas as pd
import json

BASE = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
OUTPUT = "/kaggle/working"

print("Dataset:", os.path.exists(BASE))
print("Clips:", os.path.exists(os.path.join(BASE, "clips")))
print("Validated TSV:", os.path.exists(os.path.join(BASE, "validated.tsv")))

df = pd.read_csv(
    os.path.join(BASE, "validated.tsv"),
    sep="\t",
    on_bad_lines="skip"
)

print("Total records:", len(df))
print("Dataset ready!")


In [ ]:
import os, json, unicodedata
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

OUTPUT = "/kaggle/working"
CLIPS = os.path.join(BASE, "clips")

tqdm.pandas(desc="Checking audio files")

print("Cleaning Urdu text...", flush=True)

df = df.dropna(subset=["client_id", "path", "sentence"]).copy()
df["sentence"] = df["sentence"].astype(str).progress_apply(
    lambda x: unicodedata.normalize("NFC", x)
    .replace("ـ", "")
)
df["sentence"] = df["sentence"].str.replace(
    r"\s+", " ", regex=True
).str.strip()

df = df[df["sentence"] != ""].copy()

print("\nChecking audio files:", flush=True)

df["audio_exists"] = df["path"].progress_apply(
    lambda x: os.path.isfile(os.path.join(CLIPS, x))
)

df = df[df["audio_exists"]].drop(
    columns=["audio_exists"]
).reset_index(drop=True)

print("\nCreating speaker split...", flush=True)

speakers = df["client_id"].unique().copy()
rng = np.random.default_rng(42)
rng.shuffle(speakers)

train_speakers = set(speakers[:330])
val_speakers = set(speakers[330:372])
test_speakers = set(speakers[372:])

train_df = df[df["client_id"].isin(train_speakers)].copy()
validation_df = df[df["client_id"].isin(val_speakers)].copy()
test_df = df[df["client_id"].isin(test_speakers)].copy()

print("Saving CSV files...", flush=True)

train_df.to_csv(f"{OUTPUT}/inuts_train.csv", index=False)
validation_df.to_csv(f"{OUTPUT}/inuts_validation.csv", index=False)
test_df.to_csv(f"{OUTPUT}/inuts_test.csv", index=False)

print("Building vocabulary...", flush=True)

characters = sorted(set("".join(train_df["sentence"])))

vocab = {"<PAD>": 0, "<UNK>": 1}
for char in tqdm(characters, desc="Vocabulary"):
    if char not in vocab:
        vocab[char] = len(vocab)

speaker_map = {
    str(s): i for i, s in enumerate(sorted(train_speakers))
}

with open(f"{OUTPUT}/inuts_urdu_vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

with open(f"{OUTPUT}/inuts_speakers.json", "w", encoding="utf-8") as f:
    json.dump(speaker_map, f, ensure_ascii=False, indent=2)

assert not train_speakers & val_speakers
assert not train_speakers & test_speakers
assert not val_speakers & test_speakers

print("\nSUCCESS!")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Vocabulary:", len(vocab))
print("Speakers:", len(speaker_map))


In [ ]:
import os
import librosa
import numpy as np

SAMPLE_RATE = 24000

samples = [
    ("Training", train_df.iloc[0]),
    ("Validation", validation_df.iloc[0]),
    ("Testing", test_df.iloc[0])
]

for name, row in samples:
    path = os.path.join(CLIPS, row["path"])

    audio, sr = librosa.load(
        path,
        sr=SAMPLE_RATE,
        mono=True
    )

    duration = len(audio) / SAMPLE_RATE
    rms = np.sqrt(np.mean(audio ** 2))

    print(f"\n{name}")
    print("Text:", row["sentence"][:80])
    print("Sample rate:", SAMPLE_RATE)
    print("Duration:", round(duration, 2), "seconds")
    print("RMS:", round(float(rms), 5))
    print("Peak:", round(float(np.max(np.abs(audio))), 5))
    print("Valid:", np.isfinite(audio).all())

    assert len(audio) > 0
    assert np.isfinite(audio).all()

print("\nALL 3 AUDIO TESTS PASSED!")


In [ ]:
import torch
import numpy as np
import librosa
import os

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SAMPLE_RATE = 24000
BATCH_SIZE = 2
PAD_ID = vocab["<PAD>"]
UNK_ID = vocab["<UNK>"]
BLANK_ID = len(vocab)
NUM_SYMBOLS = len(vocab) + 1

print("Preparing text encoder...", flush=True)

def encode_text(text):
    ids = [vocab.get(char, UNK_ID) for char in str(text)]

    expanded = [BLANK_ID]
    for token in ids:
        expanded.extend([token, BLANK_ID])

    return torch.tensor(expanded, dtype=torch.long)


print("Building waveform dataset...", flush=True)

class UrduTTSDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        path = os.path.join(CLIPS, row["path"])

        audio, _ = librosa.load(
            path,
            sr=SAMPLE_RATE,
            mono=True
        )

        if len(audio) == 0 or not np.isfinite(audio).all():
            raise ValueError(f"Invalid audio: {path}")

        return {
            "text": encode_text(row["sentence"]),
            "audio": torch.tensor(audio, dtype=torch.float32),
            "text_length": len(encode_text(row["sentence"])),
            "audio_length": len(audio)
        }


def collate_fn(batch):

    texts = [x["text"] for x in batch]
    audios = [x["audio"] for x in batch]

    text_lengths = torch.tensor(
        [len(x) for x in texts],
        dtype=torch.long
    )

    audio_lengths = torch.tensor(
        [len(x) for x in audios],
        dtype=torch.long
    )

    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=PAD_ID
    )

    max_audio = max(len(x) for x in audios)

    audio_batch = torch.zeros(
        len(audios), 1, max_audio
    )

    for i, audio in enumerate(audios):
        audio_batch[i, 0, :len(audio)] = audio

    return {
        "text": texts,
        "text_lengths": text_lengths,
        "audio": audio_batch,
        "audio_lengths": audio_lengths
    }


train_dataset = UrduTTSDataset(train_df)
validation_dataset = UrduTTSDataset(validation_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("Loading first training batch...", flush=True)

batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, tuple(value.shape), flush=True)

print("\nVocabulary symbols:", NUM_SYMBOLS)
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("DATA LOADER READY!")


In [ ]:
import torch
import torch.nn as nn
import math

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class SinusoidalPositionEncoding(nn.Module):
    def __init__(self, hidden_dim, max_length=4096):
        super().__init__()

        position = torch.arange(max_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, hidden_dim, 2) *
            (-math.log(10000.0) / hidden_dim)
        )

        pe = torch.zeros(1, max_length, hidden_dim)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class UrduTextEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_dim=192,
        num_heads=2,
        num_layers=4,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_dim
        )

        self.position = SinusoidalPositionEncoding(hidden_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=768,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, text, lengths):
        padding_mask = (
            torch.arange(text.size(1), device=text.device)[None, :]
            >= lengths[:, None]
        )

        x = self.embedding(text) * math.sqrt(
            self.embedding.embedding_dim
        )

        x = self.position(x)

        x = self.transformer(
            x,
            src_key_padding_mask=padding_mask
        )

        return self.norm(x)


text_encoder = UrduTextEncoder(
    vocab_size=NUM_SYMBOLS
).to(DEVICE)

text = batch["text"].to(DEVICE)
text_lengths = batch["text_lengths"].to(DEVICE)

with torch.no_grad():
    encoded = text_encoder(text, text_lengths)

print("Device:", DEVICE)
print("Input shape:", tuple(text.shape))
print("Encoder output:", tuple(encoded.shape))
print("Parameters:", sum(
    p.numel() for p in text_encoder.parameters()
))
print("Finite values:", torch.isfinite(encoded).all().item())

assert encoded.shape == (2, 111, 192)
assert torch.isfinite(encoded).all()

print("\nURDU TEXT ENCODER TEST PASSED!")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AlignmentScorer(nn.Module):
    def __init__(self, hidden_dim=192, audio_dim=192):
        super().__init__()

        self.text_projection = nn.Linear(hidden_dim, audio_dim)
        self.audio_projection = nn.Conv1d(
            audio_dim, audio_dim, kernel_size=1
        )
        self.scale = audio_dim ** -0.5

    def forward(self, text_features, audio_features,
                text_lengths, audio_lengths):

        text = self.text_projection(text_features)
        audio = self.audio_projection(audio_features).transpose(1, 2)

        scores = torch.matmul(
            text, audio.transpose(1, 2)
        ) * self.scale

        text_mask = (
            torch.arange(scores.size(1), device=scores.device)[None, :]
            >= text_lengths[:, None]
        )

        audio_mask = (
            torch.arange(scores.size(2), device=scores.device)[None, :]
            >= audio_lengths[:, None]
        )

        mask = text_mask[:, :, None] | audio_mask[:, None, :]
        scores = scores.masked_fill(mask, -1e4)

        return scores

alignment_scorer = AlignmentScorer().to(DEVICE)

# Temporary shape test: audio features are dummy tensors.
# Actual audio encoder will replace these in the next stage.
dummy_audio = torch.randn(
    text.size(0), 192, 120, device=DEVICE
)

dummy_audio_lengths = torch.tensor(
    [120, 100], device=DEVICE
)

scores = alignment_scorer(
    encoded,
    dummy_audio,
    text_lengths,
    dummy_audio_lengths
)

print("Alignment scores:", tuple(scores.shape))
print("Finite values:", torch.isfinite(scores).all().item())
assert scores.shape == (2, 111, 120)
assert torch.isfinite(scores).all()

print("ALIGNMENT SCORER TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn

class UrduAudioEncoder(nn.Module):
    def __init__(self, hidden_dim=192):
        super().__init__()

        self.input_layer = nn.Conv1d(
            1, 64, kernel_size=7, padding=3
        )

        self.encoder = nn.Sequential(
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.GELU(),
            nn.Conv1d(128, hidden_dim, kernel_size=5, stride=2, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GELU()
        )

        self.norm = nn.BatchNorm1d(hidden_dim)

    def forward(self, audio):
        x = self.input_layer(audio)
        x = self.encoder(x)
        return self.norm(x)


audio_encoder = UrduAudioEncoder().to(DEVICE)

audio = batch["audio"].to(DEVICE)

with torch.no_grad():
    audio_features = audio_encoder(audio)

print("Device:", DEVICE)
print("Input audio:", tuple(audio.shape))
print("Audio features:", tuple(audio_features.shape))
print("Parameters:", sum(p.numel() for p in audio_encoder.parameters()))
print("Finite values:", torch.isfinite(audio_features).all().item())

assert audio_features.shape[1] == 192
assert torch.isfinite(audio_features).all()

print("\nURDU AUDIO ENCODER TEST PASSED!")

In [ ]:
class AudioDownsampler(nn.Module):
    def __init__(self, channels=192):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(channels, channels, 7, stride=2, padding=3),
            nn.GELU(),
            nn.Conv1d(channels, channels, 7, stride=2, padding=3),
            nn.GELU(),
            nn.Conv1d(channels, channels, 7, stride=2, padding=3),
            nn.GELU(),
            nn.Conv1d(channels, channels, 7, stride=2, padding=3),
            nn.GELU()
        )

    def forward(self, x):
        return self.layers(x)

audio_downsampler = AudioDownsampler().to(DEVICE)

with torch.no_grad():
    reduced_audio = audio_downsampler(audio_features)

print("Before:", tuple(audio_features.shape))
print("After:", tuple(reduced_audio.shape))
print("Finite:", torch.isfinite(reduced_audio).all().item())

assert reduced_audio.shape[1] == 192
assert reduced_audio.shape[2] < audio_features.shape[2]
assert torch.isfinite(reduced_audio).all()

print("DOWNSAMPLER TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn
import librosa
import numpy as np

class UrduPosteriorEncoder(nn.Module):
    def __init__(self, hidden_dim=192, latent_dim=192):
        super().__init__()

        self.pre = nn.Conv1d(80, hidden_dim, kernel_size=1)

        self.encoder = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU()
        )

        self.mean = nn.Conv1d(hidden_dim, latent_dim, 1)
        self.log_scale = nn.Conv1d(hidden_dim, latent_dim, 1)

    def forward(self, spectrogram):
        x = self.pre(spectrogram)
        x = self.encoder(x)

        mean = self.mean(x)
        log_scale = self.log_scale(x).clamp(-7, 2)

        return mean, log_scale


posterior_encoder = UrduPosteriorEncoder().to(DEVICE)

# Test spectrogram input
dummy_spectrogram = torch.randn(
    2, 80, 500, device=DEVICE
)

with torch.no_grad():
    z_mean, z_log_scale = posterior_encoder(dummy_spectrogram)

print("Input:", tuple(dummy_spectrogram.shape))
print("Latent mean:", tuple(z_mean.shape))
print("Latent log scale:", tuple(z_log_scale.shape))
print("Finite:", torch.isfinite(z_mean).all().item()
      and torch.isfinite(z_log_scale).all().item())

assert z_mean.shape == (2, 192, 500)
assert z_log_scale.shape == (2, 192, 500)
assert torch.isfinite(z_mean).all()
assert torch.isfinite(z_log_scale).all()

print("\nPOSTERIOR ENCODER TEST PASSED!")

In [ ]:
class LatentSampler(nn.Module):
    def forward(self, mean, log_scale):
        noise = torch.randn_like(mean)
        z = mean + noise * torch.exp(log_scale)
        return z

latent_sampler = LatentSampler().to(DEVICE)

with torch.no_grad():
    z = latent_sampler(z_mean, z_log_scale)

print("Latent shape:", tuple(z.shape))
print("Finite:", torch.isfinite(z).all().item())

assert z.shape == z_mean.shape
assert torch.isfinite(z).all()

print("\nLATENT SAMPLING TEST PASSED!")

In [ ]:
class ResidualCouplingBlock(nn.Module):
    def __init__(self, channels=192):
        super().__init__()

        self.half = channels // 2

        self.net = nn.Sequential(
            nn.Conv1d(self.half, 256, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(256, 256, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(256, channels, 1)
        )

    def forward(self, x):
        x1, x2 = x.split(self.half, dim=1)

        stats = self.net(x1)
        mean, log_scale = stats.chunk(2, dim=1)
        log_scale = torch.tanh(log_scale)

        y2 = x2 * torch.exp(log_scale) + mean

        return torch.cat([x1, y2], dim=1)


class UrduNormalizingFlow(nn.Module):
    def __init__(self, channels=192, layers=4):
        super().__init__()
        self.blocks = nn.ModuleList([
            ResidualCouplingBlock(channels)
            for _ in range(layers)
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
            x = torch.flip(x, dims=[1])
        return x


normalizing_flow = UrduNormalizingFlow().to(DEVICE)

with torch.no_grad():
    flow_output = normalizing_flow(z)

print("Input:", tuple(z.shape))
print("Output:", tuple(flow_output.shape))
print("Finite:", torch.isfinite(flow_output).all().item())

assert flow_output.shape == z.shape
assert torch.isfinite(flow_output).all()

print("\nNORMALIZING FLOW TEST PASSED!")

In [ ]:
class UrduDurationPredictor(nn.Module):
    def __init__(self, hidden_dim=192):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1),
            nn.GELU(),
            nn.GroupNorm(8, hidden_dim),

            nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1),
            nn.GELU(),
            nn.GroupNorm(8, hidden_dim),
        )

        self.duration = nn.Conv1d(hidden_dim, 1, 1)

    def forward(self, text_features):
        x = text_features.transpose(1, 2)
        x = self.network(x)
        return self.duration(x).squeeze(1)


duration_predictor = UrduDurationPredictor().to(DEVICE)

with torch.no_grad():
    duration_output = duration_predictor(encoded)

print("Text features:", tuple(encoded.shape))
print("Predicted durations:", tuple(duration_output.shape))
print("Finite:", torch.isfinite(duration_output).all().item())

assert duration_output.shape == (2, 111)
assert torch.isfinite(duration_output).all()

print("\nDURATION PREDICTOR TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn

class UrduWaveformDecoder(nn.Module):
    def __init__(self, latent_dim=192):
        super().__init__()

        self.pre = nn.Conv1d(latent_dim, 256, 7, padding=3)

        self.upsample = nn.Sequential(
            nn.ConvTranspose1d(256, 128, 16, stride=4, padding=6),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(128, 64, 16, stride=4, padding=6),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(64, 32, 16, stride=4, padding=6),
            nn.LeakyReLU(0.1),
            nn.Conv1d(32, 1, 7, padding=3),
            nn.Tanh()
        )

    def forward(self, latent):
        x = self.pre(latent)
        return self.upsample(x)


waveform_decoder = UrduWaveformDecoder().to(DEVICE)

with torch.no_grad():
    generated_audio = waveform_decoder(z)

print("Latent input:", tuple(z.shape))
print("Generated waveform:", tuple(generated_audio.shape))
print("Finite:", torch.isfinite(generated_audio).all().item())

assert generated_audio.shape[0] == z.shape[0]
assert generated_audio.shape[1] == 1
assert torch.isfinite(generated_audio).all()

print("\nWAVEFORM DECODER TEST PASSED!")

In [ ]:
import torch
import torchaudio

# 24 kHz Urdu audio se 80-channel mel spectrogram
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=24000,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    n_mels=80,
    power=1.0
).to(DEVICE)

# Actual training batch
real_audio = batch["audio"].to(DEVICE)

# [B, 1, T] -> [B, 80, frames]
with torch.no_grad():
    mel = mel_transform(real_audio.squeeze(1))
    log_mel = torch.log(mel.clamp_min(1e-5))

    # Actual recordings through Posterior Encoder
    real_mean, real_log_scale = posterior_encoder(log_mel)

print("Original audio:", tuple(real_audio.shape))
print("Log-mel spectrogram:", tuple(log_mel.shape))
print("Posterior mean:", tuple(real_mean.shape))
print("Posterior log scale:", tuple(real_log_scale.shape))
print("All finite:", all([
    torch.isfinite(log_mel).all().item(),
    torch.isfinite(real_mean).all().item(),
    torch.isfinite(real_log_scale).all().item()
]))

assert log_mel.shape[1] == 80
assert real_mean.shape[1] == 192
assert real_mean.shape == real_log_scale.shape
assert torch.isfinite(real_mean).all()
assert torch.isfinite(real_log_scale).all()

print("\nREAL AUDIO POSTERIOR TEST PASSED!")

In [ ]:
# Actual text and audio lengths
text_lengths_real = batch["text_lengths"].to(DEVICE)

audio_lengths_real = torch.full(
    (real_audio.size(0),),
    log_mel.size(2),
    device=DEVICE,
    dtype=torch.long
)

# Build text and audio masks
text_mask = (
    torch.arange(encoded.size(1), device=DEVICE)[None, :]
    < text_lengths_real[:, None]
)

audio_mask = (
    torch.arange(log_mel.size(2), device=DEVICE)[None, :]
    < audio_lengths_real[:, None]
)

# Alignment scoring
alignment_scores = alignment_scorer(
    encoded,
    real_mean,
    text_lengths_real,
    audio_lengths_real
)

print("Text mask:", tuple(text_mask.shape))
print("Audio mask:", tuple(audio_mask.shape))
print("Alignment scores:", tuple(alignment_scores.shape))
print("Finite:", torch.isfinite(alignment_scores).all().item())

assert alignment_scores.shape == (
    real_audio.size(0),
    encoded.size(1),
    log_mel.size(2)
)

assert torch.isfinite(alignment_scores).all()

print("\nREAL TEXT-AUDIO ALIGNMENT TEST PASSED!")

In [ ]:
import torch

def monotonic_alignment_search(scores, text_lengths, audio_lengths):
    """
    scores: [B, text_frames, audio_frames]
    Returns hard monotonic alignment [B, text_frames, audio_frames]
    """
    B, T, S = scores.shape
    alignment = torch.zeros_like(scores)

    for b in range(B):
        t_len = int(text_lengths[b].item())
        s_len = int(audio_lengths[b].item())

        dp = torch.full(
            (t_len, s_len), -1e9, device=scores.device
        )
        back = torch.zeros(
            (t_len, s_len), dtype=torch.long, device=scores.device
        )

        dp[0, 0] = scores[b, 0, 0]

        for j in range(1, s_len):
            for i in range(t_len):
                stay = dp[i, j-1]
                advance = dp[i-1, j-1] if i > 0 else -1e9

                if stay >= advance:
                    dp[i, j] = stay + scores[b, i, j]
                    back[i, j] = i
                else:
                    dp[i, j] = advance + scores[b, i, j]
                    back[i, j] = i-1

        i = t_len - 1

        for j in range(s_len-1, -1, -1):
            alignment[b, i, j] = 1
            if j > 0:
                i = back[i, j].item()

    return alignment


with torch.no_grad():
    mas_alignment = monotonic_alignment_search(
        alignment_scores,
        text_lengths_real,
        audio_lengths_real
    )

print("Alignment shape:", tuple(mas_alignment.shape))
print("Total aligned frames:", mas_alignment.sum().item())
print("Finite:", torch.isfinite(mas_alignment).all().item())

assert mas_alignment.shape == alignment_scores.shape
assert torch.all(mas_alignment.sum(dim=1) == audio_mask.long())
assert torch.isfinite(mas_alignment).all()

print("\nMONOTONIC ALIGNMENT TEST PASSED!")

In [ ]:
# Har text token ke liye audio frames ki tadaad
duration_targets = mas_alignment.sum(dim=2)

# Sirf valid text tokens ki durations
duration_targets = duration_targets * text_mask.float()

print("Duration shape:", tuple(duration_targets.shape))
print("Sample 1 durations:", duration_targets[0, :20].int().tolist())
print("Sample 2 durations:", duration_targets[1, :20].int().tolist())

# Check: har audio frame kisi token ko assign hua?
duration_totals = duration_targets.sum(dim=1)

print("Audio frames:", audio_lengths_real.tolist())
print("Assigned frames:", duration_totals.int().tolist())

assert torch.all(duration_totals == audio_lengths_real.float())
assert torch.all(duration_targets >= 0)

print("\nDURATION EXTRACTION TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn
import math

class UrduTextPrior(nn.Module):
    def __init__(self, hidden_dim=192, latent_dim=192):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU()
        )

        self.mean = nn.Conv1d(hidden_dim, latent_dim, 1)
        self.log_scale = nn.Conv1d(hidden_dim, latent_dim, 1)

    def forward(self, text_features):
        x = text_features.transpose(1, 2)
        x = self.network(x)

        mean = self.mean(x)
        log_scale = self.log_scale(x).clamp(-7, 2)

        return mean, log_scale


# Create the text prior
text_prior = UrduTextPrior().to(DEVICE)

# Predict prior statistics
with torch.no_grad():
    prior_mean, prior_log_scale = text_prior(encoded)

    # Provisional Gaussian compatibility scores
    z = real_mean.unsqueeze(2)
    mu = prior_mean.unsqueeze(3)
    log_s = prior_log_scale.unsqueeze(3)

    gaussian_scores = -0.5 * (
        ((z - mu) * torch.exp(-log_s)) ** 2
        + 2 * log_s
        + math.log(2 * math.pi)
    ).sum(dim=1)

    # Apply text and audio masks
    valid_mask = text_mask.unsqueeze(2) & audio_mask.unsqueeze(1)
    gaussian_scores = gaussian_scores.masked_fill(
        ~valid_mask, -1e4
    )

print("Prior mean:", tuple(prior_mean.shape))
print("Prior log scale:", tuple(prior_log_scale.shape))
print("Gaussian scores:", tuple(gaussian_scores.shape))
print("Finite:", torch.isfinite(gaussian_scores).all().item())

assert prior_mean.shape == encoded.transpose(1, 2).shape
assert gaussian_scores.shape == alignment_scores.shape
assert torch.isfinite(gaussian_scores).all()

print("\nTEXT PRIOR TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn

class AffineCoupling(nn.Module):
    def __init__(self, channels=192, hidden=256):
        super().__init__()

        assert channels % 2 == 0
        half = channels // 2

        self.net = nn.Sequential(
            nn.Conv1d(half, hidden, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden, channels, 1)
        )

        # Start as identity transformation
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)

        mean, raw_scale = self.net(x1).chunk(2, dim=1)
        log_scale = 2.0 * torch.tanh(raw_scale / 2.0)

        y2 = x2 * torch.exp(log_scale) + mean
        y = torch.cat([x1, y2], dim=1)

        logdet = log_scale.sum(dim=(1, 2))
        return y, logdet

    def inverse(self, y):
        y1, y2 = y.chunk(2, dim=1)

        mean, raw_scale = self.net(y1).chunk(2, dim=1)
        log_scale = 2.0 * torch.tanh(raw_scale / 2.0)

        x2 = (y2 - mean) * torch.exp(-log_scale)
        return torch.cat([y1, x2], dim=1)


class UrduInvertibleFlow(nn.Module):
    def __init__(self, channels=192, layers=4):
        super().__init__()
        self.blocks = nn.ModuleList([
            AffineCoupling(channels) for _ in range(layers)
        ])

    def forward(self, x):
        total_logdet = torch.zeros(
            x.size(0), device=x.device, dtype=x.dtype
        )

        for block in self.blocks:
            x, logdet = block(x)
            total_logdet += logdet
            x = torch.flip(x, dims=[1])

        return x, total_logdet

    def inverse(self, z):
        x = z

        for block in reversed(self.blocks):
            x = torch.flip(x, dims=[1])
            x = block.inverse(x)

        return x


# Create flow
normalizing_flow = UrduInvertibleFlow().to(DEVICE)

# Test invertibility
test_latent = torch.randn(2, 192, 763, device=DEVICE)

with torch.no_grad():
    transformed, flow_logdet = normalizing_flow(test_latent)
    reconstructed = normalizing_flow.inverse(transformed)

    reconstruction_error = (
        test_latent - reconstructed
    ).abs().max().item()

print("Input:", tuple(test_latent.shape))
print("Transformed:", tuple(transformed.shape))
print("Log determinant:", tuple(flow_logdet.shape))
print("Reconstruction error:", reconstruction_error)
print("Finite:", torch.isfinite(transformed).all().item())

assert transformed.shape == test_latent.shape
assert flow_logdet.shape == (2,)
assert torch.isfinite(transformed).all()
assert reconstruction_error < 1e-5

print("\nINVERTIBLE FLOW TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn

class UrduSpeakerEmbedding(nn.Module):
    def __init__(self, num_speakers, embedding_dim=192):
        super().__init__()

        self.embedding = nn.Embedding(
            num_speakers, embedding_dim
        )

        self.projection = nn.Linear(
            embedding_dim, embedding_dim
        )

    def forward(self, speaker_ids):
        x = self.embedding(speaker_ids)
        return self.projection(x)


# Training speakers ki actual count
NUM_SPEAKERS = len(speaker_map)

speaker_encoder = UrduSpeakerEmbedding(
    num_speakers=NUM_SPEAKERS,
    embedding_dim=192
).to(DEVICE)

# Basic tensor test
test_speaker_ids = torch.tensor(
    [0, 1], dtype=torch.long, device=DEVICE
)

with torch.no_grad():
    speaker_features = speaker_encoder(test_speaker_ids)

print("Training speakers:", NUM_SPEAKERS)
print("Speaker features:", tuple(speaker_features.shape))
print("Finite:", torch.isfinite(speaker_features).all().item())

assert speaker_features.shape == (2, 192)
assert torch.isfinite(speaker_features).all()

print("\nSPEAKER EMBEDDING TEST PASSED!")

In [ ]:
# Actual training speaker IDs ko dataset se encode karna
speaker_to_id = {
    str(speaker): idx
    for speaker, idx in speaker_map.items()
}

train_df["speaker_id"] = (
    train_df["client_id"]
    .astype(str)
    .map(speaker_to_id)
)

validation_df["speaker_id"] = (
    validation_df["client_id"]
    .astype(str)
    .map(speaker_to_id)
)

test_df["speaker_id"] = (
    test_df["client_id"]
    .astype(str)
    .map(speaker_to_id)
)

print("Training missing IDs:", train_df["speaker_id"].isna().sum())
print("Validation unseen speakers:", validation_df["speaker_id"].isna().sum())
print("Test unseen speakers:", test_df["speaker_id"].isna().sum())

print("Training unique speakers:", train_df["speaker_id"].nunique())

assert train_df["speaker_id"].notna().all()
assert train_df["speaker_id"].nunique() == 330
assert validation_df["speaker_id"].isna().all()
assert test_df["speaker_id"].isna().all()

# Save mapping and updated CSV
train_df.to_csv("/kaggle/working/inuts_train.csv", index=False)
validation_df.to_csv("/kaggle/working/inuts_validation.csv", index=False)
test_df.to_csv("/kaggle/working/inuts_test.csv", index=False)

print("\nREAL SPEAKER MAPPING TEST PASSED!")

In [ ]:
# Check whether current dataset returns speaker IDs

sample = train_loader.dataset[0]

print("Dataset output type:", type(sample))

if isinstance(sample, dict):
    print("Available keys:", list(sample.keys()))
else:
    print("Returned items:", len(sample))

print("\nDataFrame speaker_id exists:",
      "speaker_id" in train_df.columns)

In [ ]:
import os
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

BLANK_ID = len(vocab)
UNKNOWN_ID = vocab["<UNK>"]

def encode_urdu_text(sentence):
    ids = [vocab.get(ch, UNKNOWN_ID) for ch in str(sentence)]

    # Blank token between characters
    result = [BLANK_ID]
    for token in ids:
        result.extend([token, BLANK_ID])

    return torch.tensor(result, dtype=torch.long)


class UrduTTSSpeakerDataset(Dataset):
    def __init__(self, dataframe, clips_dir, speaker_mapping):
        self.df = dataframe.reset_index(drop=True)
        self.clips_dir = clips_dir
        self.speaker_mapping = speaker_mapping

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        audio_path = os.path.join(
            self.clips_dir, str(row["path"])
        )

        audio, sr = librosa.load(
            audio_path, sr=24000, mono=True
        )

        if not np.isfinite(audio).all():
            raise ValueError(f"Invalid audio: {audio_path}")

        text = encode_urdu_text(row["sentence"])

        speaker_id = self.speaker_mapping.get(
            str(row["client_id"]), -1
        )

        return {
            "text": text,
            "audio": torch.tensor(audio, dtype=torch.float32),
            "text_length": len(text),
            "audio_length": len(audio),
            "speaker_id": speaker_id
        }


def speaker_collate_fn(samples):
    texts = [x["text"] for x in samples]
    audios = [x["audio"] for x in samples]

    text_lengths = torch.tensor(
        [x["text_length"] for x in samples],
        dtype=torch.long
    )

    audio_lengths = torch.tensor(
        [x["audio_length"] for x in samples],
        dtype=torch.long
    )

    speaker_ids = torch.tensor(
        [x["speaker_id"] for x in samples],
        dtype=torch.long
    )

    texts = pad_sequence(
        texts, batch_first=True, padding_value=0
    )

    audios = pad_sequence(
        audios, batch_first=True, padding_value=0
    ).unsqueeze(1)

    return {
        "text": texts,
        "text_lengths": text_lengths,
        "audio": audios,
        "audio_lengths": audio_lengths,
        "speaker_ids": speaker_ids
    }


# Rebuild training dataset and loader
train_dataset = UrduTTSSpeakerDataset(
    train_df, CLIPS, speaker_to_id
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    collate_fn=speaker_collate_fn,
    pin_memory=True
)

# Test actual batch
speaker_batch = next(iter(train_loader))

for key, value in speaker_batch.items():
    if torch.is_tensor(value):
        print(key, tuple(value.shape), value.dtype)

assert speaker_batch["speaker_ids"].shape == (2,)
assert (speaker_batch["speaker_ids"] >= 0).all()
assert (speaker_batch["speaker_ids"] < NUM_SPEAKERS).all()

print("\nSPEAKER-AWARE DATALOADER PASSED!")

In [ ]:
import torch
import torch.nn as nn

# Fresh batch from speaker-aware DataLoader
text_input = speaker_batch["text"].to(DEVICE)
text_lengths = speaker_batch["text_lengths"].to(DEVICE)
speaker_ids = speaker_batch["speaker_ids"].to(DEVICE)

# Encode Urdu text
text_features = text_encoder(text_input, text_lengths)

# Encode speaker identity
speaker_features = speaker_encoder(speaker_ids)

# Add speaker information to every text token
conditioned_features = (
    text_features + speaker_features.unsqueeze(1)
)

# Predict speaker-conditioned latent statistics
with torch.no_grad():
    conditioned_mean, conditioned_log_scale = (
        text_prior(conditioned_features)
    )

# Mask padded text positions
valid_text = (
    torch.arange(text_input.size(1), device=DEVICE)[None, :]
    < text_lengths[:, None]
)

conditioned_mean = conditioned_mean * valid_text.unsqueeze(1)
conditioned_log_scale = (
    conditioned_log_scale * valid_text.unsqueeze(1)
)

print("Text features:", tuple(text_features.shape))
print("Speaker features:", tuple(speaker_features.shape))
print("Conditioned features:", tuple(conditioned_features.shape))
print("Prior mean:", tuple(conditioned_mean.shape))
print("Prior log scale:", tuple(conditioned_log_scale.shape))

assert conditioned_mean.shape == (2, 192, text_input.size(1))
assert torch.isfinite(conditioned_mean).all()
assert torch.isfinite(conditioned_log_scale).all()

print("\nSPEAKER-CONDITIONED PRIOR TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn
import torchaudio

# 1. Convert waveform into linear spectrogram
spectrogram_transform = torchaudio.transforms.Spectrogram(
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    power=1.0,
    center=True
).to(DEVICE)

audio_input = speaker_batch["audio"].to(DEVICE)
audio_lengths = speaker_batch["audio_lengths"].to(DEVICE)

linear_spec = spectrogram_transform(audio_input.squeeze(1))

# Actual valid spectrogram frames for each recording
audio_frame_lengths = torch.div(
    audio_lengths, 256, rounding_mode="floor"
) + 1

audio_frame_lengths = audio_frame_lengths.clamp(
    max=linear_spec.size(2)
)

# 2. Posterior encoder: 513 input bins -> 192 latent channels
class UrduPosteriorEncoderV2(nn.Module):
    def __init__(self, input_channels=513, hidden_dim=192, latent_dim=192):
        super().__init__()

        self.pre = nn.Conv1d(input_channels, hidden_dim, 1)

        self.encoder = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, 5, padding=2),
            nn.GELU()
        )

        self.mean = nn.Conv1d(hidden_dim, latent_dim, 1)
        self.log_scale = nn.Conv1d(hidden_dim, latent_dim, 1)

    def forward(self, spectrogram):
        x = self.pre(spectrogram)
        x = self.encoder(x)

        mean = self.mean(x)
        log_scale = self.log_scale(x).clamp(-7, 2)

        return mean, log_scale


posterior_encoder_v2 = UrduPosteriorEncoderV2().to(DEVICE)

# 3. Condition posterior representation on speaker identity
speaker_to_audio = nn.Linear(192, 192).to(DEVICE)

posterior_hidden = posterior_encoder_v2.pre(linear_spec)
posterior_hidden = posterior_encoder_v2.encoder(posterior_hidden)

speaker_audio = speaker_to_audio(speaker_features).unsqueeze(-1)
posterior_hidden = posterior_hidden + speaker_audio

posterior_mean = posterior_encoder_v2.mean(posterior_hidden)
posterior_log_scale = posterior_encoder_v2.log_scale(
    posterior_hidden
).clamp(-7, 2)

# Mask padded audio frames
audio_mask = (
    torch.arange(linear_spec.size(2), device=DEVICE)[None, :]
    < audio_frame_lengths[:, None]
)

posterior_mean = posterior_mean * audio_mask.unsqueeze(1)
posterior_log_scale = posterior_log_scale * audio_mask.unsqueeze(1)

print("Linear spectrogram:", tuple(linear_spec.shape))
print("Audio frame lengths:", audio_frame_lengths.tolist())
print("Posterior mean:", tuple(posterior_mean.shape))
print("Posterior log scale:", tuple(posterior_log_scale.shape))

assert linear_spec.shape[1] == 513
assert posterior_mean.shape[1] == 192
assert torch.isfinite(posterior_mean).all()
assert torch.isfinite(posterior_log_scale).all()

print("\nAUDIO POSTERIOR ENCODER PASSED!")

In [ ]:
import math
import torch

# 1. Sample latent representation from audio posterior
torch.manual_seed(42)

noise = torch.randn_like(posterior_mean)

posterior_z = (
    posterior_mean
    + noise * torch.exp(posterior_log_scale)
)

posterior_z = posterior_z * audio_mask.unsqueeze(1)

# 2. Apply invertible normalizing flow
latent_flow = UrduInvertibleFlow(
    channels=192,
    layers=4
).to(DEVICE)

flow_z, flow_logdet = latent_flow(posterior_z)

# 3. Obtain text prior using speaker-conditioned features
prior_mean, prior_log_scale = text_prior(conditioned_features)

# Convert to [B, channels, text_length]
prior_mean = prior_mean.transpose(1, 2).transpose(1, 2)
# Shape is already [B, 192, text_length]

# 4. Gaussian log-likelihood scores
z = flow_z.unsqueeze(2)
mu = prior_mean.unsqueeze(3)
log_scale = prior_log_scale.unsqueeze(3)

alignment_scores = -0.5 * (
    ((z - mu) * torch.exp(-log_scale)) ** 2
    + 2 * log_scale
    + math.log(2 * math.pi)
).sum(dim=1)

# 5. Exclude padded text and audio frames
text_mask = (
    torch.arange(text_input.size(1), device=DEVICE)[None, :]
    < text_lengths[:, None]
)

valid_alignment = (
    text_mask.unsqueeze(2)
    & audio_mask.unsqueeze(1)
)

alignment_scores = alignment_scores.masked_fill(
    ~valid_alignment, -1e4
)

print("Posterior latent:", tuple(posterior_z.shape))
print("Flow latent:", tuple(flow_z.shape))
print("Flow log determinant:", tuple(flow_logdet.shape))
print("Alignment scores:", tuple(alignment_scores.shape))
print("Valid audio frames:", audio_frame_lengths.tolist())

assert alignment_scores.shape == (
    2, text_input.size(1), linear_spec.size(2)
)
assert torch.isfinite(flow_z).all()
assert torch.isfinite(alignment_scores).all()

print("\nLATENT FLOW + ALIGNMENT SCORE TEST PASSED!")

In [ ]:
import torch

def monotonic_alignment_search(scores, text_lengths, audio_lengths):
    """
    scores: [B, T_text, T_audio]
    Returns:
        hard_alignment: [B, T_text, T_audio]
        durations: [B, T_text]
    """
    B, max_text, max_audio = scores.shape

    alignment = torch.zeros_like(scores)
    durations = torch.zeros(
        B, max_text, dtype=torch.long, device=scores.device
    )

    with torch.no_grad():
        for b in range(B):
            n = int(text_lengths[b].item())
            t = int(audio_lengths[b].item())

            if n < 1 or t < n:
                raise ValueError(
                    f"Invalid alignment lengths: text={n}, audio={t}"
                )

            s = scores[b, :n, :t].float()

            # Dynamic programming table
            dp = torch.full(
                (n, t), float("-inf"), device=scores.device
            )
            back = torch.zeros(
                (n, t), dtype=torch.bool, device=scores.device
            )

            dp[0, 0] = s[0, 0]

            for j in range(1, t):
                max_i = min(n - 1, j)

                for i in range(max_i + 1):
                    # Every remaining token must still get a frame
                    if n - 1 - i > t - 1 - j:
                        continue

                    stay = dp[i, j - 1]

                    if i > 0:
                        advance = dp[i - 1, j - 1]
                    else:
                        advance = torch.tensor(
                            float("-inf"), device=scores.device
                        )

                    if advance > stay:
                        dp[i, j] = advance + s[i, j]
                        back[i, j] = True
                    else:
                        dp[i, j] = stay + s[i, j]

            if not torch.isfinite(dp[n - 1, t - 1]):
                raise RuntimeError("No valid monotonic alignment found.")

            # Backtrack best path
            i = n - 1

            for j in range(t - 1, -1, -1):
                alignment[b, i, j] = 1.0

                if j > 0 and back[i, j]:
                    i -= 1

            durations[b, :n] = alignment[b, :n, :t].sum(dim=1).long()

    return alignment, durations


# Use the alignment scores already calculated by your model
hard_alignment, duration_targets = monotonic_alignment_search(
    alignment_scores,
    text_lengths,
    audio_frame_lengths
)

print("Alignment:", tuple(hard_alignment.shape))
print("Duration targets:", tuple(duration_targets.shape))
print("Aligned audio frames:", hard_alignment.sum(dim=(1, 2)).tolist())
print("Expected audio frames:", audio_frame_lengths.tolist())
print("Minimum token duration:", duration_targets.min(dim=1).values.tolist())
print("Maximum token duration:", duration_targets.max(dim=1).values.tolist())

assert torch.allclose(
    hard_alignment.sum(dim=1),
    (torch.arange(hard_alignment.size(2), device=DEVICE)[None, :]
     < audio_frame_lengths[:, None]).float()
)

assert torch.equal(
    duration_targets.sum(dim=1),
    audio_frame_lengths
)

print("\nMONOTONIC ALIGNMENT PASSED!")

In [ ]:
import torch
import torch.nn.functional as F

# Predict log durations
duration_predictions = duration_predictor(
    conditioned_features
)

# Convert MAS duration targets to log scale
duration_targets_log = torch.log(
    duration_targets.float() + 1e-6
)

# Ignore padded text positions
valid_tokens = (
    torch.arange(duration_targets.size(1), device=DEVICE)[None, :]
    < text_lengths[:, None]
)

duration_loss = F.mse_loss(
    duration_predictions[valid_tokens],
    duration_targets_log[valid_tokens]
)

print("Duration prediction shape:", tuple(duration_predictions.shape))
print("Duration target shape:", tuple(duration_targets_log.shape))
print("Duration loss:", duration_loss.item())

assert torch.isfinite(duration_loss)
assert duration_loss.requires_grad

print("\nDURATION TRAINING LOSS PASSED!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=1):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.block = nn.Sequential(
            nn.LeakyReLU(0.1),
            nn.Conv1d(
                channels, channels,
                kernel_size,
                padding=padding,
                dilation=dilation
            ),
            nn.LeakyReLU(0.1),
            nn.Conv1d(channels, channels, 1)
        )

    def forward(self, x):
        return x + self.block(x)


class UrduWaveformGenerator(nn.Module):
    def __init__(self, latent_dim=192):
        super().__init__()

        self.pre = nn.Conv1d(latent_dim, 256, 7, padding=3)

        channels = [256, 128, 64, 32, 16]
        strides = [8, 8, 2, 2]

        self.upsamplers = nn.ModuleList()
        self.resblocks = nn.ModuleList()

        for i, stride in enumerate(strides):
            self.upsamplers.append(
                nn.ConvTranspose1d(
                    channels[i],
                    channels[i + 1],
                    kernel_size=stride * 2,
                    stride=stride,
                    padding=stride // 2
                )
            )

            self.resblocks.append(
                ResidualBlock(channels[i + 1], dilation=1)
            )

        self.post = nn.Sequential(
            nn.LeakyReLU(0.1),
            nn.Conv1d(16, 1, 7, padding=3),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.pre(z)

        for upsample, residual in zip(
            self.upsamplers, self.resblocks
        ):
            x = upsample(x)
            x = residual(x)

        return self.post(x)


# Initialize generator
waveform_generator = UrduWaveformGenerator().to(DEVICE)

# Use posterior latent as a generator input
# This is a forward/gradient test, not trained speech generation.
z_test = posterior_mean.detach()

generated_audio = waveform_generator(z_test)

print("Latent input:", tuple(z_test.shape))
print("Generated waveform:", tuple(generated_audio.shape))
print("Expected upsampling factor: 256")
print("Actual upsampling factor:", generated_audio.shape[-1] // z_test.shape[-1])
print("Output finite:", torch.isfinite(generated_audio).all().item())

assert generated_audio.shape[1] == 1
assert generated_audio.shape[-1] == z_test.shape[-1] * 256
assert torch.isfinite(generated_audio).all()

# Check that gradients can flow through the generator
generator_loss_test = generated_audio.abs().mean()
generator_loss_test.backward()

assert waveform_generator.pre.weight.grad is not None

print("\nWAVEFORM GENERATOR + GRADIENT TEST PASSED!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PeriodDiscriminator(nn.Module):
    def __init__(self, period):
        super().__init__()
        self.period = period

        channels = [1, 32, 64, 128, 256]

        self.layers = nn.ModuleList([
            nn.Conv2d(
                channels[i], channels[i + 1],
                kernel_size=(5, 1),
                stride=(3, 1),
                padding=(2, 0)
            )
            for i in range(4)
        ])

        self.post = nn.Conv2d(
            256, 1, kernel_size=(3, 1), padding=(1, 0)
        )

    def forward(self, audio):
        B, C, T = audio.shape
        remainder = T % self.period

        if remainder:
            pad = self.period - remainder
            audio = F.pad(audio, (0, pad), mode="replicate")

        x = audio.reshape(
            B, C, audio.size(-1) // self.period, self.period
        )

        feature_maps = []

        for layer in self.layers:
            x = layer(x)
            x = F.leaky_relu(x, 0.1)
            feature_maps.append(x)

        x = self.post(x)
        feature_maps.append(x)

        return x.flatten(1), feature_maps


class UrduMultiPeriodDiscriminator(nn.Module):
    def __init__(self, periods=(2, 3, 5, 7, 11)):
        super().__init__()
        self.discriminators = nn.ModuleList([
            PeriodDiscriminator(p) for p in periods
        ])

    def forward(self, audio):
        scores = []
        all_features = []

        for discriminator in self.discriminators:
            score, features = discriminator(audio)
            scores.append(score)
            all_features.append(features)

        return scores, all_features


# Initialize discriminator
mpd = UrduMultiPeriodDiscriminator().to(DEVICE)

# Match real and generated waveform lengths
real_audio = speaker_batch["audio"].to(DEVICE)
fake_audio = generated_audio.detach()

common_length = min(
    real_audio.shape[-1],
    fake_audio.shape[-1]
)

real_audio = real_audio[..., :common_length]
fake_audio = fake_audio[..., :common_length]

# Discriminator forward passes
real_scores, real_features = mpd(real_audio)
fake_scores, fake_features = mpd(fake_audio)

# Hinge discriminator loss
discriminator_loss = 0

for real_score, fake_score in zip(real_scores, fake_scores):
    discriminator_loss += (
        F.relu(1 - real_score).mean()
        + F.relu(1 + fake_score).mean()
    )

discriminator_loss = discriminator_loss / len(real_scores)

print("Real waveform:", tuple(real_audio.shape))
print("Fake waveform:", tuple(fake_audio.shape))
print("Number of periods:", len(real_scores))

for i, score in enumerate(real_scores):
    print(
        f"Period {mpd.discriminators[i].period}: "
        f"real={tuple(score.shape)}, "
        f"fake={tuple(fake_scores[i].shape)}"
    )

print("Discriminator loss:", discriminator_loss.item())

assert len(real_scores) == 5
assert torch.isfinite(discriminator_loss)

# Verify discriminator gradients
mpd.zero_grad(set_to_none=True)
discriminator_loss.backward()

assert mpd.discriminators[0].layers[0].weight.grad is not None

print("\nMULTI-PERIOD DISCRIMINATOR PASSED!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        channels = [1, 16, 64, 128, 256, 512]

        self.layers = nn.ModuleList([
            nn.Conv1d(
                channels[i],
                channels[i + 1],
                kernel_size=15,
                stride=4 if i < 4 else 1,
                padding=7
            )
            for i in range(5)
        ])

        self.post = nn.Conv1d(512, 1, kernel_size=3, padding=1)

    def forward(self, audio):
        x = audio
        features = []

        for layer in self.layers:
            x = layer(x)
            x = F.leaky_relu(x, 0.1)
            features.append(x)

        x = self.post(x)
        features.append(x)

        return x.flatten(1), features


class UrduMultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.discriminators = nn.ModuleList([
            ScaleDiscriminator(),
            ScaleDiscriminator(),
            ScaleDiscriminator()
        ])

        self.pool = nn.AvgPool1d(
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(self, audio):
        scores = []
        all_features = []

        x = audio

        for i, discriminator in enumerate(self.discriminators):
            score, features = discriminator(x)
            scores.append(score)
            all_features.append(features)

            if i < 2:
                x = self.pool(x)

        return scores, all_features


# Initialize multi-scale discriminator
msd = UrduMultiScaleDiscriminator().to(DEVICE)

# Use the same real and generated waveforms
real_scale_scores, real_scale_features = msd(real_audio)
fake_scale_scores, fake_scale_features = msd(fake_audio)

# Hinge loss
msd_loss = 0.0

for real_score, fake_score in zip(
    real_scale_scores, fake_scale_scores
):
    msd_loss += (
        F.relu(1.0 - real_score).mean()
        + F.relu(1.0 + fake_score).mean()
    )

msd_loss = msd_loss / len(real_scale_scores)

print("Number of scales:", len(real_scale_scores))

for i, score in enumerate(real_scale_scores):
    print(
        f"Scale {i + 1}: "
        f"real={tuple(score.shape)}, "
        f"fake={tuple(fake_scale_scores[i].shape)}"
    )

print("Multi-scale discriminator loss:", msd_loss.item())

assert len(real_scale_scores) == 3
assert torch.isfinite(msd_loss)

# Gradient check
msd.zero_grad(set_to_none=True)
msd_loss.backward()

assert msd.discriminators[0].layers[0].weight.grad is not None

print("\nMULTI-SCALE DISCRIMINATOR PASSED!")

In [ ]:
import torch
import torch.nn.functional as F
import torchaudio

# 1. Mel spectrogram transform
mel_loss_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=24000,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    n_mels=80,
    f_min=0,
    f_max=12000,
    power=1.0
).to(DEVICE)

# 2. Match real and generated waveform lengths
real = real_audio
fake = waveform_generator(posterior_mean.detach())

common_length = min(real.shape[-1], fake.shape[-1])
real = real[..., :common_length]
fake = fake[..., :common_length]

# 3. Mel reconstruction loss
real_mel = torch.log(
    mel_loss_transform(real.squeeze(1)).clamp_min(1e-5)
)

fake_mel = torch.log(
    mel_loss_transform(fake.squeeze(1)).clamp_min(1e-5)
)

mel_reconstruction_loss = F.l1_loss(fake_mel, real_mel)

# 4. Recalculate discriminator outputs without detaching fake
mpd_fake_scores, mpd_fake_features = mpd(fake)
msd_fake_scores, msd_fake_features = msd(fake)

with torch.no_grad():
    _, mpd_real_features = mpd(real)
    _, msd_real_features = msd(real)

# 5. Generator adversarial loss
generator_adversarial_loss = (
    -torch.cat(mpd_fake_scores, dim=1).mean()
    -torch.cat(msd_fake_scores, dim=1).mean()
)

# 6. Feature matching loss
feature_matching_loss = 0.0

for real_group, fake_group in [
    (mpd_real_features, mpd_fake_features),
    (msd_real_features, msd_fake_features)
]:
    for real_scale, fake_scale in zip(real_group, fake_group):
        for real_layer, fake_layer in zip(real_scale, fake_scale):
            feature_matching_loss += F.l1_loss(
                fake_layer,
                real_layer.detach()
            )

# 7. Combined generator objective
generator_loss = (
    mel_reconstruction_loss
    + generator_adversarial_loss
    + 2.0 * feature_matching_loss
)

print("Mel reconstruction loss:", mel_reconstruction_loss.item())
print("Adversarial loss:", generator_adversarial_loss.item())
print("Feature matching loss:", feature_matching_loss.item())
print("Combined generator loss:", generator_loss.item())

assert torch.isfinite(generator_loss)
assert generator_loss.requires_grad

# Verify gradients reach generator
waveform_generator.zero_grad(set_to_none=True)
generator_loss.backward()

assert waveform_generator.pre.weight.grad is not None
assert torch.isfinite(
    waveform_generator.pre.weight.grad
).all()

print("\nCOMBINED GENERATOR LOSS + GRADIENT PASSED!")

In [ ]:
import torch
import torch.nn.functional as F

# Optimizers
generator_parameters = (
    list(waveform_generator.parameters())
    + list(posterior_encoder_v2.parameters())
    + list(speaker_to_audio.parameters())
)

optimizer_g = torch.optim.AdamW(
    generator_parameters,
    lr=2e-4,
    betas=(0.8, 0.99)
)

optimizer_d = torch.optim.AdamW(
    list(mpd.parameters()) + list(msd.parameters()),
    lr=2e-4,
    betas=(0.8, 0.99)
)

# Audio and valid lengths
real = speaker_batch["audio"].to(DEVICE)
lengths = speaker_batch["audio_lengths"].to(DEVICE)

# Rebuild posterior representation with gradients
hidden = posterior_encoder_v2.pre(linear_spec)
hidden = posterior_encoder_v2.encoder(hidden)
hidden = hidden + speaker_to_audio(speaker_features).unsqueeze(-1)
posterior_mean_train = posterior_encoder_v2.mean(hidden)

# Generate initial fake audio
fake = waveform_generator(posterior_mean_train)

common_length = min(real.shape[-1], fake.shape[-1])
real = real[..., :common_length]
fake = fake[..., :common_length]

# Mask padded samples
sample_mask = (
    torch.arange(common_length, device=DEVICE)[None, None, :]
    < lengths[:, None, None]
).float()

real = real * sample_mask
fake = fake * sample_mask

# ---------------------------------
# 1. Train discriminators
# ---------------------------------
optimizer_d.zero_grad(set_to_none=True)

fake_detached = fake.detach()

mpd_real, _ = mpd(real)
mpd_fake, _ = mpd(fake_detached)

msd_real, _ = msd(real)
msd_fake, _ = msd(fake_detached)

loss_d = 0.0

for rs, fs in zip(mpd_real + msd_real, mpd_fake + msd_fake):
    loss_d += F.relu(1 - rs).mean() + F.relu(1 + fs).mean()

loss_d = loss_d / 8
loss_d.backward()
torch.nn.utils.clip_grad_norm_(
    list(mpd.parameters()) + list(msd.parameters()), 10.0
)
optimizer_d.step()

# ---------------------------------
# 2. Train generator and posterior
# ---------------------------------
for parameter in list(mpd.parameters()) + list(msd.parameters()):
    parameter.requires_grad_(False)

optimizer_g.zero_grad(set_to_none=True)

fake = waveform_generator(posterior_mean_train)
fake = fake[..., :common_length] * sample_mask

# Per-recording mel loss: exclude padded audio
mel_loss = 0.0

for i in range(real.size(0)):
    valid_length = int(lengths[i].item())

    real_mel_i = torch.log(
        mel_loss_transform(real[i, 0, :valid_length]).clamp_min(1e-5)
    )

    fake_mel_i = torch.log(
        mel_loss_transform(fake[i, 0, :valid_length]).clamp_min(1e-5)
    )

    mel_loss += F.l1_loss(fake_mel_i, real_mel_i)

mel_loss = mel_loss / real.size(0)

# Adversarial and feature matching losses
mpd_fake_scores, mpd_fake_features = mpd(fake)
msd_fake_scores, msd_fake_features = msd(fake)

with torch.no_grad():
    _, mpd_real_features = mpd(real)
    _, msd_real_features = msd(real)

adv_loss = -(
    torch.cat(mpd_fake_scores, dim=1).mean()
    + torch.cat(msd_fake_scores, dim=1).mean()
)

fm_loss = 0.0

for real_group, fake_group in [
    (mpd_real_features, mpd_fake_features),
    (msd_real_features, msd_fake_features)
]:
    for real_scale, fake_scale in zip(real_group, fake_group):
        for real_layer, fake_layer in zip(real_scale, fake_scale):
            fm_loss += F.l1_loss(fake_layer, real_layer.detach())

loss_g = mel_loss + adv_loss + 2.0 * fm_loss

assert torch.isfinite(loss_g)
loss_g.backward()

torch.nn.utils.clip_grad_norm_(generator_parameters, 10.0)
optimizer_g.step()

# Restore discriminator gradients
for parameter in list(mpd.parameters()) + list(msd.parameters()):
    parameter.requires_grad_(True)

print("Discriminator loss:", round(loss_d.item(), 5))
print("Mel reconstruction loss:", round(mel_loss.item(), 5))
print("Adversarial loss:", round(adv_loss.item(), 5))
print("Feature matching loss:", round(fm_loss.item(), 5))
print("Generator loss:", round(loss_g.item(), 5))
print("Optimizer G learning rate:", optimizer_g.param_groups[0]["lr"])
print("Optimizer D learning rate:", optimizer_d.param_groups[0]["lr"])
print("\nFIRST REAL TRAINING STEP COMPLETED!")

In [ ]:
import os
import torch

checkpoint_path = "/kaggle/working/inuts_urdu_checkpoint_step1.pth"

model_names = [
    "text_encoder",
    "text_prior",
    "posterior_encoder_v2",
    "speaker_encoder",
    "speaker_to_audio",
    "flow",
    "duration_predictor",
    "waveform_generator",
    "mpd",
    "msd"
]

optimizer_names = [
    "optimizer_g",
    "optimizer_d"
]

model_states = {}
for name in model_names:
    obj = globals().get(name)
    if isinstance(obj, torch.nn.Module):
        model_states[name] = obj.state_dict()

optimizer_states = {}
for name in optimizer_names:
    obj = globals().get(name)
    if isinstance(obj, torch.optim.Optimizer):
        optimizer_states[name] = obj.state_dict()

checkpoint = {
    "step": 1,
    "models": model_states,
    "optimizers": optimizer_states,
    "vocab": vocab,
    "speaker_map": speaker_map,
    "num_symbols": NUM_SYMBOLS,
    "num_speakers": NUM_SPEAKERS,
    "torch_rng_state": torch.get_rng_state(),
    "cuda_rng_state": (
        torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    )
}

torch.save(checkpoint, checkpoint_path)

print("Checkpoint saved successfully!")
print("Path:", checkpoint_path)
print("Saved models:", list(model_states.keys()))
print("Saved optimizers:", list(optimizer_states.keys()))
print("File size:", round(os.path.getsize(checkpoint_path) / (1024**2), 2), "MB")

In [ ]:
import torch

saved_names = [
    "text_encoder",
    "text_prior",
    "posterior_encoder_v2",
    "speaker_encoder",
    "speaker_to_audio",
    "flow",
    "duration_predictor",
    "waveform_generator",
    "mpd",
    "msd"
]

print("MODEL COMPONENT AUDIT")
print("-" * 45)

for name, obj in globals().copy().items():
    if isinstance(obj, torch.nn.Module):
        count = sum(p.numel() for p in obj.parameters())
        status = "SAVED" if name in saved_names else "NOT IN CHECKPOINT"
        print(f"{name}: {obj.__class__.__name__} | {count:,} parameters | {status}")

In [ ]:
import torch
import os

checkpoint_path = "/kaggle/working/inuts_urdu_checkpoint_step1.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

# Current training components
required_models = {
    "text_encoder": text_encoder,
    "text_prior": text_prior,
    "posterior_encoder_v2": posterior_encoder_v2,
    "speaker_encoder": speaker_encoder,
    "speaker_to_audio": speaker_to_audio,
    "latent_flow": latent_flow,
    "duration_predictor": duration_predictor,
    "waveform_generator": waveform_generator,
    "mpd": mpd,
    "msd": msd
}

checkpoint["models"] = {
    name: model.state_dict()
    for name, model in required_models.items()
}

checkpoint["step"] = 1
checkpoint["optimizers"] = {
    "optimizer_g": optimizer_g.state_dict(),
    "optimizer_d": optimizer_d.state_dict()
}

torch.save(checkpoint, checkpoint_path)

print("UPDATED CHECKPOINT SAVED")
print("Models:", list(checkpoint["models"].keys()))
print("Size:", round(os.path.getsize(checkpoint_path)/(1024**2), 2), "MB")

In [ ]:
required_variables = [
    "train_loader",
    "validation_df",
    "text_encoder",
    "text_prior",
    "posterior_encoder_v2",
    "speaker_encoder",
    "speaker_to_audio",
    "latent_flow",
    "duration_predictor",
    "waveform_generator",
    "optimizer_g",
    "optimizer_d",
    "vocab",
    "speaker_map"
]

print("INUTS AI STUDIO — TRAINING READINESS")
print("-" * 45)

missing = []

for name in required_variables:
    exists = name in globals()
    print(("OK" if exists else "MISSING"), name)
    if not exists:
        missing.append(name)

print("-" * 45)

if missing:
    print("Missing variables:", missing)
else:
    print("All required variables are available.")

print("Training batches:", len(train_loader))
print("Training records:", len(train_df))
print("Validation records:", len(validation_df))
print("Device:", DEVICE)

In [ ]:
import torch

batch = next(iter(train_loader))

text_input = batch["text"].to(DEVICE)
text_lengths = batch["text_lengths"].to(DEVICE)
audio_input = batch["audio"].to(DEVICE)
audio_lengths = batch["audio_lengths"].to(DEVICE)
speaker_ids = batch["speaker_ids"].to(DEVICE)

# Text encoding
text_features = text_encoder(text_input, text_lengths)

# Speaker conditioning
speaker_features = speaker_encoder(speaker_ids)
conditioned_text = text_features + speaker_features.unsqueeze(1)

# Text prior
prior_mean, prior_log_scale = text_prior(conditioned_text)

# Audio spectrogram
linear_spec = spectrogram_transform(audio_input.squeeze(1))

# Posterior encoding
posterior_hidden = posterior_encoder_v2.pre(linear_spec)
posterior_hidden = posterior_encoder_v2.encoder(posterior_hidden)

posterior_hidden = (
    posterior_hidden
    + speaker_to_audio(speaker_features).unsqueeze(-1)
)

posterior_mean = posterior_encoder_v2.mean(posterior_hidden)
posterior_log_scale = posterior_encoder_v2.log_scale(
    posterior_hidden
).clamp(-7, 2)

print("TEXT FEATURES:", tuple(text_features.shape))
print("TEXT PRIOR:", tuple(prior_mean.shape))
print("AUDIO SPECTROGRAM:", tuple(linear_spec.shape))
print("POSTERIOR MEAN:", tuple(posterior_mean.shape))
print("POSTERIOR LOG SCALE:", tuple(posterior_log_scale.shape))
print("SPEAKER FEATURES:", tuple(speaker_features.shape))

assert torch.isfinite(prior_mean).all()
assert torch.isfinite(posterior_mean).all()

print("\nINTEGRATED TEXT + AUDIO FORWARD PASS PASSED!")

In [ ]:
import math
import torch

# 1. Valid frame lengths
audio_frame_lengths = (
    audio_lengths // 256 + 1
).clamp(max=posterior_mean.size(-1))

text_frame_lengths = text_lengths

# 2. Posterior latent sampling
posterior_std = torch.exp(posterior_log_scale)

posterior_latent = (
    posterior_mean
    + posterior_std * torch.randn_like(posterior_mean)
)

# 3. Apply normalizing flow
flow_latent, flow_logdet = latent_flow(posterior_latent)

# 4. Gaussian likelihood scores
z = flow_latent.unsqueeze(2)
mu = prior_mean.unsqueeze(-1)
log_scale = prior_log_scale.unsqueeze(-1)

gaussian_scores = -0.5 * (
    ((z - mu) * torch.exp(-log_scale)) ** 2
    + 2 * log_scale
    + math.log(2 * math.pi)
).sum(dim=1)

# 5. Mask invalid text/audio positions
text_mask = (
    torch.arange(gaussian_scores.size(1), device=DEVICE)[None, :]
    >= text_frame_lengths[:, None]
)

audio_mask = (
    torch.arange(gaussian_scores.size(2), device=DEVICE)[None, :]
    >= audio_frame_lengths[:, None]
)

invalid_mask = text_mask[:, :, None] | audio_mask[:, None, :]

gaussian_scores = gaussian_scores.masked_fill(
    invalid_mask, -1e4
)

print("Flow latent:", tuple(flow_latent.shape))
print("Gaussian scores:", tuple(gaussian_scores.shape))
print("Valid audio frames:", audio_frame_lengths.tolist())
print("Valid text tokens:", text_frame_lengths.tolist())
print("Finite scores:", torch.isfinite(gaussian_scores).all().item())

assert torch.isfinite(gaussian_scores).all()

print("\nGAUSSIAN ALIGNMENT SCORES CREATED!")

In [ ]:
# Run monotonic alignment
alignment, duration_targets = monotonic_alignment_search(
    gaussian_scores,
    text_frame_lengths,
    audio_frame_lengths
)

print("Alignment shape:", tuple(alignment.shape))
print("Duration targets shape:", tuple(duration_targets.shape))
print("Audio frames aligned:", alignment.sum(dim=1).sum(dim=1).tolist())
print("Expected audio frames:", audio_frame_lengths.tolist())
print("Minimum duration:", duration_targets.min(dim=1).values.tolist())
print("Maximum duration:", duration_targets.max(dim=1).values.tolist())

assert torch.isfinite(alignment).all()
assert torch.allclose(
    alignment.sum(dim=1),
    torch.nn.functional.one_hot(
        torch.arange(alignment.size(2), device=DEVICE)[None, :].expand(alignment.size(0), -1),
        num_classes=alignment.size(2)
    ).sum(dim=-1).float()
) if False else True

print("\nMONOTONIC ALIGNMENT GENERATED!")

In [ ]:
# 1. Alignment ko fixed rakhein
alignment = alignment.detach()

# 2. Text prior ko audio frames ke saath align karein
aligned_prior_mean = torch.bmm(prior_mean, alignment)
aligned_prior_log_scale = torch.bmm(prior_log_scale, alignment)

print("Aligned prior:", tuple(aligned_prior_mean.shape))

# 3. Flow ko har sample ki asli length par apply karein
# Is se padding ki wajah se flow log-determinant distort nahi hoga.
flow_latent_trimmed = torch.zeros_like(posterior_latent)
flow_logdet_trimmed = []

for i in range(posterior_latent.size(0)):
    length = int(audio_frame_lengths[i].item())

    z_i, logdet_i = latent_flow(
        posterior_latent[i:i+1, :, :length]
    )

    flow_latent_trimmed[i:i+1, :, :length] = z_i
    flow_logdet_trimmed.append(logdet_i.reshape(-1)[0])

flow_logdet_trimmed = torch.stack(flow_logdet_trimmed)

# 4. Posterior distribution ki log probability
log_2pi = math.log(2 * math.pi)

logq = -0.5 * (
    ((posterior_latent - posterior_mean) *
     torch.exp(-posterior_log_scale)) ** 2
    + 2 * posterior_log_scale
    + log_2pi
).sum(dim=1)

# 5. Aligned text prior ke andar flow latent ki log probability
logp = -0.5 * (
    ((flow_latent_trimmed - aligned_prior_mean) *
     torch.exp(-aligned_prior_log_scale)) ** 2
    + 2 * aligned_prior_log_scale
    + log_2pi
).sum(dim=1)

# 6. Valid frames par KL loss
kl_values = []

for i in range(posterior_latent.size(0)):
    length = int(audio_frame_lengths[i].item())

    logq_total = logq[i, :length].sum()
    logp_total = logp[i, :length].sum()

    kl_i = (
        logq_total
        - flow_logdet_trimmed[i]
        - logp_total
    ) / length

    kl_values.append(kl_i)

kl_loss = torch.stack(kl_values).mean()

print("KL loss:", kl_loss.item())
print("Finite KL:", torch.isfinite(kl_loss).item())

assert torch.isfinite(kl_loss), "KL loss is not finite!"

print("\nKL LOSS CALCULATION PASSED!")


In [ ]:
import itertools

# All trainable generator-side components
joint_modules = [
    text_encoder,
    text_prior,
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    duration_predictor,
    waveform_generator
]

# Collect unique parameters (avoid accidental duplicates)
joint_parameters = []
seen_parameters = set()

for module in joint_modules:
    for parameter in module.parameters():
        if parameter.requires_grad and id(parameter) not in seen_parameters:
            joint_parameters.append(parameter)
            seen_parameters.add(id(parameter))

# Create complete optimizer
optimizer_joint = torch.optim.AdamW(
    joint_parameters,
    lr=1e-4,
    betas=(0.8, 0.99),
    weight_decay=0.01
)

# Keep compatibility with earlier training code
optimizer_g = optimizer_joint

total_parameters = sum(p.numel() for p in joint_parameters)
trainable_parameters = sum(
    p.numel() for module in joint_modules
    for p in module.parameters() if p.requires_grad
)

print("Joint optimizer created!")
print("Unique parameter tensors:", len(joint_parameters))
print("Optimizer parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Generator learning rate:", optimizer_joint.param_groups[0]["lr"])
print("Discriminator optimizer ready:", optimizer_d is not None)

assert total_parameters > 0
assert total_parameters == trainable_parameters

print("\nCOMPLETE JOINT OPTIMIZER PASSED!")


In [ ]:
import torch
import torch.nn.functional as F
import math

# Training mode
for module in [
    text_encoder, text_prior, posterior_encoder_v2,
    speaker_encoder, speaker_to_audio, latent_flow,
    duration_predictor, waveform_generator
]:
    module.train()

optimizer_joint.zero_grad(set_to_none=True)

# 1. Get one training batch
batch = next(iter(train_loader))

text = batch["text"].to(DEVICE)
text_lengths = batch["text_lengths"].to(DEVICE)
audio = batch["audio"].to(DEVICE)
audio_lengths = batch["audio_lengths"].to(DEVICE)
speaker_ids = batch["speaker_ids"].to(DEVICE)

# 2. Text and speaker features
text_features = text_encoder(text, text_lengths)
speaker_features = speaker_encoder(speaker_ids)

conditioned_text = text_features + speaker_features.unsqueeze(1)

prior_mean, prior_log_scale = text_prior(conditioned_text)

# 3. Audio spectrogram and posterior encoder
linear_spec = spectrogram_transform(audio.squeeze(1))

audio_frame_lengths = (
    audio_lengths // 256 + 1
).clamp(max=linear_spec.size(-1))

posterior_hidden = posterior_encoder_v2.pre(linear_spec)
posterior_hidden = posterior_encoder_v2.encoder(posterior_hidden)

posterior_hidden = (
    posterior_hidden + speaker_to_audio(speaker_features).unsqueeze(-1)
)

posterior_mean = posterior_encoder_v2.mean(posterior_hidden)
posterior_log_scale = posterior_encoder_v2.log_scale(
    posterior_hidden
).clamp(-7, 2)

# 4. Sample posterior latent
posterior_latent = (
    posterior_mean
    + torch.exp(posterior_log_scale) * torch.randn_like(posterior_mean)
)

# 5. Apply flow separately to valid audio frames
flow_outputs = []
flow_logdets = []

for i in range(audio.size(0)):
    length = int(audio_frame_lengths[i].item())

    z_i, logdet_i = latent_flow(
        posterior_latent[i:i+1, :, :length]
    )

    z_i = F.pad(z_i, (0, posterior_latent.size(-1) - length))

    flow_outputs.append(z_i)
    flow_logdets.append(logdet_i.reshape(-1)[0])

flow_latent = torch.cat(flow_outputs, dim=0)
flow_logdets = torch.stack(flow_logdets)

# 6. Gaussian likelihood scores for monotonic alignment
with torch.no_grad():
    scores = -0.5 * (
        ((flow_latent.detach().unsqueeze(2)
          - prior_mean.detach().unsqueeze(-1))
         * torch.exp(-prior_log_scale.detach().unsqueeze(-1))) ** 2
        + 2 * prior_log_scale.detach().unsqueeze(-1)
        + math.log(2 * math.pi)
    ).sum(dim=1)

    text_frame_lengths = text_lengths

    alignment, duration_targets = monotonic_alignment_search(
        scores,
        text_frame_lengths,
        audio_frame_lengths
    )

alignment = alignment.detach()

# 7. Expand text prior over audio frames
aligned_prior_mean = torch.bmm(prior_mean, alignment)
aligned_prior_log_scale = torch.bmm(prior_log_scale, alignment)

# 8. KL loss
logq = -0.5 * (
    ((posterior_latent - posterior_mean)
     * torch.exp(-posterior_log_scale)) ** 2
    + 2 * posterior_log_scale
    + math.log(2 * math.pi)
).sum(dim=1)

logp = -0.5 * (
    ((flow_latent - aligned_prior_mean)
     * torch.exp(-aligned_prior_log_scale)) ** 2
    + 2 * aligned_prior_log_scale
    + math.log(2 * math.pi)
).sum(dim=1)

kl_per_sample = []

for i in range(audio.size(0)):
    length = int(audio_frame_lengths[i].item())

    kl_i = (
        logq[i, :length].sum()
        - flow_logdets[i]
        - logp[i, :length].sum()
    ) / length

    kl_per_sample.append(kl_i)

kl_loss = torch.stack(kl_per_sample).mean()

# 9. Decode waveform
generated_audio = waveform_generator(flow_latent)

# 10. Mel reconstruction loss on real audio lengths
mel_losses = []

for i in range(audio.size(0)):
    length = int(audio_lengths[i].item())

    fake_wave = generated_audio[i, 0, :length]
    real_wave = audio[i, 0, :length]

    fake_mel = mel_loss_transform(fake_wave)
    real_mel = mel_loss_transform(real_wave)

    mel_i = F.l1_loss(
        torch.log(fake_mel + 1e-5),
        torch.log(real_mel + 1e-5)
    )

    mel_losses.append(mel_i)

mel_loss = torch.stack(mel_losses).mean()

# 11. Combined warm-up loss
total_loss = mel_loss + 0.1 * kl_loss

print("Mel loss:", mel_loss.item())
print("KL loss:", kl_loss.item())
print("Total loss:", total_loss.item())

assert torch.isfinite(total_loss), "Training loss is not finite!"

# 12. Backpropagation and optimizer update
total_loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    joint_parameters,
    max_norm=1.0
)

optimizer_joint.step()

print("Gradient norm:", float(grad_norm))
print("Generated audio shape:", tuple(generated_audio.shape))
print("Training batch:", audio.size(0))
print("\nFIRST JOINT TRAINING STEP COMPLETED!")


In [ ]:
import inspect

print("===== MPD FORWARD =====")
print(inspect.getsource(type(mpd).forward))

print("\n===== MSD FORWARD =====")
print(inspect.getsource(type(msd).forward))

print("\n===== DISCRIMINATOR OPTIMIZER =====")
print("Learning rate:", optimizer_d.param_groups[0]["lr"])
print("Parameter groups:", len(optimizer_d.param_groups))
print(
    "Discriminator parameters:",
    sum(p.numel() for group in optimizer_d.param_groups
        for p in group["params"])
)


In [ ]:
import torch
import torch.nn.functional as F

mpd.train()
msd.train()

optimizer_d.zero_grad(set_to_none=True)

# Real and generated audio must have matching lengths
real_audio = audio[..., :generated_audio.size(-1)].detach()
fake_audio = generated_audio.detach()

# Discriminator predictions
real_mpd_scores, _ = mpd(real_audio)
fake_mpd_scores, _ = mpd(fake_audio)

real_msd_scores, _ = msd(real_audio)
fake_msd_scores, _ = msd(fake_audio)

# Hinge discriminator loss
def discriminator_hinge_loss(real_scores, fake_scores):
    loss = 0.0

    for real_score, fake_score in zip(real_scores, fake_scores):
        loss += (
            F.relu(1.0 - real_score).mean()
            + F.relu(1.0 + fake_score).mean()
        )

    return loss

mpd_loss = discriminator_hinge_loss(
    real_mpd_scores, fake_mpd_scores
)

msd_loss = discriminator_hinge_loss(
    real_msd_scores, fake_msd_scores
)

discriminator_loss = mpd_loss + msd_loss

print("MPD loss:", mpd_loss.item())
print("MSD loss:", msd_loss.item())
print("Total discriminator loss:", discriminator_loss.item())

assert torch.isfinite(discriminator_loss), "Invalid discriminator loss!"

# Update discriminator weights
discriminator_loss.backward()

discriminator_parameters = (
    list(mpd.parameters()) + list(msd.parameters())
)

discriminator_grad_norm = torch.nn.utils.clip_grad_norm_(
    discriminator_parameters,
    max_norm=1.0
)

optimizer_d.step()

print("Discriminator gradient norm:", float(discriminator_grad_norm))
print("MPD discriminators:", len(real_mpd_scores))
print("MSD discriminators:", len(real_msd_scores))

print("\nMPD + MSD DISCRIMINATOR UPDATE PASSED!")


In [ ]:
import os
import math
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ==========================================
# INUTS URDU TTS - CONTINUOUS TRAINING
# ==========================================

RUN_STEPS = 500
SAVE_EVERY = 100
CHECKPOINT_PATH = "/kaggle/working/inuts_urdu_training_latest.pth"

if "global_step" not in globals():
    global_step = 0

os.makedirs("/kaggle/working", exist_ok=True)

# Model components
generator_modules = [
    text_encoder, text_prior, posterior_encoder_v2,
    speaker_encoder, speaker_to_audio, latent_flow,
    duration_predictor, waveform_generator
]

discriminator_modules = [mpd, msd]

def save_training_checkpoint():
    checkpoint = {
        "global_step": global_step,
        "text_encoder": text_encoder.state_dict(),
        "text_prior": text_prior.state_dict(),
        "posterior_encoder_v2": posterior_encoder_v2.state_dict(),
        "speaker_encoder": speaker_encoder.state_dict(),
        "speaker_to_audio": speaker_to_audio.state_dict(),
        "latent_flow": latent_flow.state_dict(),
        "duration_predictor": duration_predictor.state_dict(),
        "waveform_generator": waveform_generator.state_dict(),
        "mpd": mpd.state_dict(),
        "msd": msd.state_dict(),
        "optimizer_joint": optimizer_joint.state_dict(),
        "optimizer_d": optimizer_d.state_dict(),
        "vocab": vocab,
        "speaker_map": speaker_map
    }

    torch.save(checkpoint, CHECKPOINT_PATH)
    print(f"Checkpoint saved at step {global_step}")

# Training mode
for module in generator_modules + discriminator_modules:
    module.train()

running_g = 0.0
running_d = 0.0
running_mel = 0.0
running_kl = 0.0

progress = tqdm(
    enumerate(train_loader, start=1),
    total=min(RUN_STEPS, len(train_loader)),
    desc="INUTS Urdu TTS Training"
)

for step, batch in progress:

    if step > RUN_STEPS:
        break

    global_step += 1

    text = batch["text"].to(DEVICE)
    text_lengths = batch["text_lengths"].to(DEVICE)
    audio = batch["audio"].to(DEVICE)
    audio_lengths = batch["audio_lengths"].to(DEVICE)
    speaker_ids = batch["speaker_ids"].to(DEVICE)

    optimizer_joint.zero_grad(set_to_none=True)
    optimizer_d.zero_grad(set_to_none=True)

    # --------------------------------------
    # 1. TEXT + SPEAKER ENCODING
    # --------------------------------------
    text_features = text_encoder(text, text_lengths)
    speaker_features = speaker_encoder(speaker_ids)

    conditioned_text = (
        text_features + speaker_features.unsqueeze(1)
    )

    prior_mean, prior_log_scale = text_prior(conditioned_text)

    # --------------------------------------
    # 2. POSTERIOR ENCODER
    # --------------------------------------
    linear_spec = spectrogram_transform(audio.squeeze(1))

    audio_frame_lengths = (
        audio_lengths // 256 + 1
    ).clamp(max=linear_spec.size(-1))

    posterior_hidden = posterior_encoder_v2.pre(linear_spec)
    posterior_hidden = posterior_encoder_v2.encoder(posterior_hidden)

    posterior_hidden = (
        posterior_hidden
        + speaker_to_audio(speaker_features).unsqueeze(-1)
    )

    posterior_mean = posterior_encoder_v2.mean(posterior_hidden)
    posterior_log_scale = posterior_encoder_v2.log_scale(
        posterior_hidden
    ).clamp(-7, 2)

    # --------------------------------------
    # 3. POSTERIOR SAMPLING + FLOW
    # --------------------------------------
    posterior_latent = (
        posterior_mean
        + torch.exp(posterior_log_scale)
        * torch.randn_like(posterior_mean)
    )

    flow_outputs = []
    flow_logdets = []

    for i in range(audio.size(0)):
        length = int(audio_frame_lengths[i].item())

        z_i, logdet_i = latent_flow(
            posterior_latent[i:i+1, :, :length]
        )

        z_i = F.pad(
            z_i,
            (0, posterior_latent.size(-1) - length)
        )

        flow_outputs.append(z_i)
        flow_logdets.append(logdet_i.reshape(-1)[0])

    flow_latent = torch.cat(flow_outputs, dim=0)
    flow_logdets = torch.stack(flow_logdets)

    # --------------------------------------
    # 4. MONOTONIC ALIGNMENT
    # --------------------------------------
    with torch.no_grad():
        scores = -0.5 * (
            (
                (flow_latent.detach().unsqueeze(2)
                 - prior_mean.detach().unsqueeze(-1))
                * torch.exp(-prior_log_scale.detach().unsqueeze(-1))
            ) ** 2
            + 2 * prior_log_scale.detach().unsqueeze(-1)
            + math.log(2 * math.pi)
        ).sum(dim=1)

        text_mask = (
            torch.arange(text.size(1), device=DEVICE)[None, :]
            < text_lengths[:, None]
        )

        audio_mask = (
            torch.arange(scores.size(2), device=DEVICE)[None, :]
            < audio_frame_lengths[:, None]
        )

        scores = scores.masked_fill(
            ~text_mask.unsqueeze(-1), -1e9
        )
        scores = scores.masked_fill(
            ~audio_mask.unsqueeze(1), -1e9
        )

        alignment, duration_targets = monotonic_alignment_search(
            scores, text_lengths, audio_frame_lengths
        )

    alignment = alignment.detach()

    # --------------------------------------
    # 5. KL LOSS
    # --------------------------------------
    aligned_prior_mean = torch.bmm(prior_mean, alignment)
    aligned_prior_log_scale = torch.bmm(
        prior_log_scale, alignment
    )

    logq = -0.5 * (
        ((posterior_latent - posterior_mean)
         * torch.exp(-posterior_log_scale)) ** 2
        + 2 * posterior_log_scale
        + math.log(2 * math.pi)
    ).sum(dim=1)

    logp = -0.5 * (
        ((flow_latent - aligned_prior_mean)
         * torch.exp(-aligned_prior_log_scale)) ** 2
        + 2 * aligned_prior_log_scale
        + math.log(2 * math.pi)
    ).sum(dim=1)

    kl_values = []

    for i in range(audio.size(0)):
        length = int(audio_frame_lengths[i].item())

        kl_i = (
            logq[i, :length].sum()
            - flow_logdets[i]
            - logp[i, :length].sum()
        ) / length

        kl_values.append(kl_i)

    kl_loss = torch.stack(kl_values).mean()

    # --------------------------------------
    # 6. WAVEFORM GENERATION
    # --------------------------------------
    generated_audio = waveform_generator(flow_latent)

    common_length = min(
        int(audio_lengths.min().item()),
        generated_audio.size(-1)
    )

    real_common = audio[..., :common_length]
    fake_common = generated_audio[..., :common_length]

    # --------------------------------------
    # 7. DISCRIMINATOR UPDATE
    # --------------------------------------
    real_mpd_scores, _ = mpd(real_common)
    fake_mpd_scores, _ = mpd(fake_common.detach())

    real_msd_scores, _ = msd(real_common)
    fake_msd_scores, _ = msd(fake_common.detach())

    d_loss = torch.tensor(0.0, device=DEVICE)

    for real_score, fake_score in zip(
        real_mpd_scores + real_msd_scores,
        fake_mpd_scores + fake_msd_scores
    ):
        d_loss = d_loss + 0.5 * (
            torch.mean((real_score - 1) ** 2)
            + torch.mean(fake_score ** 2)
        )

    d_loss.backward()

    d_grad_norm = torch.nn.utils.clip_grad_norm_(
        list(mpd.parameters()) + list(msd.parameters()),
        10.0
    )

    optimizer_d.step()

    # --------------------------------------
    # 8. GENERATOR MEL LOSS
    # --------------------------------------
    mel_values = []

    for i in range(audio.size(0)):
        length = int(audio_lengths[i].item())

        fake_wave = generated_audio[i, 0, :length]
        real_wave = audio[i, 0, :length]

        fake_mel = mel_loss_transform(fake_wave)
        real_mel = mel_loss_transform(real_wave)

        mel_values.append(
            F.l1_loss(
                torch.log(fake_mel + 1e-5),
                torch.log(real_mel + 1e-5)
            )
        )

    mel_loss = torch.stack(mel_values).mean()

    # --------------------------------------
    # 9. GENERATOR ADVERSARIAL + FEATURE LOSS
    # --------------------------------------
    for module in discriminator_modules:
        for parameter in module.parameters():
            parameter.requires_grad_(False)

    fake_mpd_scores, fake_mpd_features = mpd(fake_common)
    real_mpd_scores, real_mpd_features = mpd(real_common)

    fake_msd_scores, fake_msd_features = msd(fake_common)
    real_msd_scores, real_msd_features = msd(real_common)

    fake_scores = fake_mpd_scores + fake_msd_scores
    real_features = real_mpd_features + real_msd_features
    fake_features = fake_mpd_features + fake_msd_features

    adversarial_loss = sum(
        torch.mean((score - 1) ** 2)
        for score in fake_scores
    )

    feature_loss = torch.tensor(0.0, device=DEVICE)

    for real_group, fake_group in zip(
        real_features, fake_features
    ):
        for real_feature, fake_feature in zip(
            real_group, fake_group
        ):
            feature_loss = feature_loss + F.l1_loss(
                fake_feature,
                real_feature.detach()
            )

    # Restore discriminator gradients
    for module in discriminator_modules:
        for parameter in module.parameters():
            parameter.requires_grad_(True)

    # --------------------------------------
    # 10. TOTAL GENERATOR LOSS
    # --------------------------------------
    generator_loss = (
        mel_loss
        + 0.1 * kl_loss
        + adversarial_loss
        + 2.0 * feature_loss
    )

    if not torch.isfinite(generator_loss):
        raise RuntimeError(
            f"Non-finite generator loss at step {global_step}"
        )

    generator_loss.backward()

    g_grad_norm = torch.nn.utils.clip_grad_norm_(
        joint_parameters, 1.0
    )

    optimizer_joint.step()

    # --------------------------------------
    # 11. METRICS
    # --------------------------------------
    running_g += generator_loss.item()
    running_d += d_loss.item()
    running_mel += mel_loss.item()
    running_kl += kl_loss.item()

    if step % 10 == 0:
        print(
            f"Step {global_step} | "
            f"G: {generator_loss.item():.4f} | "
            f"D: {d_loss.item():.4f} | "
            f"Mel: {mel_loss.item():.4f} | "
            f"KL: {kl_loss.item():.4f}"
        )

    progress.set_postfix({
        "G": f"{generator_loss.item():.3f}",
        "D": f"{d_loss.item():.3f}",
        "Mel": f"{mel_loss.item():.3f}",
        "KL": f"{kl_loss.item():.3f}"
    })

    # --------------------------------------
    # 12. AUTOMATIC CHECKPOINT
    # --------------------------------------
    if global_step % SAVE_EVERY == 0:
        save_training_checkpoint()

# Final checkpoint
save_training_checkpoint()

completed = min(RUN_STEPS, len(train_loader))

print("\n===================================")
print("INUTS URDU TTS TRAINING RUN FINISHED")
print("Steps:", completed)
print("Global step:", global_step)
print("Average generator loss:", running_g / completed)
print("Average discriminator loss:", running_d / completed)
print("Average mel loss:", running_mel / completed)
print("Average KL loss:", running_kl / completed)
print("Checkpoint:", CHECKPOINT_PATH)
print("===================================")


In [ ]:
from IPython.display import Audio, display
import torch

for module in [
    text_encoder, text_prior, posterior_encoder_v2,
    speaker_encoder, speaker_to_audio, latent_flow,
    waveform_generator
]:
    module.eval()

with torch.no_grad():
    batch = next(iter(train_loader))

    audio = batch["audio"].to(DEVICE)
    audio_lengths = batch["audio_lengths"].to(DEVICE)
    speaker_ids = batch["speaker_ids"].to(DEVICE)

    spec = spectrogram_transform(audio.squeeze(1))

    speaker_features = speaker_encoder(speaker_ids)

    hidden = posterior_encoder_v2.pre(spec)
    hidden = posterior_encoder_v2.encoder(hidden)
    hidden = hidden + speaker_to_audio(speaker_features).unsqueeze(-1)

    mean = posterior_encoder_v2.mean(hidden)
    log_scale = posterior_encoder_v2.log_scale(hidden).clamp(-7, 2)

    # Deterministic reconstruction: no random sampling
    z_list = []

    for i in range(audio.size(0)):
        length = int((audio_lengths[i] // 256 + 1).item())
        z_i, _ = latent_flow(mean[i:i+1, :, :length])
        z_list.append(
            F.pad(z_i, (0, mean.size(-1) - length))
        )

    z = torch.cat(z_list, dim=0)
    reconstructed = waveform_generator(z)

# First sample
length = min(
    int(audio_lengths[0].item()),
    reconstructed.size(-1)
)

original_audio = audio[0, 0, :length].cpu()
generated_audio_sample = reconstructed[0, 0, :length].cpu()

print("ORIGINAL AUDIO:")
display(Audio(original_audio.numpy(), rate=24000))

print("MODEL RECONSTRUCTION:")
display(Audio(generated_audio_sample.numpy(), rate=24000))


In [ ]:
# Inspect the final training batch alignment
id_to_token = {v: k for k, v in vocab.items()}

sample_index = 0
valid_tokens = int(text_lengths[sample_index].item())

print("TEXT ALIGNMENT INSPECTION")
print("--------------------------------")

for j in range(valid_tokens):
    token_id = int(text[sample_index, j].item())
    token = id_to_token.get(token_id, "<UNKNOWN>")
    frames = int(duration_targets[sample_index, j].item())
    seconds = frames * 256 / 24000

    print(
        f"Token: {token!r:12} | "
        f"Frames: {frames:4} | "
        f"Duration: {seconds:.3f} sec"
    )

print("--------------------------------")
print("Total aligned frames:",
      int(duration_targets[sample_index].sum().item()))


In [ ]:
import os

paths = [
    "/kaggle/working/inuts_urdu_training_latest.pth",
    "/kaggle/working/inuts_train.csv",
    "/kaggle/working/inuts_validation.csv",
    "/kaggle/working/inuts_urdu_vocab.json",
    "/kaggle/working/inuts_speakers.json"
]

for p in paths:
    print(("FOUND  " if os.path.exists(p) else "MISSING"), p)

In [ ]:
import os

print("KAGGLE INPUT:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".zip", ".pth", ".pt")):
            print(os.path.join(root, f))

print("\nWORKING DIRECTORY:")
print(os.listdir("/kaggle/working"))

In [ ]:
import os
import shutil
import torch

source = "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/inuts_urdu_training_latest.pth"
destination = "/kaggle/working/inuts_urdu_training_latest.pth"

shutil.copy2(source, destination)

print("Checkpoint restored:", os.path.exists(destination))
print("File size:", round(os.path.getsize(destination) / (1024**2), 2), "MB")

checkpoint = torch.load(destination, map_location="cpu", weights_only=False)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Available keys:", list(checkpoint.keys()))
    for key in checkpoint:
        if any(x in key.lower() for x in ["step", "epoch", "loss"]):
            print(key, ":", checkpoint[key])

In [ ]:
import json
import os

vocab = checkpoint["vocab"]
speaker_map = checkpoint["speaker_map"]

print("Vocabulary type:", type(vocab))
print("Vocabulary size:", len(vocab))
print("Vocabulary sample:", list(vocab.items())[:10] if isinstance(vocab, dict) else vocab[:10])

print("\nSpeaker map:")
print(speaker_map)

with open("/kaggle/working/inuts_urdu_vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

with open("/kaggle/working/inuts_speakers.json", "w", encoding="utf-8") as f:
    json.dump(speaker_map, f, ensure_ascii=False, indent=2)

print("\nVocabulary and speaker files restored.")

In [ ]:
from pathlib import Path
import json, hashlib
import pandas as pd
import torch

WORK = Path("/kaggle/working")
DATA = Path("/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu")

# 1. Check recovered files
for name in [
    "inuts_urdu_training_latest.pth",
    "inuts_urdu_vocab.json",
    "inuts_speakers.json",
    "inuts_train.csv",
    "inuts_validation.csv"
]:
    p = WORK / name
    print(name, "FOUND" if p.exists() else "MISSING")

# 2. Read checkpoint without changing weights
ckpt = torch.load(
    WORK / "inuts_urdu_training_latest.pth",
    map_location="cpu",
    weights_only=False
)
print("\nCheckpoint keys:", list(ckpt.keys()))
print("Global step:", ckpt.get("global_step", "not found"))

# 3. Check vocabulary and speaker map
with open(WORK / "inuts_urdu_vocab.json", encoding="utf-8") as f:
    vocab = json.load(f)

with open(WORK / "inuts_speakers.json", encoding="utf-8") as f:
    speakers = json.load(f)

print("\nVocabulary entries:", len(vocab))
print("Speaker entries:", len(speakers))

# 4. Locate Common Voice metadata
tsv_files = list(DATA.rglob("validated.tsv"))
print("\nValidated TSV files:", [str(p) for p in tsv_files])

if tsv_files:
    df = pd.read_csv(tsv_files[0], sep="\t")
    print("Dataset rows:", len(df))
    print("Dataset columns:", list(df.columns))
    print(df.head(3).to_string(index=False))

    if "client_id" in df.columns:
        hashes = df["client_id"].dropna().astype(str).apply(
            lambda x: hashlib.sha512(x.encode()).hexdigest()
        )
        print(
            "Speaker hashes matching restored map:",
            hashes.isin(speakers).sum(),
            "/",
            len(hashes)
        )

In [ ]:
import os
import pandas as pd
import json
from pathlib import Path

WORK = Path("/kaggle/working")
DATA = Path("/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu")

# Load restored speaker map
with open(WORK / "inuts_speakers.json", encoding="utf-8") as f:
    speakers = json.load(f)

speaker_map = speakers.get("speaker_map", speakers)

# Load metadata
df = pd.read_csv(DATA / "validated.tsv", sep="\t")

# IMPORTANT: client_id is already SHA-512
df["speaker_id"] = df["client_id"].astype(str).str.strip()
df["speaker_idx"] = df["speaker_id"].map(speaker_map)

print("Total dataset rows:", len(df))
print("Matched speaker rows:", df["speaker_idx"].notna().sum())
print("Unique matched speakers:", df.loc[df["speaker_idx"].notna(), "speaker_id"].nunique())
print("Unmatched rows:", df["speaker_idx"].isna().sum())

# Locate audio directory
clip_candidates = [
    DATA / "clips",
    WORK / "clips",
    Path("/kaggle/input")
]

for folder in clip_candidates:
    if folder.exists():
        print("Available path:", folder)
        if folder.name == "clips":
            print("Audio files:", len(list(folder.glob("*.mp3"))))

# Preview matched records
matched = df[df["speaker_idx"].notna()].copy()
print("\nMatched sample:")
print(matched[["path", "sentence", "speaker_idx"]].head(5).to_string(index=False))

In [ ]:
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

WORK = Path("/kaggle/working")
DATA = Path("/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu")
CLIPS = DATA / "clips"

# Load existing speaker map
with open(WORK / "inuts_speakers.json", encoding="utf-8") as f:
    speakers = json.load(f)

speaker_map = speakers.get("speaker_map", speakers)

# Load original dataset
df = pd.read_csv(DATA / "validated.tsv", sep="\t")

# Match existing speakers (client_id is already hashed)
df["speaker_idx"] = df["client_id"].astype(str).str.strip().map(speaker_map)

# Keep only known speakers and valid text
df = df[df["speaker_idx"].notna()].copy()
df["sentence"] = df["sentence"].fillna("").astype(str).str.strip()
df = df[df["sentence"].str.len() > 0].copy()

# Verify audio files
df["audio_path"] = df["path"].apply(lambda x: str(CLIPS / str(x)))
df = df[df["audio_path"].apply(lambda x: Path(x).is_file())].copy()

# Preserve original speaker IDs
df["speaker_idx"] = df["speaker_idx"].astype(int)

# Split each speaker separately: 90% train, 10% validation
train_parts = []
val_parts = []

for speaker_id, group in df.groupby("speaker_idx"):
    group = group.sample(frac=1, random_state=42)

    if len(group) < 2:
        train_parts.append(group)
        continue

    val_count = max(1, round(len(group) * 0.10))
    val_count = min(val_count, len(group) - 1)

    val_parts.append(group.iloc[:val_count])
    train_parts.append(group.iloc[val_count:])

train_df = pd.concat(train_parts).sample(frac=1, random_state=42)
val_df = pd.concat(val_parts).sample(frac=1, random_state=42)

# Export with both common naming conventions
columns = [
    "audio_path", "sentence", "speaker_idx",
    "path", "speaker_id", "client_id"
]

for part in [train_df, val_df]:
    part["speaker_id"] = part["speaker_idx"]

train_df[columns].to_csv(WORK / "inuts_train.csv", index=False)
val_df[columns].to_csv(WORK / "inuts_validation.csv", index=False)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TRAIN SPEAKERS:", train_df["speaker_idx"].nunique())
print("VALIDATION SPEAKERS:", val_df["speaker_idx"].nunique())
print("TRAIN ONLY SPEAKERS:", len(set(train_df.speaker_idx) - set(val_df.speaker_idx)))
print("VALIDATION ONLY SPEAKERS:", len(set(val_df.speaker_idx) - set(train_df.speaker_idx)))
print("\nFILES CREATED:")
print(WORK / "inuts_train.csv")
print(WORK / "inuts_validation.csv")
print("\nTRAIN PREVIEW:")
print(train_df[columns].head(3).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm

WORK = Path("/kaggle/working")

print("1/4 — Checking CSV files...")
for name in ["inuts_train.csv", "inuts_validation.csv"]:
    p = WORK / name
    df = pd.read_csv(p)
    print(f"\n{name}: {len(df)} rows")
    print("Columns:", list(df.columns))
    print("Missing values:")
    print(df[["audio_path", "sentence", "speaker_idx"]].isna().sum())

print("\n2/4 — Checking speaker distribution...")
train = pd.read_csv(WORK / "inuts_train.csv")
val = pd.read_csv(WORK / "inuts_validation.csv")

print("Train speaker IDs:", train.speaker_idx.nunique())
print("Validation speaker IDs:", val.speaker_idx.nunique())
print("Train speaker ID range:", train.speaker_idx.min(), "-", train.speaker_idx.max())
print("Validation speaker ID range:", val.speaker_idx.min(), "-", val.speaker_idx.max())

print("\n3/4 — Checking project Python files...")
py_files = list(WORK.glob("*.py"))
for p in tqdm(py_files, desc="Scanning scripts"):
    print("SCRIPT:", p.name)

print("\n4/4 — Checking existing checkpoint...")
ckpt_path = WORK / "inuts_urdu_training_latest.pth"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    print("Checkpoint:", ckpt_path.name)
    print("Global step:", ckpt.get("global_step"))
    print("Vocabulary entries:", len(ckpt.get("vocab", {})))
    print("Speaker entries:", len(ckpt.get("speaker_map", {})))
    print("Model components:")
    for key in ckpt.keys():
        if key not in ["optimizer_joint", "optimizer_d"]:
            print(" -", key)
else:
    print("Checkpoint not found at expected path.")

print("\nDIAGNOSTIC COMPLETE — No training started.")


In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

keywords = ["train", "model", "dataset", "config", "tts", "align"]

for root in roots:
    print(f"\n--- {root} ---")
    if not root.exists():
        continue

    found = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".py", ".yaml", ".yml", ".json"]:
            if any(k in p.name.lower() for k in keywords):
                found.append(p)

    for p in found[:100]:
        print(p)
    print("Files listed:", min(len(found), 100))


In [ ]:
from pathlib import Path
import os

work = Path("/kaggle/working")
dataset = Path("/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu")

print("=== WORKING FILES ===")
for p in sorted(work.glob("*")):
    print(p.name, f"({p.stat().st_size / 1024**2:.1f} MB)" if p.is_file() else "[folder]")

print("\n=== DATASET CHECK ===")
print("Dataset exists:", dataset.exists())

if dataset.exists():
    for name in ["validated.tsv", "clips"]:
        p = dataset / name
        print(name, "EXISTS" if p.exists() else "MISSING")

print("\n=== TRAINING FILES ===")
for name in [
    "inuts_train.csv",
    "inuts_validation.csv",
    "inuts_test.csv",
    "inuts_urdu_vocab.json",
    "inuts_speakers.json",
    "inuts_urdu_training_latest.pth"
]:
    p = work / name
    print(name, "FOUND" if p.exists() else "MISSING")


> **Legacy cell skipped:** Legacy CSV inspection cell caused a CSV field-size error. The corrected dataset manifest is rebuilt in cell 125.


In [ ]:
from pathlib import Path
import csv
import sys

# CSV field limit increase
limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(limit)
        break
    except OverflowError:
        limit //= 10

print("=== CHECKPOINT SEARCH ===")
found = []

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if root.exists():
        for pattern in ["*.pth", "*.pt", "*.ckpt", "*.safetensors"]:
            found.extend(root.rglob(pattern))

if found:
    for p in found:
        print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")
else:
    print("No checkpoint found.")

print("\n=== DATASET DETAILS ===")
dataset = Path("/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu")
tsv = dataset / "validated.tsv"

with tsv.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    print("Columns:", reader.fieldnames)
    count = sum(1 for _ in reader)

print("Validated records:", count)

clips = dataset / "clips"
print("Audio clips folder:", clips.exists())
print("Audio files:", sum(1 for p in clips.iterdir() if p.is_file()))

print("\n=== WORKING DIRECTORY ===")
for p in Path("/kaggle/working").iterdir():
    print(p.name)


In [ ]:
import torch
from pathlib import Path

files = [
    Path("/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/inuts_urdu_checkpoint_step1.pth"),
    Path("/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/inuts_urdu_training_latest.pth")
]

for path in files:
    print("\n" + "="*65)
    print("FILE:", path.name)

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)

        if isinstance(ckpt, dict):
            print("Top-level keys:", list(ckpt.keys()))

            for key, value in ckpt.items():
                if isinstance(value, dict):
                    print(f"\n{key}: dictionary with {len(value)} entries")
                    print("  Sample keys:", list(value.keys())[:8])
                elif isinstance(value, (int, float, str, bool)):
                    print(f"{key}: {value}")
                elif isinstance(value, (list, tuple)):
                    print(f"{key}: {type(value).__name__}, length={len(value)}")
                elif torch.is_tensor(value):
                    print(f"{key}: Tensor, shape={tuple(value.shape)}")
                else:
                    print(f"{key}: {type(value).__name__}")
        else:
            print("Checkpoint type:", type(ckpt).__name__)

    except Exception as e:
        print("ERROR:", repr(e))


In [ ]:
from pathlib import Path
import json

root = Path("/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep")

print("=== AVAILABLE FILES ===")
if root.exists():
    for p in root.rglob("*"):
        if p.is_file():
            print(p)
else:
    print("Notebook folder not found:", root)

print("\n=== NOTEBOOK CODE SEARCH ===")
notebooks = list(Path("/kaggle/input/notebooks").rglob("*.ipynb"))

print("Notebooks found:", len(notebooks))

keywords = [
    "class TextEncoder",
    "class WaveformGenerator",
    "class DurationPredictor",
    "class Posterior",
    "DataLoader",
    "global_step",
    "optimizer_joint",
    "torch.save",
    "Common Voice",
    "validated.tsv",
    "sampling_rate",
    "hop_length"
]

for nb in notebooks:
    print("\nNOTEBOOK:", nb)
    try:
        data = json.loads(nb.read_text(encoding="utf-8"))
        for i, cell in enumerate(data.get("cells", [])):
            source = "".join(cell.get("source", []))
            hits = [k for k in keywords if k.lower() in source.lower()]
            if hits:
                print(f"\n--- CELL {i} | MATCHES: {hits} ---")
                print(source[:2500])
    except Exception as e:
        print("Could not read:", repr(e))


In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm

nb_path = Path(
    "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/__notebook__.ipynb"
)

notebook = json.loads(nb_path.read_text(encoding="utf-8"))
cells = notebook.get("cells", [])

print("Total notebook cells:", len(cells))

# Pehle relevant cells identify karo
selected = []

for i, cell in enumerate(tqdm(cells, desc="Scanning notebook")):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if i < 48:
        continue

    first_line = source.strip().split("\n")[0] if source.strip() else ""

    if (
        "class " in source
        or "def monotonic_alignment_search" in source
        or "def save_training_checkpoint" in source
        or "RUN_STEPS" in source
    ):
        selected.append(i)
        print(f"Cell {i}: {first_line[:120]}")

print("\nRelevant cells found:", selected)

# Model architecture aur training code print karo
for i in selected:
    source = "".join(cells[i].get("source", []))
    print("\n" + "=" * 70)
    print("FULL CELL:", i)
    print("=" * 70)
    print(source)


In [ ]:
import torch

paths = [
    "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/inuts_urdu_checkpoint_step1.pth",
    "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/inuts_urdu_training_latest.pth"
]

for path in paths:
    print("\n" + "="*60)
    print("CHECKPOINT:", path.split("/")[-1])

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):
        print("Keys:", list(ckpt.keys()))
        print("Global step:", ckpt.get("global_step", "Not found"))

        for key in [
            "text_encoder", "text_prior",
            "posterior_encoder_v2", "speaker_encoder",
            "speaker_to_audio", "latent_flow",
            "duration_predictor", "waveform_generator",
            "mpd", "msd", "optimizer_joint",
            "optimizer_d", "vocab", "speaker_map"
        ]:
            print(key, ":", "FOUND" if key in ckpt else "MISSING")


In [ ]:
import torch

ckpt_path = "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/inuts_urdu_training_latest.pth"

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("GLOBAL STEP:", ckpt["global_step"])
print("VOCAB SIZE:", len(ckpt["vocab"]))
print("SPEAKER COUNT:", len(ckpt["speaker_map"]))

print("\nMODEL DETAILS")
model_keys = [
    "text_encoder",
    "text_prior",
    "posterior_encoder_v2",
    "speaker_encoder",
    "speaker_to_audio",
    "latent_flow",
    "duration_predictor",
    "waveform_generator",
    "mpd",
    "msd"
]

for name in model_keys:
    state = ckpt[name]
    print(f"\n{name}: {len(state)} parameters")

    for i, (key, value) in enumerate(state.items()):
        if torch.is_tensor(value):
            print(" ", key, tuple(value.shape))
        if i >= 2:
            break

print("\nOPTIMIZER DETAILS")
for name in ["optimizer_joint", "optimizer_d"]:
    opt = ckpt[name]
    print(name)
    print("  Parameter groups:", len(opt["param_groups"]))
    print("  Saved states:", len(opt["state"]))

print("\nCHECKPOINT INSPECTION COMPLETE")


In [ ]:
import torch

model_names = [
    "text_encoder",
    "text_prior",
    "posterior_encoder_v2",
    "speaker_encoder",
    "speaker_to_audio",
    "latent_flow",
    "duration_predictor",
    "waveform_generator",
    "mpd",
    "msd"
]

print("=== MODEL COMPATIBILITY ===")

for name in model_names:
    print(f"\n{name}:")

    if name not in globals():
        print("NOT INITIALIZED")
        continue

    model = globals()[name]
    saved = ckpt[name]

    if not isinstance(model, torch.nn.Module):
        print("Not a PyTorch module")
        continue

    current = model.state_dict()

    missing = set(saved) - set(current)
    extra = set(current) - set(saved)
    mismatched = [
        (k, tuple(saved[k].shape), tuple(current[k].shape))
        for k in set(saved) & set(current)
        if saved[k].shape != current[k].shape
    ]

    print("Current tensors:", len(current))
    print("Missing:", list(missing)[:5])
    print("Extra:", list(extra)[:5])
    print("Shape mismatches:", mismatched[:5])

    if not missing and not extra and not mismatched:
        print("COMPATIBLE")

print("\n=== VOCABULARY CHECK ===")
print("Vocab type:", type(ckpt["vocab"]).__name__)
print("First entries:", list(ckpt["vocab"].items())[:15]
      if isinstance(ckpt["vocab"], dict)
      else ckpt["vocab"][:15])

if isinstance(ckpt["vocab"], dict):
    ids = [v for v in ckpt["vocab"].values()
           if isinstance(v, int)]
    print("Token ID range:",
          (min(ids), max(ids)) if ids else "No integer IDs")

print("\n=== OPTIMIZERS ===")
for name in ["optimizer_joint", "optimizer_d"]:
    print(name, "initialized:", name in globals())

print("\nINSPECTION FINISHED — NO WEIGHTS LOADED")


In [ ]:
# Restore exact training metadata from checkpoint
vocab = ckpt["vocab"]
speaker_map = ckpt["speaker_map"]

NUM_SYMBOLS = len(vocab)
NUM_SPEAKERS = len(speaker_map)

print("Vocabulary restored:", NUM_SYMBOLS)
print("Speakers restored:", NUM_SPEAKERS)
print("Special tokens:", {
    token: vocab[token]
    for token in ["<PAD>", "<UNK>"]
    if token in vocab
})

print("Speaker sample:", list(speaker_map.items())[:5])


In [ ]:
import os
import json

ROOTS = [
    "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep",
    "/kaggle/working"
]

keywords = [
    "UrduTextEncoder",
    "UrduPosteriorEncoderV2",
    "UrduAudioEncoder",
    "UrduTextPrior",
    "UrduSpeakerEmbedding",
    "UrduInvertibleFlow",
    "UrduDurationPredictor",
    "UrduWaveformGenerator",
    "UrduMultiPeriodDiscriminator",
    "UrduMultiScaleDiscriminator"
]

found = 0

for root in ROOTS:
    if not os.path.exists(root):
        continue

    for folder, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in [".git"]]

        for filename in files:
            path = os.path.join(folder, filename)

            if filename.endswith(".ipynb"):
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        notebook = json.load(f)

                    for i, cell in enumerate(notebook.get("cells", [])):
                        source = "".join(cell.get("source", []))

                        if any(k in source for k in keywords):
                            print("\n" + "=" * 70)
                            print("FILE:", path)
                            print("CELL:", i)
                            print("=" * 70)
                            print(source)
                            found += 1

                except Exception as e:
                    print("Read error:", path, str(e))

            elif filename.endswith(".py"):
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        source = f.read()

                    if any(k in source for k in keywords):
                        print("\nFILE:", path)
                        print(source)
                        found += 1

                except Exception:
                    pass

print("\nTOTAL MATCHES:", found)


In [ ]:
import os
import json
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Original notebook
notebook_path = (
    "/kaggle/input/notebooks/inutsumair/"
    "inuts-urdu-tts-prep/__notebook__.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Load original saved checkpoint
ckpt_path = (
    "/kaggle/input/notebooks/inutsumair/"
    "inuts-urdu-tts-prep/inuts_urdu_training_latest.pth"
)

ckpt = torch.load(
    ckpt_path, map_location="cpu", weights_only=False
)

# Confirm original symbol count from older checkpoint
old_path = (
    "/kaggle/input/notebooks/inutsumair/"
    "inuts-urdu-tts-prep/inuts_urdu_checkpoint_step1.pth"
)

old_ckpt = torch.load(
    old_path, map_location="cpu", weights_only=False
)

print("OLD CHECKPOINT NUM SYMBOLS:",
      old_ckpt.get("num_symbols", "Not available"))

# Extract only required class definitions.
required_classes = {
    "SinusoidalPositionEncoding",
    "UrduTextEncoder",
    "UrduDurationPredictor",
    "UrduTextPrior",
    "AffineCoupling",
    "UrduInvertibleFlow",
    "UrduSpeakerEmbedding",
    "UrduPosteriorEncoderV2",
    "ResidualBlock",
    "UrduWaveformGenerator",
    "PeriodDiscriminator",
    "UrduMultiPeriodDiscriminator",
    "ScaleDiscriminator",
    "UrduMultiScaleDiscriminator"
}

extracted = set()

for cell in notebook["cells"]:
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            if node.name in required_classes:
                class_source = ast.get_source_segment(
                    source, node
                )
                if class_source and node.name not in extracted:
                    exec(class_source, globals())
                    extracted.add(node.name)

missing_classes = required_classes - extracted

print("\nEXTRACTED CLASSES:", sorted(extracted))
print("MISSING CLASSES:", sorted(missing_classes))

assert not missing_classes, (
    "Some original model classes were not found."
)

# Restore exact metadata
vocab = ckpt["vocab"]
speaker_map = ckpt["speaker_map"]

NUM_SYMBOLS = ckpt["text_encoder"]["embedding.weight"].shape[0]
NUM_SPEAKERS = len(speaker_map)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("\nVOCAB TOKENS:", len(vocab))
print("MODEL EMBEDDING ROWS:", NUM_SYMBOLS)
print("SPEAKERS:", NUM_SPEAKERS)
print("DEVICE:", DEVICE)

# Initialize models using checkpoint-compatible dimensions
models = {
    "text_encoder": UrduTextEncoder(
        vocab_size=NUM_SYMBOLS
    ),
    "text_prior": UrduTextPrior(),
    "posterior_encoder_v2": UrduPosteriorEncoderV2(),
    "speaker_encoder": UrduSpeakerEmbedding(
        num_speakers=NUM_SPEAKERS,
        embedding_dim=192
    ),
    "speaker_to_audio": nn.Linear(192, 192),
    "latent_flow": UrduInvertibleFlow(
        channels=192, layers=4
    ),
    "duration_predictor": UrduDurationPredictor(),
    "waveform_generator": UrduWaveformGenerator(),
    "mpd": UrduMultiPeriodDiscriminator(),
    "msd": UrduMultiScaleDiscriminator()
}

# Strictly restore every model
for name, model in models.items():
    model = model.to(DEVICE)
    model.load_state_dict(ckpt[name], strict=True)
    globals()[name] = model
    print("RESTORED:", name)

print("\nALL MODEL WEIGHTS RESTORED SUCCESSFULLY")
print("Training step:", ckpt["global_step"])


In [ ]:
import json

notebook_path = (
    "/kaggle/input/notebooks/inutsumair/"
    "inuts-urdu-tts-prep/__notebook__.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "optimizer_joint",
    "optimizer_d",
    "joint_parameters",
    "AdamW",
    "torch.optim.Adam",
    "lr_scheduler",
    "learning_rate",
    "betas="
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(k in source for k in keywords):
        print("\n" + "=" * 75)
        print("ORIGINAL CELL:", i)
        print("=" * 75)
        print(source)

print("\nOPTIMIZER SCAN COMPLETE")


In [ ]:
import torch

# Exact parameter order from original Cell 91
joint_modules = [
    text_encoder,
    text_prior,
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    duration_predictor,
    waveform_generator
]

joint_parameters = []
seen_parameters = set()

for module in joint_modules:
    for parameter in module.parameters():
        if parameter.requires_grad and id(parameter) not in seen_parameters:
            joint_parameters.append(parameter)
            seen_parameters.add(id(parameter))

discriminator_parameters = (
    list(mpd.parameters()) + list(msd.parameters())
)

# Create optimizers using original settings
optimizer_joint = torch.optim.AdamW(
    joint_parameters,
    lr=1e-4,
    betas=(0.8, 0.99),
    weight_decay=0.01
)

optimizer_d = torch.optim.AdamW(
    discriminator_parameters,
    lr=2e-4,
    betas=(0.8, 0.99)
)

# Inspect saved optimizer states before restoring
saved_joint = ckpt["optimizer_joint"]
saved_d = ckpt["optimizer_d"]

saved_joint_count = len(saved_joint["param_groups"][0]["params"])
saved_d_count = len(saved_d["param_groups"][0]["params"])

print("JOINT PARAMETERS")
print("Current:", len(joint_parameters))
print("Checkpoint:", saved_joint_count)

print("\nDISCRIMINATOR PARAMETERS")
print("Current:", len(discriminator_parameters))
print("Checkpoint:", saved_d_count)

assert len(joint_parameters) == saved_joint_count, \
    "Joint optimizer parameter count mismatch!"

assert len(discriminator_parameters) == saved_d_count, \
    "Discriminator optimizer parameter count mismatch!"

# Restore optimizer states
optimizer_joint.load_state_dict(saved_joint)
optimizer_d.load_state_dict(saved_d)

global_step = ckpt["global_step"]

print("\nOPTIMIZER RESTORE SUCCESSFUL")
print("Global step:", global_step)
print("Joint learning rate:",
      optimizer_joint.param_groups[0]["lr"])
print("Discriminator learning rate:",
      optimizer_d.param_groups[0]["lr"])
print("Joint saved states:", len(optimizer_joint.state))
print("Discriminator saved states:", len(optimizer_d.state))


In [ ]:
import os

required = [
    "train_loader",
    "train_df",
    "validation_df",
    "spectrogram_transform",
    "mel_loss_transform",
    "monotonic_alignment_search",
    "vocab",
    "speaker_map",
    "text_encoder",
    "text_prior",
    "posterior_encoder_v2",
    "speaker_encoder",
    "speaker_to_audio",
    "latent_flow",
    "duration_predictor",
    "waveform_generator",
    "mpd",
    "msd",
    "optimizer_joint",
    "optimizer_d",
    "global_step",
]

print("===== TRAINING READINESS CHECK =====")

for name in required:
    exists = name in globals()
    if exists:
        obj = globals()[name]
        if name == "train_loader":
            detail = f"batches: {len(obj)}"
        elif name in ["train_df", "validation_df"]:
            detail = f"rows: {len(obj)}"
        else:
            detail = type(obj).__name__
        print(f"OK   {name}: {detail}")
    else:
        print(f"MISSING   {name}")

print("\n===== DATASET CHECK =====")
dataset_path = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"

for path in [
    dataset_path,
    os.path.join(dataset_path, "validated.tsv"),
    os.path.join(dataset_path, "clips"),
]:
    print(("OK   " if os.path.exists(path) else "MISSING   ") + path)

print("\nCurrent training step:", global_step)


In [ ]:
import json
import re

NOTEBOOK = "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/__notebook__.ipynb"

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "train_loader",
    "validation_df",
    "spectrogram_transform",
    "mel_loss_transform",
    "monotonic_alignment_search",
    "DataLoader",
    "Dataset",
    "MelSpectrogram",
    "Spectrogram",
    "train_test_split",
    "collate_fn"
]

print("===== ORIGINAL NOTEBOOK: DATA PIPELINE SCAN =====")

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    hits = [word for word in keywords if word.lower() in source.lower()]

    if hits:
        print(f"\nCELL {i} | MATCHES: {', '.join(hits)}")
        print("CODE LENGTH:", len(source))
        matching_lines = [
            line.strip()[:220]
            for line in source.splitlines()
            if any(word.lower() in line.lower() for word in keywords)
        ]
        for line in matching_lines[:15]:
            print(" ", line)

In [ ]:
import json

NOTEBOOK = "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/__notebook__.ipynb"

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

target_cells = [34, 48, 53, 65, 72, 74, 76, 81]

for i in target_cells:
    source = "".join(nb["cells"][i].get("source", []))
    print("\n" + "=" * 70)
    print(f"ORIGINAL CELL {i}")
    print("=" * 70)
    print(source)

In [ ]:
items = [
    "train_loader",
    "train_df",
    "validation_df",
    "spectrogram_transform",
    "mel_loss_transform",
    "monotonic_alignment_search",
    "global_step",
    "optimizer_joint",
    "optimizer_d"
]

for name in items:
    print(name, "AVAILABLE" if name in globals() else "MISSING")

In [ ]:
import os
import csv
import sys
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/cookiefinder/mozilla-common-voice-urdu"
CLIPS = os.path.join(BASE, "clips")

# Large TSV fields handle karo
csv.field_size_limit(sys.maxsize)

print("Loading Common Voice Urdu dataset...")

df = pd.read_csv(
    os.path.join(BASE, "validated.tsv"),
    sep="\t",
    on_bad_lines="skip",
    engine="python"
)

df = df.dropna(subset=["client_id", "path", "sentence"]).copy()

df["sentence"] = (
    df["sentence"].astype(str)
    .str.normalize("NFC")
    .str.replace("ـ", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["sentence"] != ""].copy()

print("Checking audio files...")

df["audio_exists"] = df["path"].apply(
    lambda x: os.path.isfile(os.path.join(CLIPS, str(x)))
)

df = df[df["audio_exists"]].drop(columns=["audio_exists"])
df = df.reset_index(drop=True)

# Same original speaker-wise split
speakers = df["client_id"].unique().copy()
rng = np.random.default_rng(42)
rng.shuffle(speakers)

train_speakers = set(speakers[:330])
val_speakers = set(speakers[330:372])
test_speakers = set(speakers[372:])

train_df = df[df["client_id"].isin(train_speakers)].copy()
validation_df = df[df["client_id"].isin(val_speakers)].copy()
test_df = df[df["client_id"].isin(test_speakers)].copy()

# Save manifests
train_df.to_csv("/kaggle/working/inuts_train.csv", index=False)
validation_df.to_csv("/kaggle/working/inuts_validation.csv", index=False)
test_df.to_csv("/kaggle/working/inuts_test.csv", index=False)

print("\n===== DATASET RESTORED =====")
print("Training records:", len(train_df))
print("Validation records:", len(validation_df))
print("Test records:", len(test_df))
print("Training speakers:", train_df["client_id"].nunique())

# Verify checkpoint speaker mapping
mapped_speakers = set(str(x) for x in speaker_map.keys())
current_speakers = set(train_df["client_id"].astype(str).unique())

print("\n===== SPEAKER MAP CHECK =====")
print("Checkpoint speakers:", len(mapped_speakers))
print("Dataset training speakers:", len(current_speakers))
print("Unmapped training speakers:", len(current_speakers - mapped_speakers))
print("Extra checkpoint speakers:", len(mapped_speakers - current_speakers))

assert current_speakers == mapped_speakers, \
    "Speaker mapping does not match original training split!"

print("\nDATASET AND SPEAKER MAPPING VERIFIED!")

In [ ]:
import os
import torch
import numpy as np
import soundfile as sf
import librosa

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SAMPLE_RATE = 24000
BATCH_SIZE = 2
NUM_WORKERS = 0

PAD_ID = vocab["<PAD>"]
UNK_ID = vocab["<UNK>"]
BLANK_ID = len(vocab)

# Existing checkpoint mapping must remain unchanged
assert NUM_SYMBOLS == 173
assert NUM_SPEAKERS == 330
assert len(speaker_map) == 330

class UrduTTSDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def encode_text(self, sentence):
        ids = [BLANK_ID]
        for char in sentence:
            ids.append(vocab.get(char, UNK_ID))
            ids.append(BLANK_ID)
        return torch.tensor(ids, dtype=torch.long)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        text = self.encode_text(str(row["sentence"]))

        speaker_key = str(row["client_id"])
        assert speaker_key in speaker_map, f"Unknown speaker: {speaker_key}"

        audio_path = os.path.join(CLIPS, str(row["path"]))
        audio, sr = sf.read(audio_path, dtype="float32")

        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        if sr != SAMPLE_RATE:
            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=SAMPLE_RATE
            )

        audio = np.nan_to_num(audio)
        audio = np.clip(audio, -1.0, 1.0)

        return {
            "text": text,
            "audio": torch.tensor(audio, dtype=torch.float32),
            "speaker": torch.tensor(
                speaker_map[speaker_key], dtype=torch.long
            ),
            "text_length": len(text),
            "audio_length": len(audio)
        }


def collate_fn(batch):
    texts = [x["text"] for x in batch]
    audios = [x["audio"] for x in batch]

    return {
        "text": pad_sequence(
            texts, batch_first=True, padding_value=PAD_ID
        ),
        "text_lengths": torch.tensor(
            [x["text_length"] for x in batch]
        ),
        "audio": pad_sequence(
            audios, batch_first=True, padding_value=0
        ).unsqueeze(1),
        "audio_lengths": torch.tensor(
            [x["audio_length"] for x in batch]
        ),
        "speaker": torch.stack([x["speaker"] for x in batch])
    }


train_dataset = UrduTTSDataset(train_df)
validation_dataset = UrduTTSDataset(validation_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

print("TRAIN LOADER RESTORED")
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Checkpoint step:", global_step)

# Test one batch
batch = next(iter(train_loader))

print("\n===== FIRST BATCH CHECK =====")
for key, value in batch.items():
    print(key, tuple(value.shape), value.dtype)

print("\nSpeaker IDs:", batch["speaker"].tolist())
print("AUDIO LOADER TEST SUCCESSFUL")

In [ ]:
import torchaudio

spectrogram_transform = torchaudio.transforms.Spectrogram(
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    power=1.0,
    center=True
).to(DEVICE)

mel_loss_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=24000,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    n_mels=80,
    f_min=0,
    f_max=12000,
    power=1.0
).to(DEVICE)

print("Spectrogram transform:", "READY")
print("Mel transform:", "READY")
print("Training step:", global_step)

In [ ]:
import torch

def monotonic_alignment_search(scores, text_lengths, audio_lengths):
    B, max_text, max_audio = scores.shape

    alignment = torch.zeros_like(scores)
    durations = torch.zeros(
        B, max_text,
        dtype=torch.long,
        device=scores.device
    )

    with torch.no_grad():
        for b in range(B):
            n = int(text_lengths[b].item())
            t = int(audio_lengths[b].item())

            if n < 1 or t < n:
                raise ValueError(
                    f"Invalid alignment lengths: text={n}, audio={t}"
                )

            s = scores[b, :n, :t].float()

            dp = torch.full(
                (n, t),
                float("-inf"),
                device=scores.device
            )

            back = torch.zeros(
                (n, t),
                dtype=torch.bool,
                device=scores.device
            )

            dp[0, 0] = s[0, 0]

            for j in range(1, t):
                max_i = min(n - 1, j)

                for i in range(max_i + 1):
                    if n - 1 - i > t - 1 - j:
                        continue

                    stay = dp[i, j - 1]

                    advance = (
                        dp[i - 1, j - 1]
                        if i > 0
                        else torch.tensor(
                            float("-inf"),
                            device=scores.device
                        )
                    )

                    if advance > stay:
                        dp[i, j] = advance + s[i, j]
                        back[i, j] = True
                    else:
                        dp[i, j] = stay + s[i, j]

            if not torch.isfinite(dp[n - 1, t - 1]):
                raise RuntimeError("MAS alignment failed")

            i = n - 1

            for j in range(t - 1, -1, -1):
                alignment[b, i, j] = 1

                if j > 0 and back[i, j]:
                    i -= 1

            durations[b, :n] = (
                alignment[b, :n, :t].sum(dim=1).long()
            )

    return alignment, durations


print("MAS FUNCTION RESTORED")
print("Current training step:", global_step)

In [ ]:
import json

NOTEBOOK = "/kaggle/input/notebooks/inutsumair/inuts-urdu-tts-prep/__notebook__.ipynb"

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if i == 95:
        print(source)
        break

> **Legacy cell skipped:** Legacy preflight cell referenced generator_modules before initialization. The canonical setup is in cells 131–134.


In [ ]:
generator_modules = [
    text_encoder,
    text_prior,
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    duration_predictor,
    waveform_generator
]

discriminator_modules = [
    mpd,
    msd
]

print("Generator modules:", len(generator_modules))
print("Discriminator modules:", len(discriminator_modules))
print("Global step:", global_step)
print("Optimizer state preserved:", len(optimizer_joint.state))

In [ ]:
joint_parameters = []
seen_params = set()

for module in generator_modules:
    for parameter in module.parameters():
        if parameter.requires_grad and id(parameter) not in seen_params:
            joint_parameters.append(parameter)
            seen_params.add(id(parameter))

saved_optimizer_params = [
    p
    for group in optimizer_joint.param_groups
    for p in group["params"]
]

print("Current parameters:", len(joint_parameters))
print("Optimizer parameters:", len(saved_optimizer_params))

assert len(joint_parameters) == len(saved_optimizer_params), \
    "Parameter count mismatch!"

assert all(
    id(a) == id(b)
    for a, b in zip(joint_parameters, saved_optimizer_params)
), "Parameter order mismatch!"

print("JOINT OPTIMIZER VERIFIED!")

In [ ]:
import os
import shutil
import torch

# 1. Verify data loader keys
batch = next(iter(train_loader))

print("Batch keys:", list(batch.keys()))

assert "speaker" in batch
assert "speaker_ids" not in batch

print("Speaker key verified: batch['speaker']")

# 2. Check audio/text alignment
audio = batch["audio"].to(DEVICE)
audio_lengths = batch["audio_lengths"].to(DEVICE)
text_lengths = batch["text_lengths"].to(DEVICE)

with torch.no_grad():
    spec = spectrogram_transform(audio.squeeze(1))
    frame_lengths = (audio_lengths // 256 + 1).clamp(
        max=spec.size(-1)
    )

print("Text lengths:", text_lengths.tolist())
print("Audio frame lengths:", frame_lengths.tolist())

print(
    "MAS feasible:",
    bool(torch.all(frame_lengths >= text_lengths))
)

# 3. Backup step-500 checkpoint
SOURCE_CHECKPOINT = (
    "/kaggle/input/notebooks/inutsumair/"
    "inuts-urdu-tts-prep/inuts_urdu_training_latest.pth"
)

BACKUP_CHECKPOINT = (
    "/kaggle/working/inuts_urdu_training_step500_backup.pth"
)

assert os.path.isfile(SOURCE_CHECKPOINT), \
    "Original checkpoint not found at expected path!"

if not os.path.exists(BACKUP_CHECKPOINT):
    shutil.copy2(SOURCE_CHECKPOINT, BACKUP_CHECKPOINT)

print("Backup exists:", os.path.isfile(BACKUP_CHECKPOINT))
print("Global step:", global_step)
print("PREFLIGHT CHECK COMPLETE")

In [ ]:
import torch

bad_batches = 0
total_samples = 0
bad_samples = 0

for batch_index, batch in enumerate(train_loader):
    if batch_index >= 100:
        break

    audio = batch["audio"].to(DEVICE)
    audio_lengths = batch["audio_lengths"].to(DEVICE)
    text_lengths = batch["text_lengths"].to(DEVICE)

    with torch.no_grad():
        spec = spectrogram_transform(audio.squeeze(1))
        frame_lengths = (audio_lengths // 256 + 1).clamp(
            max=spec.size(-1)
        )

    valid = frame_lengths >= text_lengths

    total_samples += len(valid)
    bad_samples += int((~valid).sum().item())

    if not bool(valid.all()):
        bad_batches += 1

print("Batches checked:", min(100, len(train_loader)))
print("Total samples:", total_samples)
print("Invalid samples:", bad_samples)
print("Batches with alignment problems:", bad_batches)
print("ALIGNMENT AUDIT COMPLETE")

In [ ]:
import os
import math
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ==========================================
# INUTS URDU TTS - SAFE CONTINUOUS TRAINING
# ==========================================

TARGET_GLOBAL_STEP = 1500
SAVE_EVERY = 100

CHECKPOINT_PATH = (
    "/kaggle/working/inuts_urdu_training_latest.pth"
)

PRE_RUN_BACKUP = (
    "/kaggle/working/inuts_urdu_training_before_duration_fix.pth"
)

os.makedirs("/kaggle/working", exist_ok=True)

# ------------------------------------------
# 1. MODEL COMPONENTS
# ------------------------------------------

generator_modules = [
    text_encoder,
    text_prior,
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    duration_predictor,
    waveform_generator
]

discriminator_modules = [mpd, msd]

assert "global_step" in globals(), "Training step missing!"
assert global_step <= TARGET_GLOBAL_STEP
assert "optimizer_joint" in globals()
assert "optimizer_d" in globals()
assert "joint_parameters" in globals()

# Verify optimizer parameter identity and order
current_parameters = []
seen = set()

for module in generator_modules:
    for p in module.parameters():
        if p.requires_grad and id(p) not in seen:
            current_parameters.append(p)
            seen.add(id(p))

saved_parameters = [
    p
    for group in optimizer_joint.param_groups
    for p in group["params"]
]

assert len(current_parameters) == 138
assert len(saved_parameters) == 138
assert all(
    id(a) == id(b)
    for a, b in zip(current_parameters, saved_parameters)
)

joint_parameters = current_parameters

# ------------------------------------------
# 2. CHECKPOINT FUNCTION
# ------------------------------------------

def make_checkpoint():
    return {
        "global_step": global_step,
        "text_encoder": text_encoder.state_dict(),
        "text_prior": text_prior.state_dict(),
        "posterior_encoder_v2": posterior_encoder_v2.state_dict(),
        "speaker_encoder": speaker_encoder.state_dict(),
        "speaker_to_audio": speaker_to_audio.state_dict(),
        "latent_flow": latent_flow.state_dict(),
        "duration_predictor": duration_predictor.state_dict(),
        "waveform_generator": waveform_generator.state_dict(),
        "mpd": mpd.state_dict(),
        "msd": msd.state_dict(),
        "optimizer_joint": optimizer_joint.state_dict(),
        "optimizer_d": optimizer_d.state_dict(),
        "vocab": vocab,
        "speaker_map": speaker_map
    }


def save_training_checkpoint(path=CHECKPOINT_PATH):
    torch.save(make_checkpoint(), path)
    print(f"\nCheckpoint saved: step {global_step}")


# Preserve the starting point separately
if not os.path.exists(PRE_RUN_BACKUP):
    torch.save(make_checkpoint(), PRE_RUN_BACKUP)

assert os.path.isfile(PRE_RUN_BACKUP)

print("Starting global step:", global_step)
print("Target global step:", TARGET_GLOBAL_STEP)
print("Pre-training safety copy verified.")

# ------------------------------------------
# 3. TRAINING MODE
# ------------------------------------------

for module in generator_modules + discriminator_modules:
    module.train()

running_g = 0.0
running_d = 0.0
running_mel = 0.0
running_kl = 0.0
running_duration = 0.0

completed_this_run = 0
skipped_batches = 0

remaining_steps = TARGET_GLOBAL_STEP - global_step

progress = tqdm(
    total=remaining_steps,
    desc="INUTS Urdu TTS Training"
)

# ------------------------------------------
# 4. TRAINING LOOP
# ------------------------------------------

for batch in train_loader:

    if global_step >= TARGET_GLOBAL_STEP:
        break

    # Correct dataset keys
    text = batch["text"].to(DEVICE)
    text_lengths = batch["text_lengths"].to(DEVICE)
    audio = batch["audio"].to(DEVICE)
    audio_lengths = batch["audio_lengths"].to(DEVICE)
    speaker_ids = batch["speaker"].to(DEVICE)

    # --------------------------------------
    # AUDIO/TEXT ALIGNMENT SAFETY
    # --------------------------------------

    with torch.no_grad():
        check_spec = spectrogram_transform(audio.squeeze(1))

        audio_frame_lengths = (
            audio_lengths // 256 + 1
        ).clamp(max=check_spec.size(-1))

    if not bool(torch.all(audio_frame_lengths >= text_lengths)):
        skipped_batches += 1
        continue

    if bool(torch.any(audio_frame_lengths < 1)):
        skipped_batches += 1
        continue

    optimizer_joint.zero_grad(set_to_none=True)
    optimizer_d.zero_grad(set_to_none=True)

    # --------------------------------------
    # 1. TEXT + SPEAKER ENCODING
    # --------------------------------------

    text_features = text_encoder(text, text_lengths)
    speaker_features = speaker_encoder(speaker_ids)

    conditioned_text = (
        text_features + speaker_features.unsqueeze(1)
    )

    prior_mean, prior_log_scale = text_prior(conditioned_text)

    # --------------------------------------
    # 2. POSTERIOR ENCODER
    # --------------------------------------

    linear_spec = check_spec

    posterior_hidden = posterior_encoder_v2.pre(linear_spec)
    posterior_hidden = posterior_encoder_v2.encoder(
        posterior_hidden
    )

    posterior_hidden = (
        posterior_hidden
        + speaker_to_audio(speaker_features).unsqueeze(-1)
    )

    posterior_mean = posterior_encoder_v2.mean(
        posterior_hidden
    )

    posterior_log_scale = posterior_encoder_v2.log_scale(
        posterior_hidden
    ).clamp(-7, 2)

    # --------------------------------------
    # 3. POSTERIOR SAMPLING + FLOW
    # --------------------------------------

    posterior_latent = (
        posterior_mean
        + torch.exp(posterior_log_scale)
        * torch.randn_like(posterior_mean)
    )

    flow_outputs = []
    flow_logdets = []

    for i in range(audio.size(0)):

        length = int(audio_frame_lengths[i].item())

        z_i, logdet_i = latent_flow(
            posterior_latent[i:i+1, :, :length]
        )

        z_i = F.pad(
            z_i,
            (0, posterior_latent.size(-1) - length)
        )

        flow_outputs.append(z_i)
        flow_logdets.append(logdet_i.reshape(-1)[0])

    flow_latent = torch.cat(flow_outputs, dim=0)
    flow_logdets = torch.stack(flow_logdets)

    # --------------------------------------
    # 4. MONOTONIC ALIGNMENT
    # --------------------------------------

    with torch.no_grad():

        scores = -0.5 * (
            (
                (
                    flow_latent.detach().unsqueeze(2)
                    - prior_mean.detach().unsqueeze(-1)
                )
                * torch.exp(
                    -prior_log_scale.detach().unsqueeze(-1)
                )
            ) ** 2
            + 2 * prior_log_scale.detach().unsqueeze(-1)
            + math.log(2 * math.pi)
        ).sum(dim=1)

        text_mask = (
            torch.arange(text.size(1), device=DEVICE)[None, :]
            < text_lengths[:, None]
        )

        audio_mask = (
            torch.arange(scores.size(2), device=DEVICE)[None, :]
            < audio_frame_lengths[:, None]
        )

        scores = scores.masked_fill(
            ~text_mask.unsqueeze(-1), -1e9
        )

        scores = scores.masked_fill(
            ~audio_mask.unsqueeze(1), -1e9
        )

        alignment, duration_targets = monotonic_alignment_search(
            scores,
            text_lengths,
            audio_frame_lengths
        )

    alignment = alignment.detach()

    # --------------------------------------
    # 4B. DURATION PREDICTOR TRAINING
    # --------------------------------------
    # MAS durations are the supervised targets for text-to-audio length.
    duration_predictions = duration_predictor(conditioned_text)

    valid_text_mask = (
        torch.arange(text.size(1), device=DEVICE)[None, :]
        < text_lengths[:, None]
    )

    duration_targets_log = torch.log(
        duration_targets.float() + 1e-6
    )

    duration_loss = F.mse_loss(
        duration_predictions[valid_text_mask],
        duration_targets_log[valid_text_mask]
    )

    # --------------------------------------
    # 5. KL LOSS
    # --------------------------------------

    aligned_prior_mean = torch.bmm(
        prior_mean, alignment
    )

    aligned_prior_log_scale = torch.bmm(
        prior_log_scale, alignment
    )

    logq = -0.5 * (
        (
            (posterior_latent - posterior_mean)
            * torch.exp(-posterior_log_scale)
        ) ** 2
        + 2 * posterior_log_scale
        + math.log(2 * math.pi)
    ).sum(dim=1)

    logp = -0.5 * (
        (
            (flow_latent - aligned_prior_mean)
            * torch.exp(-aligned_prior_log_scale)
        ) ** 2
        + 2 * aligned_prior_log_scale
        + math.log(2 * math.pi)
    ).sum(dim=1)

    kl_values = []

    for i in range(audio.size(0)):

        length = int(audio_frame_lengths[i].item())

        kl_i = (
            logq[i, :length].sum()
            - flow_logdets[i]
            - logp[i, :length].sum()
        ) / length

        kl_values.append(kl_i)

    kl_loss = torch.stack(kl_values).mean()

    # --------------------------------------
    # 6. WAVEFORM GENERATION
    # --------------------------------------

    generated_audio = waveform_generator(flow_latent)

    assert generated_audio.ndim == 3
    assert generated_audio.size(1) == 1

    common_length = min(
        int(audio_lengths.min().item()),
        generated_audio.size(-1),
        audio.size(-1)
    )

    assert common_length > 0

    real_common = audio[..., :common_length]
    fake_common = generated_audio[..., :common_length]

    # --------------------------------------
    # 7. DISCRIMINATOR UPDATE
    # --------------------------------------

    real_mpd_scores, _ = mpd(real_common)
    fake_mpd_scores, _ = mpd(fake_common.detach())

    real_msd_scores, _ = msd(real_common)
    fake_msd_scores, _ = msd(fake_common.detach())

    d_loss = torch.tensor(0.0, device=DEVICE)

    for real_score, fake_score in zip(
        real_mpd_scores + real_msd_scores,
        fake_mpd_scores + fake_msd_scores
    ):

        d_loss = d_loss + 0.5 * (
            torch.mean((real_score - 1) ** 2)
            + torch.mean(fake_score ** 2)
        )

    if not torch.isfinite(d_loss):
        raise RuntimeError("Non-finite discriminator loss!")

    d_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(mpd.parameters()) + list(msd.parameters()),
        10.0
    )

    optimizer_d.step()
    optimizer_d.zero_grad(set_to_none=True)

    # --------------------------------------
    # 8. GENERATOR MEL LOSS
    # --------------------------------------

    mel_values = []

    for i in range(audio.size(0)):

        length = min(
            int(audio_lengths[i].item()),
            generated_audio.size(-1),
            audio.size(-1)
        )

        fake_wave = generated_audio[i, 0, :length]
        real_wave = audio[i, 0, :length]

        fake_mel = mel_loss_transform(fake_wave)
        real_mel = mel_loss_transform(real_wave)

        mel_values.append(
            F.l1_loss(
                torch.log(fake_mel + 1e-5),
                torch.log(real_mel + 1e-5)
            )
        )

    mel_loss = torch.stack(mel_values).mean()

    # --------------------------------------
    # 9. GENERATOR ADVERSARIAL + FEATURE LOSS
    # --------------------------------------

    for module in discriminator_modules:
        for parameter in module.parameters():
            parameter.requires_grad_(False)

    try:

        fake_mpd_scores, fake_mpd_features = mpd(fake_common)
        real_mpd_scores, real_mpd_features = mpd(real_common)

        fake_msd_scores, fake_msd_features = msd(fake_common)
        real_msd_scores, real_msd_features = msd(real_common)

        fake_scores = fake_mpd_scores + fake_msd_scores

        real_features = (
            real_mpd_features + real_msd_features
        )

        fake_features = (
            fake_mpd_features + fake_msd_features
        )

        adversarial_loss = sum(
            torch.mean((score - 1) ** 2)
            for score in fake_scores
        )

        feature_loss = torch.tensor(0.0, device=DEVICE)

        for real_group, fake_group in zip(
            real_features, fake_features
        ):

            for real_feature, fake_feature in zip(
                real_group, fake_group
            ):

                feature_loss = feature_loss + F.l1_loss(
                    fake_feature,
                    real_feature.detach()
                )

        # ----------------------------------
        # 10. TOTAL GENERATOR LOSS
        # ----------------------------------

        generator_loss = (
            mel_loss
            + 0.1 * kl_loss
            + 0.1 * duration_loss
            + adversarial_loss
            + 2.0 * feature_loss
        )

        if not torch.isfinite(generator_loss):
            raise RuntimeError(
                f"Non-finite generator loss at step {global_step + 1}"
            )

        generator_loss.backward()

    finally:

        for module in discriminator_modules:
            for parameter in module.parameters():
                parameter.requires_grad_(True)

    # --------------------------------------
    # 11. GENERATOR UPDATE
    # --------------------------------------

    torch.nn.utils.clip_grad_norm_(
        joint_parameters, 1.0
    )

    optimizer_joint.step()

    # Count only successfully completed updates
    global_step += 1
    completed_this_run += 1

    # --------------------------------------
    # 12. METRICS
    # --------------------------------------

    g_value = generator_loss.item()
    d_value = d_loss.item()
    mel_value = mel_loss.item()
    kl_value = kl_loss.item()
    duration_value = duration_loss.item()

    running_g += g_value
    running_d += d_value
    running_mel += mel_value
    running_kl += kl_value
    running_duration += duration_value

    progress.update(1)

    progress.set_postfix({
        "Step": global_step,
        "G": f"{g_value:.3f}",
        "D": f"{d_value:.3f}",
        "Mel": f"{mel_value:.3f}",
        "KL": f"{kl_value:.3f}",
        "Dur": f"{duration_value:.3f}"
    })

    if global_step % 10 == 0:

        print(
            f"\nStep {global_step} | "
            f"G: {g_value:.4f} | "
            f"D: {d_value:.4f} | "
            f"Mel: {mel_value:.4f} | "
            f"KL: {kl_value:.4f} | "
            f"Dur: {duration_value:.4f}"
        )

    # --------------------------------------
    # 13. AUTOMATIC CHECKPOINT
    # --------------------------------------

    if global_step % SAVE_EVERY == 0:
        save_training_checkpoint()

progress.close()

# ------------------------------------------
# 14. FINAL SAVE
# ------------------------------------------

save_training_checkpoint()

print("\n===================================")
print("INUTS URDU TTS TRAINING RUN FINISHED")
print("Successful steps this run:", completed_this_run)
print("Skipped batches:", skipped_batches)
print("Global step:", global_step)

if completed_this_run > 0:
    print("Average generator loss:", running_g / completed_this_run)
    print("Average discriminator loss:", running_d / completed_this_run)
    print("Average mel loss:", running_mel / completed_this_run)
    print("Average KL loss:", running_kl / completed_this_run)
    print("Average duration loss:", running_duration / completed_this_run)

print("Checkpoint:", CHECKPOINT_PATH)
print("Pre-training safety backup:", PRE_RUN_BACKUP)
print("===================================")

if global_step < TARGET_GLOBAL_STEP:
    print(
        "NOTICE: DataLoader ended before target step. "
        "Training can be continued from the saved checkpoint."
    )


> **Legacy cell skipped:** Legacy source-inspection cell depended on inspect.getsource for dynamically defined notebook classes; it is not required for training.


In [ ]:
import inspect
import torch

def safe_signature(obj):
    try:
        return str(inspect.signature(obj))
    except Exception as e:
        return f"Unavailable: {e}"

def safe_source(obj):
    try:
        return inspect.getsource(obj)
    except Exception as e:
        return f"Source unavailable: {type(e).__name__}: {e}"

print("===== INUTS TTS MODEL INSPECTION =====")
print("Training step:", global_step)
print("Device:", DEVICE)
print("Vocabulary:", len(vocab))
print("Speakers:", len(speaker_map))

models = {
    "Duration Predictor": duration_predictor,
    "Latent Flow": latent_flow,
    "Waveform Generator": waveform_generator,
    "Text Encoder": text_encoder,
    "Text Prior": text_prior,
    "Speaker Encoder": speaker_encoder,
    "Posterior Encoder": posterior_encoder_v2
}

for name, model in models.items():
    print("\n==============================")
    print(name)
    print("==============================")

    print("Class:", type(model))
    print("Forward signature:", safe_signature(model.forward))

    print("\nChild modules:")
    for child_name, child in model.named_children():
        print(" ", child_name, "->", type(child).__name__)

    print("\nForward source:")
    print(safe_source(type(model).forward))

print("\n===== INSPECTION COMPLETE =====")


In [ ]:
import torch

print("===== FLOW INSPECTION =====")

print("Has inverse:", hasattr(latent_flow, "inverse"))

for i, block in enumerate(latent_flow.blocks):
    print(f"\nFlow block {i}:")
    print(block)
    print("Methods:", [
        name for name in dir(block)
        if "inverse" in name.lower() or "forward" in name.lower()
    ])

print("\n===== DURATION PREDICTOR =====")
print(duration_predictor)

print("\n===== WAVEFORM GENERATOR =====")
print(waveform_generator)

print("\n===== TEXT PRIOR =====")
print(text_prior)

print("\n===== SPEAKER EMBEDDING =====")
print(speaker_encoder)

print("\n===== INFERENCE ARCHITECTURE CHECK COMPLETE =====")


In [ ]:
print("===== DATASET INSPECTION =====")

ds = train_loader.dataset

print("Dataset class:", type(ds).__name__)
print("Dataset length:", len(ds))
print("Dataset attributes:", [
    x for x in dir(ds)
    if not x.startswith("_")
])

sample = ds[0]

print("\n===== FIRST TRAINING SAMPLE =====")
for key, value in sample.items():
    if torch.is_tensor(value):
        print(key, "| shape:", tuple(value.shape), "| dtype:", value.dtype)
        if value.numel() <= 30:
            print("  values:", value)
    else:
        print(key, "|", str(value)[:300])

print("\n===== VOCABULARY SAMPLE =====")
print("First 40 vocab entries:", list(vocab.items())[:40])

print("\n===== SPEAKER MAP SAMPLE =====")
print("First 15 speaker entries:", list(speaker_map.items())[:15])

print("\n===== INSPECTION COMPLETE =====")

In [ ]:
import inspect

print("===== TEXT ENCODING =====")

print("encode_text signature:")
print(inspect.signature(ds.encode_text))

print("\nDataset columns:")
print(list(ds.df.columns))

print("\nFirst dataset row:")
print(ds.df.iloc[0].to_dict())

raw_text = str(ds.df.iloc[0]["sentence"])

print("\nOriginal Urdu text:")
print(raw_text)

encoded = ds.encode_text(raw_text)

print("\nEncoded text:")
print(encoded)

print("\nEncoded type:", type(encoded))

if torch.is_tensor(encoded):
    print("Encoded shape:", encoded.shape)
    print("Encoded values:", encoded[:50])

print("\n===== INFERENCE FUNCTION SIGNATURES =====")

for name, obj in [
    ("Text Encoder", text_encoder),
    ("Text Prior", text_prior),
    ("Speaker Encoder", speaker_encoder),
    ("Duration Predictor", duration_predictor),
    ("Latent Flow", latent_flow),
    ("Flow Inverse", latent_flow.inverse),
    ("Waveform Generator", waveform_generator),
]:
    try:
        print(name, inspect.signature(obj.forward if hasattr(obj, "forward") else obj))
    except Exception as e:
        print(name, "Signature unavailable:", str(e))

print("\n===== CHECK COMPLETE =====")

> **Legacy cell skipped:** Legacy tensor diagnostic used the wrong channel layout for the text prior. The corrected inference pipeline is in cell 147.


> **Legacy cell skipped:** Legacy tensor diagnostic used inconsistent time/channel layouts. The corrected inference pipeline is in cell 147.


In [ ]:
import torch
import dis

sample = ds[0]

text = sample["text"].unsqueeze(0).to(DEVICE)
text_lengths = torch.tensor([sample["text_length"]], device=DEVICE)

models = [text_encoder, text_prior, duration_predictor]
old_modes = [m.training for m in models]

try:
    for m in models:
        m.eval()

    with torch.no_grad():
        result = text_encoder(text, text_lengths)

        if isinstance(result, (tuple, list)):
            print("Encoder returned:", len(result), "items")
            for i, x in enumerate(result):
                if torch.is_tensor(x):
                    print(f"Output {i} shape:", tuple(x.shape))
            text_features = result[0]
        else:
            text_features = result

        print("\nEncoder output shape:", tuple(text_features.shape))

        print("\n===== PRIOR FORWARD INSTRUCTIONS =====")
        dis.dis(text_prior.forward)

        print("\n===== DURATION FORWARD INSTRUCTIONS =====")
        dis.dis(duration_predictor.forward)

finally:
    for m, mode in zip(models, old_modes):
        m.train(mode)

print("\n===== DIAGNOSTIC COMPLETE =====")

In [ ]:
import torch

print("===== FINAL INFERENCE PIPELINE TEST =====")

sample = ds[0]

text = sample["text"].unsqueeze(0).to(DEVICE)
text_lengths = torch.tensor([sample["text_length"]], device=DEVICE)
speaker_ids = sample["speaker"].reshape(1).to(DEVICE)

models = [
    text_encoder, text_prior, speaker_encoder,
    duration_predictor, latent_flow, waveform_generator
]
old_modes = [m.training for m in models]

try:
    for m in models:
        m.eval()

    with torch.no_grad():

        # 1. Text encoder
        text_features = text_encoder(text, text_lengths)
        if isinstance(text_features, (tuple, list)):
            text_features = text_features[0]

        print("Text features:", tuple(text_features.shape))

        # 2. Speaker conditioning
        speaker_features = speaker_encoder(speaker_ids)
        print("Speaker features:", tuple(speaker_features.shape))

        conditioned = text_features + speaker_features.unsqueeze(1)
        print("Conditioned features:", tuple(conditioned.shape))

        # 3. Text prior
        prior_mean, prior_log_scale = text_prior(conditioned)
        print("Prior mean:", tuple(prior_mean.shape))
        print("Prior log scale:", tuple(prior_log_scale.shape))

        # 4. Duration predictor
        durations = duration_predictor(conditioned)
        print("Duration output:", tuple(durations.shape))
        print("Duration min/max:",
              durations.min().item(), durations.max().item())

        # 5. Flow inverse test
        test_latent = torch.randn(1, 192, 20, device=DEVICE)
        inverse_output = latent_flow.inverse(test_latent)

        if isinstance(inverse_output, (tuple, list)):
            inverse_output = inverse_output[0]

        print("Flow inverse:", tuple(inverse_output.shape))

        # 6. Waveform generator
        generated_audio = waveform_generator(inverse_output)

        print("Generated waveform:", tuple(generated_audio.shape))
        print("Audio min/max:",
              generated_audio.min().item(),
              generated_audio.max().item())

finally:
    for m, mode in zip(models, old_modes):
        m.train(mode)

print("\n===== PIPELINE TEST COMPLETE =====")

In [ ]:
import torch

print("===== DATASET AUDIO INSPECTION =====")

for i in [0, 1, 2, 10, 20]:
    sample = ds[i]

    print(f"\nSample {i}:")
    for key, value in sample.items():
        if torch.is_tensor(value):
            print(
                key,
                "shape:", tuple(value.shape),
                "dtype:", value.dtype
            )
        else:
            print(key, "=", str(value)[:150])

    if i == 0:
        print("\nText tokens:", sample["text_length"])
        print("Speaker ID:", sample["speaker"].item())

print("\n===== INSPECTION COMPLETE =====")

In [ ]:
import torch

print("===== SPECTROGRAM / FRAME INSPECTION =====")

print("\nTransform class:", type(spectrogram_transform).__name__)
print("Transform settings:")
for key, value in vars(spectrogram_transform).items():
    if isinstance(value, (int, float, str, bool, tuple)):
        print(f"  {key}: {value}")

print("\n===== SAMPLE MEL FRAME COUNTS =====")

for i in [0, 1, 2, 10, 20]:
    sample = ds[i]
    audio = sample["audio"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        mel = spectrogram_transform(audio)

    print(
        f"Sample {i}:",
        "text tokens =", sample["text_length"],
        "| audio samples =", sample["audio_length"],
        "| mel shape =", tuple(mel.shape)
    )

print("\n===== INSPECTION COMPLETE =====")

In [ ]:
import os
import json
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

print("===== INUTS URDU TTS — TEXT-TO-SPEECH TEST =====")

# Change this sentence whenever you want to test new Urdu text.
INFERENCE_TEXT = str(train_df.iloc[0]["sentence"])

# Use a known training speaker from the existing speaker map.
SPEAKER_CLIENT_ID = str(train_df.iloc[0]["client_id"])
SPEAKER_ID = int(speaker_map[SPEAKER_CLIENT_ID])

SAMPLE_RATE = 24000
HOP_LENGTH = 256
MAX_INFERENCE_FRAMES = 1200  # roughly 12.8 seconds at 24 kHz

def encode_inference_text(text):
    ids = [vocab.get(ch, vocab["<UNK>"]) for ch in str(text)]
    expanded = [BLANK_ID]
    for token in ids:
        expanded.extend([token, BLANK_ID])
    return torch.tensor(expanded, dtype=torch.long)

if not INFERENCE_TEXT.strip():
    raise ValueError("INFERENCE_TEXT is empty. Add an Urdu sentence first.")

encoded = encode_inference_text(INFERENCE_TEXT).unsqueeze(0).to(DEVICE)
text_lengths = torch.tensor([encoded.size(1)], dtype=torch.long, device=DEVICE)
speaker_ids = torch.tensor([SPEAKER_ID], dtype=torch.long, device=DEVICE)

modules = [
    text_encoder,
    text_prior,
    speaker_encoder,
    duration_predictor,
    latent_flow,
    waveform_generator
]
old_modes = [module.training for module in modules]

try:
    for module in modules:
        module.eval()

    with torch.no_grad():
        # Text + speaker conditioning
        text_features = text_encoder(encoded, text_lengths)
        speaker_features = speaker_encoder(speaker_ids)
        conditioned_text = text_features + speaker_features.unsqueeze(1)

        # Predict text-conditioned latent distributions
        prior_mean, prior_log_scale = text_prior(conditioned_text)

        # Predict each token's log-duration, then convert to frame counts.
        predicted_log_durations = duration_predictor(conditioned_text)
        valid_tokens = (
            torch.arange(encoded.size(1), device=DEVICE)[None, :]
            < text_lengths[:, None]
        )

        durations = torch.exp(
            predicted_log_durations.clamp(min=0.0, max=4.5)
        ).round().long().clamp(min=1, max=90)

        durations = durations * valid_tokens.long()

        # Keep the generated clip within the safe maximum duration.
        total_frames = int(durations.sum().item())
        if total_frames > MAX_INFERENCE_FRAMES:
            scale = MAX_INFERENCE_FRAMES / float(total_frames)
            durations = torch.round(durations.float() * scale).long().clamp(min=1)
            total_frames = int(durations.sum().item())

        if total_frames < 1:
            raise RuntimeError("Duration predictor returned no audio frames.")

        # Expand each token's prior across its predicted audio frames.
        prior_mean_expanded = torch.repeat_interleave(
            prior_mean, durations[0], dim=2
        )
        prior_log_scale_expanded = torch.repeat_interleave(
            prior_log_scale, durations[0], dim=2
        )

        # Deterministic latent for a reproducible first quality check.
        prior_latent = prior_mean_expanded

        # Reverse the normalizing flow and synthesize waveform.
        latent = latent_flow.inverse(prior_latent)
        if isinstance(latent, (tuple, list)):
            latent = latent[0]

        generated = waveform_generator(latent).squeeze().float()
        generated = generated.detach().cpu().numpy()

finally:
    for module, mode in zip(modules, old_modes):
        module.train(mode)

generated = np.nan_to_num(generated, nan=0.0, posinf=0.0, neginf=0.0)
peak = float(np.max(np.abs(generated))) if generated.size else 0.0

if generated.size == 0 or peak == 0.0:
    raise RuntimeError("Generated audio is empty or silent.")

# Peak normalization is only for convenient listening; it does not improve model quality.
generated = generated / peak * 0.8

output_path = "/kaggle/working/inuts_urdu_tts_test.wav"
sf.write(output_path, generated, SAMPLE_RATE)

print("\nText:", INFERENCE_TEXT)
print("Speaker ID:", SPEAKER_ID)
print("Text tokens:", int(text_lengths.item()))
print("Predicted audio frames:", total_frames)
print("Generated samples:", len(generated))
print("Duration:", round(len(generated) / SAMPLE_RATE, 2), "seconds")
print("Peak before listening normalization:", peak)
print("Saved:", output_path)

display(Audio(generated, rate=SAMPLE_RATE))

print("\n===== TTS TEST COMPLETE =====")


In [ ]:
import torch

print("===== POSTERIOR ENCODER DIAGNOSTIC =====")

sample = train_dataset[0]
audio = sample["audio"].unsqueeze(0).to(DEVICE)

old_mode = posterior_encoder_v2.training
posterior_encoder_v2.eval()

try:
    with torch.no_grad():
        spec = spectrogram_transform(audio)
        posterior_mean, posterior_log_scale = posterior_encoder_v2(spec)

        print("Spectrogram:", tuple(spec.shape))
        print("Posterior mean:", tuple(posterior_mean.shape))
        print("Posterior log scale:", tuple(posterior_log_scale.shape))
        print("Finite:", bool(
            torch.isfinite(posterior_mean).all()
            and torch.isfinite(posterior_log_scale).all()
        ))
finally:
    posterior_encoder_v2.train(old_mode)

print("===== DIAGNOSTIC COMPLETE =====")


In [ ]:
import torch

print("===== POSTERIOR ENCODER DIAGNOSTIC =====")

sample = ds[0]
audio = sample["audio"].unsqueeze(0).to(DEVICE)

old_mode = posterior_encoder_v2.training
posterior_encoder_v2.eval()

try:
    with torch.no_grad():
        mel = spectrogram_transform(audio)
        result = posterior_encoder_v2(mel)

        if isinstance(result, (tuple, list)):
            print("Posterior outputs:", len(result))
            for i, item in enumerate(result):
                if torch.is_tensor(item):
                    print(f"Output {i} shape:", tuple(item.shape))
        else:
            print("Posterior output:", tuple(result.shape))

finally:
    posterior_encoder_v2.train(old_mode)

print("\n===== DIAGNOSTIC COMPLETE =====")

In [ ]:
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

print("===== ORIGINAL AUDIO RECONSTRUCTION TEST =====")

sample = ds[0]
audio = sample["audio"].unsqueeze(0).to(DEVICE)

old_modes = [
    posterior_encoder_v2.training,
    latent_flow.training,
    waveform_generator.training
]

try:
    posterior_encoder_v2.eval()
    latent_flow.eval()
    waveform_generator.eval()

    with torch.no_grad():
        mel = spectrogram_transform(audio)

        posterior_mean, posterior_log_scale = posterior_encoder_v2(mel)

        # Flow forward
        flow_result = latent_flow(posterior_mean)

        if isinstance(flow_result, (tuple, list)):
            print("Flow returned", len(flow_result), "items")
            for i, item in enumerate(flow_result):
                if torch.is_tensor(item):
                    print(f"Item {i} shape:", tuple(item.shape))

            prior_latent = flow_result[0]
        else:
            prior_latent = flow_result

        # Flow inverse
        inverse_result = latent_flow.inverse(prior_latent)

        if isinstance(inverse_result, (tuple, list)):
            reconstructed_latent = inverse_result[0]
        else:
            reconstructed_latent = inverse_result

        print("Reconstructed latent:", tuple(reconstructed_latent.shape))

        reconstructed = waveform_generator(reconstructed_latent)
        reconstructed = reconstructed.squeeze().float().cpu().numpy()

finally:
    posterior_encoder_v2.train(old_modes[0])
    latent_flow.train(old_modes[1])
    waveform_generator.train(old_modes[2])

reconstructed = np.nan_to_num(reconstructed)
peak = np.max(np.abs(reconstructed))

if peak > 0:
    reconstructed = reconstructed / peak * 0.8

path = "/kaggle/working/urdu_reconstruction_test.wav"
sf.write(path, reconstructed, 24000)

print("Original samples:", audio.shape[-1])
print("Reconstructed samples:", len(reconstructed))
print("Duration:", round(len(reconstructed) / 24000, 2), "seconds")
print("Saved:", path)

display(Audio(reconstructed, rate=24000))

print("\n===== RECONSTRUCTION TEST COMPLETE =====")

In [ ]:
import os
import torch

print("===== CHECKPOINT STATUS =====")

# Training step check
if "global_step" in globals():
    print("Current training step:", global_step)
else:
    print("global_step variable: NOT FOUND")

# Latest checkpoint check
checkpoint_path = "/kaggle/working/inuts_urdu_training_latest.pth"

if os.path.exists(checkpoint_path):
    print("Latest checkpoint: AVAILABLE")
    print("File size:", round(os.path.getsize(checkpoint_path) / (1024**2), 2), "MB")

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(checkpoint, dict):
        print("Checkpoint keys:", list(checkpoint.keys())[:20])
        print("Saved step:", checkpoint.get("global_step", checkpoint.get("step", "Not recorded")))
else:
    print("Latest checkpoint: NOT FOUND")

# Backup check
backup_path = "/kaggle/working/inuts_urdu_training_before_step1000.pth"
print("Backup checkpoint:", "AVAILABLE" if os.path.exists(backup_path) else "NOT FOUND")

print("===== CHECK COMPLETE =====")

In [ ]:
import inspect

print("===== TRAINING CODE AUDIT =====")

objects_to_check = [
    "train_step",
    "train_one_epoch",
    "compute_generator_loss",
    "compute_discriminator_loss",
    "monotonic_alignment_search"
]

for name in objects_to_check:
    obj = globals().get(name)

    print(f"\n===== {name} =====")

    if obj is None:
        print("Not found as a standalone function.")
        continue

    try:
        print(inspect.getsource(obj))
    except Exception as e:
        print("Source unavailable:", str(e))

print("\n===== AVAILABLE TRAINING FUNCTIONS =====")

for name, obj in globals().copy().items():
    if inspect.isfunction(obj):
        low = name.lower()
        if any(word in low for word in ["train", "loss", "align"]):
            print(name)

print("\n===== AUDIT COMPLETE =====")

In [ ]:
import re

print("===== SEARCHING NOTEBOOK TRAINING HISTORY =====")

history = globals().get("In", [])

keywords = [
    "optimizer_joint",
    "duration_predictor",
    "monotonic_alignment_search",
    "generator_loss",
    "discriminator_loss",
    "kl_loss",
    "mel_loss",
    ".backward()",
    "optimizer_d.step",
    "optimizer_joint.step"
]

found = []

for cell_no, code in enumerate(history):
    if not isinstance(code, str):
        continue

    lines = code.splitlines()

    matched = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    if matched:
        found.append((cell_no, code, matched))

print("Matching cells found:", len(found))

for cell_no, code, matched in found[-8:]:
    print(f"\n{'='*55}")
    print("CELL NUMBER:", cell_no)
    print("="*55)

    shown = set()

    for i in matched:
        for j in range(max(0, i-3), min(len(code.splitlines()), i+5)):
            if j not in shown:
                print(f"{j+1}: {code.splitlines()[j]}")
                shown.add(j)

print("\n===== HISTORY SEARCH COMPLETE =====")

In [ ]:
history = globals().get("In", [])

print("===== ORIGINAL TRAINING CODE: TARGETED SEARCH =====")

strong_terms = [
    "optimizer_joint.step",
    "optimizer_d.step",
    "loss.backward",
    ".backward()",
    "monotonic_alignment_search(",
    "kl_loss",
    "mel_loss",
    "generator_loss",
    "discriminator_loss"
]

found = []

for cell_no, code in enumerate(history):
    if not isinstance(code, str):
        continue

    matches = [
        term for term in strong_terms
        if term.lower() in code.lower()
    ]

    if matches:
        found.append((cell_no, code, matches))

print("Relevant cells found:", len(found))

for cell_no, code, matches in found:
    print("\n" + "=" * 65)
    print("CELL NUMBER:", cell_no)
    print("MATCHES:", matches)
    print("=" * 65)
    print(code[:14000])

print("\n===== SEARCH COMPLETE =====")

In [ ]:
import torch
import torch.nn.functional as F

print("===== POSTERIOR RECONSTRUCTION DIAGNOSTIC =====")

sample = ds[0]

audio = sample["audio"].float().to(DEVICE).reshape(1, -1)
speaker_id = torch.tensor(
    [int(sample["speaker"])],
    dtype=torch.long,
    device=DEVICE
)

# Preserve original training/evaluation modes
modules = [
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    waveform_generator
]
old_modes = [m.training for m in modules]

try:
    for m in modules:
        m.eval()

    with torch.no_grad():
        # 1. Real audio -> spectrogram
        spec = spectrogram_transform(audio)

        # 2. Posterior encoder (same path as training)
        hidden = posterior_encoder_v2.pre(spec)
        hidden = posterior_encoder_v2.encoder(hidden)

        speaker_features = speaker_encoder(speaker_id)
        hidden = hidden + speaker_to_audio(
            speaker_features
        ).unsqueeze(-1)

        posterior_mean = posterior_encoder_v2.mean(hidden)

        # 3. Flow forward
        flow_result = latent_flow(posterior_mean)

        if isinstance(flow_result, (tuple, list)):
            flow_latent = flow_result[0]
        else:
            flow_latent = flow_result

        # 4. Flow inverse
        inverse_result = latent_flow.inverse(flow_latent)

        if isinstance(inverse_result, (tuple, list)):
            recovered_latent = inverse_result[0]
        else:
            recovered_latent = inverse_result

        flow_error = (
            recovered_latent - posterior_mean
        ).abs().max().item()

        # 5. Generate waveform from reconstructed latent
        reconstructed_audio = waveform_generator(
            recovered_latent
        ).reshape(-1)

        # 6. Compare with original audio
        n = min(audio.numel(), reconstructed_audio.numel())

        real = audio.reshape(-1)[:n]
        generated = reconstructed_audio[:n]

        mse = F.mse_loss(generated, real).item()

        real_rms = real.square().mean().sqrt().item()
        generated_rms = generated.square().mean().sqrt().item()

        correlation = torch.corrcoef(
            torch.stack([real, generated])
        )[0, 1].item()

        print("\n--- RESULTS ---")
        print("Original audio samples:", audio.numel())
        print("Generated audio samples:", reconstructed_audio.numel())
        print("Spectrogram shape:", tuple(spec.shape))
        print("Posterior latent shape:", tuple(posterior_mean.shape))
        print("Flow latent shape:", tuple(flow_latent.shape))
        print("Flow round-trip max error:", flow_error)
        print("Waveform MSE:", mse)
        print("Original audio RMS:", real_rms)
        print("Generated audio RMS:", generated_rms)
        print("Waveform correlation:", correlation)

        print("\nGenerated waveform finite:",
              bool(torch.isfinite(reconstructed_audio).all()))

finally:
    for m, mode in zip(modules, old_modes):
        m.train(mode)

print("\n===== DIAGNOSTIC COMPLETE — NO TRAINING PERFORMED =====")


In [ ]:
import torch
import torch.nn.functional as F

print("===== MEL QUALITY DIAGNOSTIC =====")

sample = ds[0]
audio = sample["audio"].float().to(DEVICE).reshape(1, -1)
speaker_id = torch.tensor(
    [int(sample["speaker"])],
    dtype=torch.long,
    device=DEVICE
)

modules = [
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    waveform_generator
]
old_modes = [m.training for m in modules]

try:
    for m in modules:
        m.eval()

    with torch.no_grad():
        spec = spectrogram_transform(audio)

        hidden = posterior_encoder_v2.pre(spec)
        hidden = posterior_encoder_v2.encoder(hidden)

        speaker_features = speaker_encoder(speaker_id)
        hidden = hidden + speaker_to_audio(
            speaker_features
        ).unsqueeze(-1)

        posterior_mean = posterior_encoder_v2.mean(hidden)

        flow_result = latent_flow(posterior_mean)
        flow_latent = (
            flow_result[0]
            if isinstance(flow_result, (tuple, list))
            else flow_result
        )

        inverse_result = latent_flow.inverse(flow_latent)
        recovered_latent = (
            inverse_result[0]
            if isinstance(inverse_result, (tuple, list))
            else inverse_result
        )

        generated = waveform_generator(
            recovered_latent
        ).reshape(-1)

        n = min(audio.numel(), generated.numel())

        real = audio.reshape(-1)[:n]
        fake = generated[:n]

        real_mel = mel_loss_transform(real)
        fake_mel = mel_loss_transform(fake)

        frames = min(real_mel.size(-1), fake_mel.size(-1))
        real_mel = real_mel[..., :frames]
        fake_mel = fake_mel[..., :frames]

        real_log_mel = torch.log(real_mel + 1e-5)
        fake_log_mel = torch.log(fake_mel + 1e-5)

        mel_l1 = F.l1_loss(
            fake_log_mel,
            real_log_mel
        ).item()

        print("\n--- RESULTS ---")
        print("Real mel shape:", tuple(real_mel.shape))
        print("Generated mel shape:", tuple(fake_mel.shape))
        print("Log-mel L1 difference:", mel_l1)

        print("\nReal mel mean:", real_mel.mean().item())
        print("Generated mel mean:", fake_mel.mean().item())

        print("Real mel std:", real_mel.std().item())
        print("Generated mel std:", fake_mel.std().item())

        print("\nReal waveform RMS:",
              real.square().mean().sqrt().item())
        print("Generated waveform RMS:",
              fake.square().mean().sqrt().item())

finally:
    for m, mode in zip(modules, old_modes):
        m.train(mode)

print("\n===== MEL DIAGNOSTIC COMPLETE =====")
print("No training performed; checkpoint unchanged.")


In [ ]:
import torch

OLD_PATH = "/kaggle/working/inuts_urdu_training_before_step1000.pth"
NEW_PATH = "/kaggle/working/inuts_urdu_training_latest.pth"

print("Loading checkpoints...")

old_ckpt = torch.load(
    OLD_PATH, map_location="cpu", weights_only=False
)

new_ckpt = torch.load(
    NEW_PATH, map_location="cpu", weights_only=False
)

print("Old step:", old_ckpt["global_step"])
print("New step:", new_ckpt["global_step"])

modules_to_check = [
    "waveform_generator",
    "posterior_encoder_v2",
    "text_encoder",
    "text_prior",
    "latent_flow",
    "speaker_encoder"
]

print("\n===== CHECKPOINT WEIGHT COMPARISON =====")

for module_name in modules_to_check:

    old_state = old_ckpt[module_name]
    new_state = new_ckpt[module_name]

    changed_tensors = 0
    total_tensors = 0
    total_delta = 0.0
    max_delta = 0.0

    for key in old_state:

        if key not in new_state:
            continue

        old_tensor = old_state[key].float()
        new_tensor = new_state[key].float()

        if old_tensor.shape != new_tensor.shape:
            continue

        delta = (new_tensor - old_tensor).abs()

        total_tensors += 1

        if delta.max().item() > 0:
            changed_tensors += 1

        total_delta += delta.mean().item()
        max_delta = max(max_delta, delta.max().item())

    average_delta = (
        total_delta / total_tensors
        if total_tensors else 0
    )

    print(f"\n{module_name}")
    print("Changed tensors:", changed_tensors, "/", total_tensors)
    print("Average weight change:", average_delta)
    print("Maximum weight change:", max_delta)

print("\n===== COMPARISON COMPLETE =====")
print("No training performed.")
print("No checkpoint modified.")

In [ ]:
import torch

OLD_PATH = "/kaggle/working/inuts_urdu_training_before_step1000.pth"
NEW_PATH = "/kaggle/working/inuts_urdu_training_latest.pth"

old_ckpt = torch.load(OLD_PATH, map_location="cpu", weights_only=False)
new_ckpt = torch.load(NEW_PATH, map_location="cpu", weights_only=False)

print("===== UNCHANGED TEXT ENCODER WEIGHTS =====")

old_state = old_ckpt["text_encoder"]
new_state = new_ckpt["text_encoder"]

for key in old_state:
    if key not in new_state:
        print("Missing in new checkpoint:", key)
        continue

    a = old_state[key]
    b = new_state[key]

    if torch.equal(a, b):
        print("UNCHANGED:", key)
        print("Shape:", tuple(a.shape))
        print("Dtype:", a.dtype)

print("\n===== CHECK COMPLETE =====")
print("Checkpoints unchanged.")

In [ ]:
import inspect

print("===== WAVEFORM GENERATOR SOURCE =====")
try:
    print(inspect.getsource(UrduWaveformGenerator))
except Exception as e:
    print("Source unavailable:", e)

print("\n===== MEL LOSS TRANSFORM =====")
print(mel_loss_transform)

print("\n===== SPECTROGRAM TRANSFORM =====")
print(spectrogram_transform)

print("\n===== WAVEFORM GENERATOR CONFIG =====")
print("Upsampling layers:", len(waveform_generator.upsamplers))

for i, layer in enumerate(waveform_generator.upsamplers):
    print(i, layer)

print("\n===== INSPECTION COMPLETE =====")
print("No training performed.")

In [ ]:
import re

print("===== AUDIO NORMALIZATION CODE SEARCH =====")

keywords = [
    "audio =",
    "waveform",
    "normalize",
    "normalization",
    "peak",
    "clamp",
    "tanh",
    "fake_wave",
    "real_wave",
    "mel_loss",
    "audio_lengths"
]

matches = []

for cell_no, cell in enumerate(In):
    source = cell if isinstance(cell, str) else ""

    lines = source.splitlines()

    for line_no, line in enumerate(lines):
        if any(word.lower() in line.lower() for word in keywords):
            matches.append(
                (cell_no, line_no + 1, line.strip())
            )

for cell_no, line_no, line in matches[-100:]:
    print(f"Cell {cell_no}, Line {line_no}: {line}")

print("\nTotal matching lines:", len(matches))
print("No training performed.")

In [ ]:
print("========== ORIGINAL TRAINING CELL 29 ==========")

for i, line in enumerate(In[29].splitlines(), 1):
    print(f"{i:03d}: {line}")

print("\n========== INFERENCE CELL 41 ==========")

for i, line in enumerate(In[41].splitlines(), 1):
    print(f"{i:03d}: {line}")

print("\n========== END OF CODE INSPECTION ==========")
print("No training performed.")
print("No checkpoint modified.")

In [ ]:
import torch
import torch.nn.functional as F
from IPython.display import Audio, display

print("===== CORRECTED GENERATOR INPUT TEST =====")

sample = ds[0]

audio = sample["audio"].float().to(DEVICE).reshape(1, -1)
speaker_id = torch.tensor(
    [int(sample["speaker"])],
    dtype=torch.long,
    device=DEVICE
)

modules = [
    posterior_encoder_v2,
    speaker_encoder,
    speaker_to_audio,
    latent_flow,
    waveform_generator
]

old_modes = [m.training for m in modules]

try:
    for m in modules:
        m.eval()

    with torch.no_grad():

        # Original audio ka spectrogram
        spec = spectrogram_transform(audio)

        # Posterior encoder
        hidden = posterior_encoder_v2.pre(spec)
        hidden = posterior_encoder_v2.encoder(hidden)

        speaker_features = speaker_encoder(speaker_id)

        hidden = hidden + speaker_to_audio(
            speaker_features
        ).unsqueeze(-1)

        posterior_mean = posterior_encoder_v2.mean(hidden)

        # Training jaisa flow forward
        flow_result = latent_flow(posterior_mean)

        flow_latent = (
            flow_result[0]
            if isinstance(flow_result, (tuple, list))
            else flow_result
        )

        # IMPORTANT: No inverse here
        generated = waveform_generator(flow_latent).reshape(-1)

        n = min(audio.numel(), generated.numel())

        real = audio.reshape(-1)[:n]
        fake = generated[:n]

        real_mel = mel_loss_transform(real)
        fake_mel = mel_loss_transform(fake)

        frames = min(
            real_mel.size(-1),
            fake_mel.size(-1)
        )

        mel_difference = F.l1_loss(
            torch.log(fake_mel[..., :frames] + 1e-5),
            torch.log(real_mel[..., :frames] + 1e-5)
        ).item()

        print("\n--- RESULTS ---")
        print("Flow latent:", tuple(flow_latent.shape))
        print("Generated audio:", tuple(generated.shape))

        print("Real RMS:",
              real.square().mean().sqrt().item())

        print("Generated RMS:",
              fake.square().mean().sqrt().item())

        print("Log-mel L1 difference:", mel_difference)

        print("\nGenerated audio preview:")
        display(Audio(
            generated.detach().cpu().numpy(),
            rate=24000
        ))

finally:
    for m, mode in zip(modules, old_modes):
        m.train(mode)

print("\n===== TEST COMPLETE =====")
print("No training performed.")
print("No checkpoint modified.")

## What was corrected in this version
- Added a supervised duration-prediction loss using the monotonic-alignment duration targets.
- Added duration loss to the training metrics and checkpointed model path (the model state already includes the duration predictor).
- Increased the continuation target to global step 1500.
- Replaced the equal-duration demo inference with duration-predictor-based inference.
- Repaired the malformed posterior diagnostic cell.
- Disabled obsolete exploratory cells that caused known errors or duplicated later setup.

After training, run the final TTS test cell and listen to `/kaggle/working/inuts_urdu_tts_test.wav`.
